In [1]:
!pip install requests beautifulsoup4 pandas lxml

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [8]:
url = "https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

print(response.status_code)

200


In [9]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)

Health Innovation Funding Opportunities - Health Innovation Network


In [10]:
print(response.url)

https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/


In [11]:
print(response.text[:1000])

<!doctype html><html lang="en-GB" prefix="og: https://ogp.me/ns#"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width, initial-scale=1"><link rel="profile" href="https://gmpg.org/xfn/11"><title>Health Innovation Funding Opportunities - Health Innovation Network</title><meta name="description" content="We try to make this as relevant to you as possible so please let us know if there is anything we have missed that you feel should be included. Covid-19 Specific Funding: New Future Fund To Support"/><meta name="robots" content="index, follow, max-snippet:-1, max-video-preview:-1, max-image-preview:large"/><link rel="canonical" href="https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/" /><meta property="og:locale" content="en_GB" /><meta property="og:type" content="article" /><meta property="og:title" content="Health Innovation Funding Opportunities - Health Innovation Network" /><meta property="og:description" content="We try to make

In [12]:
for heading in soup.find_all(["h1", "h2", "h3"]):
    print(heading.get_text(strip=True))

Innovators and industry
Services
Programmes
Latest Projects
Blogs
News
Events
Services
Programmes
Health Innovation Funding Opportunities
Featured funding opportunities
General health innovation funding opportunities
Trusts and Charities
Post navigation


In [13]:
h2_tags = soup.find_all("h2")

for h2 in h2_tags:
    print("=" * 50)
    print(h2.get_text(strip=True))

    sibling = h2.find_next_sibling()

    for i in range(3):
        if sibling:
            print(sibling.name)
            print(sibling.get_text(" ", strip=True)[:300])
            print()
            sibling = sibling.find_next_sibling()

Featured funding opportunities
p
Horizon Europe 2026-2027: Health, Cluster 1 Deadline : various (inc. September 2026, April 2027, and September 2027.

p
The indicative amount of EU funding contributions for projects in this cluster is mostly between €1.5m and €10m. This can cover up to 100% of costs depending on the call. The European Commission has published all of the 2026-27 Horizon Europe Work Programmes, which list the funding opportunities und

p


General health innovation funding opportunities
Trusts and Charities
Post navigation
div
Improving elective care pathways through early screening and smarter design Previous   |



In [14]:
paragraphs = soup.find_all("p")

print("Number of paragraphs:", len(paragraphs))

Number of paragraphs: 36


In [15]:
for i, p in enumerate(paragraphs[:20]):
    print(f"\nParagraph {i}")
    print(p.get_text(" ", strip=True))


Paragraph 0
The latest funding opportunities and grants for innovation in healthcare.

Paragraph 1
We update this page frequently – check back for the latest opportunities or subscribe to our newsletter for updates.

Paragraph 2
Horizon Europe 2026-2027: Health, Cluster 1 Deadline : various (inc. September 2026, April 2027, and September 2027.

Paragraph 3
The indicative amount of EU funding contributions for projects in this cluster is mostly between €1.5m and €10m. This can cover up to 100% of costs depending on the call. The European Commission has published all of the 2026-27 Horizon Europe Work Programmes, which list the funding opportunities under the 2026-27 call topics.

Paragraph 4


Paragraph 5
UK-Switzerland CR&D Round 3 Deadline : 03 September 2026, at 11:00am

Paragraph 6
UK registered businesses can apply for a share of up to £3 million for innovative projects in specific technology areas. You must collaborate with at least one Swiss implementation partner applying under

In [16]:
columns = [
    "Funding Name",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Website",
    "Category",
    "Status"
]

df = pd.DataFrame(columns=columns)

df

,Funding Name,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status


In [18]:
paragraphs = [
    p.get_text(" ", strip=True)
    for p in soup.find_all("p")
]

print(len(paragraphs))

36


In [20]:
clean_paragraphs = [
    p for p in paragraphs if p != ""
]

print(len(clean_paragraphs))

31


In [21]:
for i, p in enumerate(clean_paragraphs):
    print(i, ":", p)

0 : The latest funding opportunities and grants for innovation in healthcare.
1 : We update this page frequently – check back for the latest opportunities or subscribe to our newsletter for updates.
2 : Horizon Europe 2026-2027: Health, Cluster 1 Deadline : various (inc. September 2026, April 2027, and September 2027.
3 : The indicative amount of EU funding contributions for projects in this cluster is mostly between €1.5m and €10m. This can cover up to 100% of costs depending on the call. The European Commission has published all of the 2026-27 Horizon Europe Work Programmes, which list the funding opportunities under the 2026-27 call topics.
4 : UK-Switzerland CR&D Round 3 Deadline : 03 September 2026, at 11:00am
5 : UK registered businesses can apply for a share of up to £3 million for innovative projects in specific technology areas. You must collaborate with at least one Swiss implementation partner applying under the equivalent Swiss Innosuisse programme.
6 : Digital Catapult: Di

In [22]:
funding_data = []

In [23]:
for i, p in enumerate(clean_paragraphs):
    if "Deadline" in p:
        print(i, p)

2 Horizon Europe 2026-2027: Health, Cluster 1 Deadline : various (inc. September 2026, April 2027, and September 2027.
4 UK-Switzerland CR&D Round 3 Deadline : 03 September 2026, at 11:00am
7 Deadline : 06 September 2026, at 11:59pm
10 Deadline : 16 September 2026, at 3:00pm
13 Deadline : 08 October 2026, at 5:00pm (GMT+1) Brussels time
16 Deadline : 11 November 2026, at 4:00pm


In [24]:
import re

In [25]:
def extract_amount(text):
    amounts = re.findall(r"(?:£|€)\s?\d+(?:\.\d+)?(?:m|k| million|,?\d+)?", text)
    return ", ".join(amounts) if amounts else "Not stated"

In [26]:
print(extract_amount(clean_paragraphs[3]))

€1.5m, €10m


In [27]:
funding_data = []

In [28]:
opportunities = [
    clean_paragraphs[2:4],
    clean_paragraphs[4:6],
    clean_paragraphs[6:9],
    clean_paragraphs[9:12],
    clean_paragraphs[12:15],
    clean_paragraphs[15:18]
]

len(opportunities)

6

In [29]:
for opp in opportunities:
    print("-----")
    for item in opp:
        print(item)

-----
Horizon Europe 2026-2027: Health, Cluster 1 Deadline : various (inc. September 2026, April 2027, and September 2027.
The indicative amount of EU funding contributions for projects in this cluster is mostly between €1.5m and €10m. This can cover up to 100% of costs depending on the call. The European Commission has published all of the 2026-27 Horizon Europe Work Programmes, which list the funding opportunities under the 2026-27 call topics.
-----
UK-Switzerland CR&D Round 3 Deadline : 03 September 2026, at 11:00am
UK registered businesses can apply for a share of up to £3 million for innovative projects in specific technology areas. You must collaborate with at least one Swiss implementation partner applying under the equivalent Swiss Innosuisse programme.
-----
Digital Catapult: Digital Twin Adoption Accelerator 2026
Deadline : 06 September 2026, at 11:59pm
The Digital Twin Adoption Accelerator Programme 2026 is a nine-month programme that supports partnerships between UK techno

In [30]:
import re

def extract_deadline(text):
    if "Deadline" in text:
        return text.split("Deadline")[1].replace(":", "").strip()
    return ""

def extract_name(text):
    if "Deadline" in text:
        return text.split("Deadline")[0].strip()
    return text.strip()

In [31]:
funding_data = []

for opp in opportunities:
    
    name = extract_name(opp[0])
    
    deadline = ""
    description = ""
    
    for item in opp:
        if "Deadline" in item:
            deadline = extract_deadline(item)
        
        elif item != opp[0]:
            description += item + " "
            
    funding_data.append({
        "Funding Name": name,
        "Type": "Grant",
        "Eligibility": "Not stated",
        "Deadline": deadline,
        "States/Country/Region Covered": "Not stated",
        "Funding Amount": extract_amount(description),
        "Website": url,
        "Category": "Health Innovation",
        "Status": "Upcoming"
    })

In [32]:
df = pd.DataFrame(funding_data)

df

,Funding Name,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,"Horizon Europe 2026-2027: Health, Cluster 1",Grant,Not stated,"various (inc. September 2026, April 2027, and ...",Not stated,"€1.5m, €10m",https://healthinnovationnetwork.com/news/healt...,Health Innovation,Upcoming
1,UK-Switzerland CR&D Round 3,Grant,Not stated,"03 September 2026, at 1100am",Not stated,£3 million,https://healthinnovationnetwork.com/news/healt...,Health Innovation,Upcoming
2,Digital Catapult: Digital Twin Adoption Accele...,Grant,Not stated,"06 September 2026, at 1159pm",Not stated,"£100,000",https://healthinnovationnetwork.com/news/healt...,Health Innovation,Upcoming
3,EIT Health Innovation Uptake Call 2026,Grant,Not stated,"16 September 2026, at 300pm",Not stated,"€650k, €650k, €400k, €500k",https://healthinnovationnetwork.com/news/healt...,Health Innovation,Upcoming
4,IHI (Innovative Health Initiative) 13th Call f...,Grant,Not stated,"08 October 2026, at 500pm (GMT+1) Brussels time",Not stated,"€9m, €35m",https://healthinnovationnetwork.com/news/healt...,Health Innovation,Upcoming
5,UKRI Translation: MRC Impact Acceleration Awards,Grant,Not stated,"11 November 2026, at 400pm",Not stated,"£50k, £300k",https://healthinnovationnetwork.com/news/healt...,Health Innovation,Upcoming


In [33]:
def get_type(text):
    text = text.lower()

    if "accelerator" in text:
        return "Accelerator"
    elif "call" in text:
        return "Call for Proposals"
    elif "award" in text:
        return "Award"
    elif "programme" in text or "program" in text:
        return "Programme"
    elif "grant" in text:
        return "Grant"
    else:
        return "Funding Opportunity"

In [34]:
df["Type"] = df["Funding Name"].apply(get_type)

In [35]:
df[["Funding Name","Type"]]

,Funding Name,Type
0,"Horizon Europe 2026-2027: Health, Cluster 1",Funding Opportunity
1,UK-Switzerland CR&D Round 3,Funding Opportunity
2,Digital Catapult: Digital Twin Adoption Accele...,Accelerator
3,EIT Health Innovation Uptake Call 2026,Call for Proposals
4,IHI (Innovative Health Initiative) 13th Call f...,Call for Proposals
5,UKRI Translation: MRC Impact Acceleration Awards,Award


In [43]:
funding_data.append({
    "Funding Name": name,
    "Description": description,
    "Type": get_type(name),
    "Eligibility": "Not stated",
    "Deadline": deadline,
    "States/Country/Region Covered": "",
    "Funding Amount": extract_amount(description),
    "Website": url,
    "Category": "Health Innovation",
    "Status": "Upcoming"
})

In [44]:
df = pd.DataFrame(funding_data)

In [45]:
df.columns

Index(['Funding Name', 'Type', 'Eligibility', 'Deadline',
       'States/Country/Region Covered', 'Funding Amount', 'Website',
       'Category', 'Status', 'Description'],
      dtype='object')

In [49]:
def extract_region(text):
    if not isinstance(text, str):
        return "Not stated"

    regions = []

    keywords = {
        "UK": ["UK", "United Kingdom", "British"],
        "Europe": ["Europe", "European", "EU"],
        "Switzerland": ["Swiss", "Switzerland"],
        "Global": ["worldwide", "global"]
    }

    for region, words in keywords.items():
        for word in words:
            if word.lower() in text.lower():
                regions.append(region)

    return ", ".join(set(regions)) if regions else "Not stated"

In [50]:
df["States/Country/Region Covered"] = df["Description"].apply(extract_region)

In [51]:
df[["Funding Name", "States/Country/Region Covered"]]

,Funding Name,States/Country/Region Covered
0,"Horizon Europe 2026-2027: Health, Cluster 1",Not stated
1,UK-Switzerland CR&D Round 3,Not stated
2,Digital Catapult: Digital Twin Adoption Accele...,Not stated
3,EIT Health Innovation Uptake Call 2026,Not stated
4,IHI (Innovative Health Initiative) 13th Call f...,Not stated
5,UKRI Translation: MRC Impact Acceleration Awards,Not stated
6,UKRI Translation: MRC Impact Acceleration Awards,Not stated
7,UKRI Translation: MRC Impact Acceleration Awards,Not stated
8,UKRI Translation: MRC Impact Acceleration Awards,Not stated


In [52]:
funding_data = []

In [53]:
df = pd.DataFrame(funding_data)

In [54]:
df.shape

(0, 0)

In [55]:
funding_data = []

In [56]:
opportunities = [
    clean_paragraphs[2:4],
    clean_paragraphs[4:6],
    clean_paragraphs[6:9],
    clean_paragraphs[9:12],
    clean_paragraphs[12:15],
    clean_paragraphs[15:18]
]

In [57]:
len(opportunities)

6

In [58]:
for opp in opportunities:
    
    name = extract_name(opp[0])
    
    deadline = ""
    description = ""

    for item in opp:
        
        if "Deadline" in item:
            deadline = extract_deadline(item)
        
        elif item != opp[0]:
            description += item + " "

    funding_data.append({
        "Funding Name": name,
        "Description": description.strip(),
        "Type": get_type(name),
        "Eligibility": "Not stated",
        "Deadline": deadline,
        "States/Country/Region Covered": "",
        "Funding Amount": extract_amount(description),
        "Website": url,
        "Category": "Health Innovation",
        "Status": "Upcoming"
    })

In [59]:
df = pd.DataFrame(funding_data)

In [60]:
df.shape

(6, 10)

In [61]:
df[["Funding Name","Description"]]

,Funding Name,Description
0,"Horizon Europe 2026-2027: Health, Cluster 1",The indicative amount of EU funding contributi...
1,UK-Switzerland CR&D Round 3,UK registered businesses can apply for a share...
2,Digital Catapult: Digital Twin Adoption Accele...,The Digital Twin Adoption Accelerator Programm...
3,EIT Health Innovation Uptake Call 2026,"EIT Health is calling for mature digital, data..."
4,IHI (Innovative Health Initiative) 13th Call f...,Topics on accelerating healthcare innovation t...
5,UKRI Translation: MRC Impact Acceleration Awards,This funding opportunity was formerly known as...


In [62]:
df["States/Country/Region Covered"] = df["Description"].apply(extract_region)

In [63]:
df[["Funding Name", "States/Country/Region Covered"]]

,Funding Name,States/Country/Region Covered
0,"Horizon Europe 2026-2027: Health, Cluster 1",Europe
1,UK-Switzerland CR&D Round 3,"UK, Switzerland"
2,Digital Catapult: Digital Twin Adoption Accele...,UK
3,EIT Health Innovation Uptake Call 2026,Europe
4,IHI (Innovative Health Initiative) 13th Call f...,Not stated
5,UKRI Translation: MRC Impact Acceleration Awards,Not stated


In [64]:
def extract_region(row):
    text = str(row["Funding Name"]) + " " + str(row["Description"])

    regions = []

    keywords = {
        "United Kingdom": [
            "UK",
            "United Kingdom",
            "British",
            "UKRI",
            "MRC",
            "Innovate UK",
            "England"
        ],
        "Europe": [
            "Europe",
            "European",
            "EU",
            "Brussels",
            "Horizon",
            "EIT Health",
            "IHI"
        ],
        "Switzerland": [
            "Swiss",
            "Switzerland"
        ],
        "Global": [
            "worldwide",
            "global"
        ]
    }

    for region, words in keywords.items():
        for word in words:
            if word.lower() in text.lower():
                regions.append(region)

    return ", ".join(sorted(set(regions))) if regions else "Not stated"

In [65]:
df["States/Country/Region Covered"] = df.apply(extract_region, axis=1)

In [66]:
df[["Funding Name", "States/Country/Region Covered"]]

,Funding Name,States/Country/Region Covered
0,"Horizon Europe 2026-2027: Health, Cluster 1",Europe
1,UK-Switzerland CR&D Round 3,"Switzerland, United Kingdom"
2,Digital Catapult: Digital Twin Adoption Accele...,United Kingdom
3,EIT Health Innovation Uptake Call 2026,Europe
4,IHI (Innovative Health Initiative) 13th Call f...,Europe
5,UKRI Translation: MRC Impact Acceleration Awards,United Kingdom


In [67]:
def extract_eligibility(row):
    text = str(row["Funding Name"]) + " " + str(row["Description"])

    sentences = text.split(".")

    eligibility_sentences = []

    keywords = [
        "apply",
        "applicants",
        "businesses",
        "SMEs",
        "research",
        "organisations",
        "organizations",
        "partners",
        "collaborate",
        "led by",
        "eligible"
    ]

    for sentence in sentences:
        if any(word.lower() in sentence.lower() for word in keywords):
            eligibility_sentences.append(sentence.strip())

    return ". ".join(eligibility_sentences) if eligibility_sentences else "Not stated"

In [68]:
df["Eligibility"] = df.apply(extract_eligibility, axis=1)

In [69]:
df[["Funding Name", "Eligibility"]]

,Funding Name,Eligibility
0,"Horizon Europe 2026-2027: Health, Cluster 1",Not stated
1,UK-Switzerland CR&D Round 3,UK-Switzerland CR&D Round 3 UK registered busi...
2,Digital Catapult: Digital Twin Adoption Accele...,Digital Catapult: Digital Twin Adoption Accele...
3,EIT Health Innovation Uptake Call 2026,SMEs leading projects receive €400k to €500k w...
4,IHI (Innovative Health Initiative) 13th Call f...,A percentage of costs must be provided by cont...
5,UKRI Translation: MRC Impact Acceleration Awards,Led by MRC-eligible research organisations; he...


In [70]:
def extract_eligibility(row):
    text = str(row["Description"])

    sentences = text.split(".")

    eligibility_sentences = []

    keywords = [
        "apply",
        "applicants",
        "businesses",
        "SMEs",
        "research organisations",
        "research organizations",
        "partners",
        "collaborate",
        "eligible",
        "led by"
    ]

    for sentence in sentences:
        sentence = sentence.strip()

        if any(word.lower() in sentence.lower() for word in keywords):
            eligibility_sentences.append(sentence)

    return ". ".join(eligibility_sentences) if eligibility_sentences else "Not stated"

In [71]:
df["Eligibility"] = df.apply(extract_eligibility, axis=1)

In [72]:
df[["Funding Name","Eligibility"]]

,Funding Name,Eligibility
0,"Horizon Europe 2026-2027: Health, Cluster 1",Not stated
1,UK-Switzerland CR&D Round 3,UK registered businesses can apply for a share...
2,Digital Catapult: Digital Twin Adoption Accele...,The Digital Twin Adoption Accelerator Programm...
3,EIT Health Innovation Uptake Call 2026,SMEs leading projects receive €400k to €500k w...
4,IHI (Innovative Health Initiative) 13th Call f...,A percentage of costs must be provided by cont...
5,UKRI Translation: MRC Impact Acceleration Awards,Led by MRC-eligible research organisations; he...


In [73]:
def extract_eligibility(row):
    text = str(row["Description"])

    sentences = text.split(".")

    eligibility_sentences = []

    keywords = [
        "apply",
        "applicants",
        "businesses",
        "SMEs",
        "research organisations",
        "research organizations",
        "partners",
        "collaborate",
        "eligible",
        "led by",
        "participants"
    ]

    exclude_words = [
        "cost",
        "costs",
        "funding covers",
        "contribution",
        "percentage",
        "grant amount",
        "receive up to"
    ]

    for sentence in sentences:
        sentence = sentence.strip()

        if any(word.lower() in sentence.lower() for word in keywords):
            
            if not any(word.lower() in sentence.lower() for word in exclude_words):
                eligibility_sentences.append(sentence)

    return ". ".join(eligibility_sentences) if eligibility_sentences else "Not stated"

In [74]:
df["Eligibility"] = df.apply(extract_eligibility, axis=1)

In [75]:
df[["Funding Name","Eligibility"]]

,Funding Name,Eligibility
0,"Horizon Europe 2026-2027: Health, Cluster 1",Not stated
1,UK-Switzerland CR&D Round 3,UK registered businesses can apply for a share...
2,Digital Catapult: Digital Twin Adoption Accele...,The Digital Twin Adoption Accelerator Programm...
3,EIT Health Innovation Uptake Call 2026,SMEs leading projects receive €400k to €500k w...
4,IHI (Innovative Health Initiative) 13th Call f...,Not stated
5,UKRI Translation: MRC Impact Acceleration Awards,Led by MRC-eligible research organisations; he...


In [76]:
df


,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,"Horizon Europe 2026-2027: Health, Cluster 1",The indicative amount of EU funding contributi...,Funding Opportunity,Not stated,"various (inc. September 2026, April 2027, and ...",Europe,"€1.5m, €10m",https://healthinnovationnetwork.com/news/healt...,Health Innovation,Upcoming
1,UK-Switzerland CR&D Round 3,UK registered businesses can apply for a share...,Funding Opportunity,UK registered businesses can apply for a share...,"03 September 2026, at 1100am","Switzerland, United Kingdom",£3 million,https://healthinnovationnetwork.com/news/healt...,Health Innovation,Upcoming
2,Digital Catapult: Digital Twin Adoption Accele...,The Digital Twin Adoption Accelerator Programm...,Accelerator,The Digital Twin Adoption Accelerator Programm...,"06 September 2026, at 1159pm",United Kingdom,"£100,000",https://healthinnovationnetwork.com/news/healt...,Health Innovation,Upcoming
3,EIT Health Innovation Uptake Call 2026,"EIT Health is calling for mature digital, data...",Call for Proposals,SMEs leading projects receive €400k to €500k w...,"16 September 2026, at 300pm",Europe,"€650k, €650k, €400k, €500k",https://healthinnovationnetwork.com/news/healt...,Health Innovation,Upcoming
4,IHI (Innovative Health Initiative) 13th Call f...,Topics on accelerating healthcare innovation t...,Call for Proposals,Not stated,"08 October 2026, at 500pm (GMT+1) Brussels time",Europe,"€9m, €35m",https://healthinnovationnetwork.com/news/healt...,Health Innovation,Upcoming
5,UKRI Translation: MRC Impact Acceleration Awards,This funding opportunity was formerly known as...,Award,Led by MRC-eligible research organisations; he...,"11 November 2026, at 400pm",United Kingdom,"£50k, £300k",https://healthinnovationnetwork.com/news/healt...,Health Innovation,Upcoming


In [77]:
trusts = clean_paragraphs[18:30]

for i, item in enumerate(trusts):
    print(i, ":", item)

0 : Innovate UK Innovate UK is part of UK Research and Innovation, a non-departmental public body funded by a grant-in-aid from the UK government.
1 : Biotechnology and Biological Sciences Research Council (BBSRC) Biotechnology and Biological Sciences Research Council, part of UK Research and Innovation, is a non-departmental public body, and is the largest UK public funder of non-medical bioscience. It predominantly funds scientific research institutes and university research departments in the UK.
2 : BBSRC standard research grant You can apply for research grants at any time in any area within the remit of BBSRC. BBSRC funds research in plants, microbes, animals (including humans), and the tools and technology underpinning biological research from the level of molecules and cells, to tissues, whole organisms, populations and landscapes.
3 : National Institute for Health and Care Research (NIHR) The National Institute for Health and Care Research is the British government’s major fun

In [78]:
trust_data = []

In [80]:
trust_names = [
    "Innovate UK",
    "Biotechnology and Biological Sciences Research Council (BBSRC)",
    "BBSRC standard research grant",
    "National Institute for Health and Care Research (NIHR)",
    "UK Defence Innovation (UKDI)",
    "Medical Research Council (MRC)",
    "Economic and Social Research Council (ESRC)",
    "NC3RS",
    "The Health Foundation (HF)",
    "The British Heart Foundation (BHF)",
    "Association of Medical Research Charities (AMRC)"
]


trust_data = []

for item, name in zip(trusts, trust_names):

    description = item.replace(name, "").strip()

    trust_data.append({
        "Funding Name": name,
        "Description": description,
        "Type": "Funding Organisation",
        "Eligibility": "Not stated",
        "Deadline": "Not stated",
        "States/Country/Region Covered": "",
        "Funding Amount": "Not stated",
        "Website": url,
        "Category": "Health Innovation",
        "Status": "Active"
    })

In [81]:
trust_df = pd.DataFrame(trust_data)

trust_df

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,Innovate UK,"is part of UK Research and Innovation, a non-d...",Funding Organisation,Not stated,Not stated,,Not stated,https://healthinnovationnetwork.com/news/healt...,Health Innovation,Active
1,Biotechnology and Biological Sciences Research...,Biotechnology and Biological Sciences Research...,Funding Organisation,Not stated,Not stated,,Not stated,https://healthinnovationnetwork.com/news/healt...,Health Innovation,Active
2,BBSRC standard research grant,You can apply for research grants at any time ...,Funding Organisation,Not stated,Not stated,,Not stated,https://healthinnovationnetwork.com/news/healt...,Health Innovation,Active
3,National Institute for Health and Care Researc...,The National Institute for Health and Care Res...,Funding Organisation,Not stated,Not stated,,Not stated,https://healthinnovationnetwork.com/news/healt...,Health Innovation,Active
4,UK Defence Innovation (UKDI),UKDI aims to find and fund exploitable innovat...,Funding Organisation,Not stated,Not stated,,Not stated,https://healthinnovationnetwork.com/news/healt...,Health Innovation,Active
5,Medical Research Council (MRC),The improves the health of people in the UK –...,Funding Organisation,Not stated,Not stated,,Not stated,https://healthinnovationnetwork.com/news/healt...,Health Innovation,Active
6,Economic and Social Research Council (ESRC),,Funding Organisation,Not stated,Not stated,,Not stated,https://healthinnovationnetwork.com/news/healt...,Health Innovation,Active
7,NC3RS,The Economic and Social Research Council (ESRC...,Funding Organisation,Not stated,Not stated,,Not stated,https://healthinnovationnetwork.com/news/healt...,Health Innovation,Active
8,The Health Foundation (HF),NC3RS The NC3RS is a UK-based scientific organ...,Funding Organisation,Not stated,Not stated,,Not stated,https://healthinnovationnetwork.com/news/healt...,Health Innovation,Active
9,The British Heart Foundation (BHF),The Health Foundation (HF) HF’s aim is a healt...,Funding Organisation,Not stated,Not stated,,Not stated,https://healthinnovationnetwork.com/news/healt...,Health Innovation,Active


In [82]:
trust_data = [
    {
        "Funding Name": "Innovate UK",
        "Description": "Innovate UK is part of UK Research and Innovation, a non-departmental public body funded by a grant-in-aid from the UK government.",
        "Type": "Funding Organisation"
    },
    {
        "Funding Name": "Biotechnology and Biological Sciences Research Council (BBSRC)",
        "Description": "BBSRC is part of UK Research and Innovation and funds scientific research institutes and university research departments in the UK.",
        "Type": "Funding Organisation"
    },
    {
        "Funding Name": "BBSRC standard research grant",
        "Description": "Research grants supporting biological research including plants, microbes, animals and technology underpinning biological research.",
        "Type": "Grant"
    },
    {
        "Funding Name": "National Institute for Health and Care Research (NIHR)",
        "Description": "UK government funder of clinical, public health, social care and translational research.",
        "Type": "Funding Organisation"
    },
    {
        "Funding Name": "UK Defence Innovation (UKDI)",
        "Description": "Supports exploitable innovation for UK defence and security.",
        "Type": "Funding Organisation"
    },
    {
        "Funding Name": "Medical Research Council (MRC)",
        "Description": "Supports excellent science and training scientists to improve health in the UK and globally.",
        "Type": "Funding Organisation"
    },
    {
        "Funding Name": "Economic and Social Research Council (ESRC)",
        "Description": "Funds research benefiting economy, society, public policy, health and quality of life.",
        "Type": "Funding Organisation"
    },
    {
        "Funding Name": "NC3RS",
        "Description": "Supports research communities to replace, reduce and refine animal use in medical testing.",
        "Type": "Funding Organisation"
    },
    {
        "Funding Name": "The Health Foundation (HF)",
        "Description": "Supports research, analysis and practical solutions for improving healthcare.",
        "Type": "Charity Funder"
    },
    {
        "Funding Name": "The British Heart Foundation (BHF)",
        "Description": "Provides grants and support for cardiovascular researchers.",
        "Type": "Charity Funder"
    },
    {
        "Funding Name": "Association of Medical Research Charities (AMRC)",
        "Description": "Network of medical research charities supporting medical research funding.",
        "Type": "Funding Network"
    }
]

In [83]:
trust_df = pd.DataFrame(trust_data)

In [84]:
trust_df = pd.DataFrame(trust_data)

In [85]:
trust_df

,Funding Name,Description,Type
0,Innovate UK,Innovate UK is part of UK Research and Innovat...,Funding Organisation
1,Biotechnology and Biological Sciences Research...,BBSRC is part of UK Research and Innovation an...,Funding Organisation
2,BBSRC standard research grant,Research grants supporting biological research...,Grant
3,National Institute for Health and Care Researc...,"UK government funder of clinical, public healt...",Funding Organisation
4,UK Defence Innovation (UKDI),Supports exploitable innovation for UK defence...,Funding Organisation
5,Medical Research Council (MRC),Supports excellent science and training scient...,Funding Organisation
6,Economic and Social Research Council (ESRC),"Funds research benefiting economy, society, pu...",Funding Organisation
7,NC3RS,"Supports research communities to replace, redu...",Funding Organisation
8,The Health Foundation (HF),"Supports research, analysis and practical solu...",Charity Funder
9,The British Heart Foundation (BHF),Provides grants and support for cardiovascular...,Charity Funder


In [86]:
trust_df = pd.DataFrame(trust_data)

trust_df

,Funding Name,Description,Type
0,Innovate UK,Innovate UK is part of UK Research and Innovat...,Funding Organisation
1,Biotechnology and Biological Sciences Research...,BBSRC is part of UK Research and Innovation an...,Funding Organisation
2,BBSRC standard research grant,Research grants supporting biological research...,Grant
3,National Institute for Health and Care Researc...,"UK government funder of clinical, public healt...",Funding Organisation
4,UK Defence Innovation (UKDI),Supports exploitable innovation for UK defence...,Funding Organisation
5,Medical Research Council (MRC),Supports excellent science and training scient...,Funding Organisation
6,Economic and Social Research Council (ESRC),"Funds research benefiting economy, society, pu...",Funding Organisation
7,NC3RS,"Supports research communities to replace, redu...",Funding Organisation
8,The Health Foundation (HF),"Supports research, analysis and practical solu...",Charity Funder
9,The British Heart Foundation (BHF),Provides grants and support for cardiovascular...,Charity Funder


In [87]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [88]:
trust_df

,Funding Name,Description,Type
0,Innovate UK,"Innovate UK is part of UK Research and Innovation, a non-departmental public body funded by a grant-in-aid from the UK government.",Funding Organisation
1,Biotechnology and Biological Sciences Research Council (BBSRC),BBSRC is part of UK Research and Innovation and funds scientific research institutes and university research departments in the UK.,Funding Organisation
2,BBSRC standard research grant,"Research grants supporting biological research including plants, microbes, animals and technology underpinning biological research.",Grant
3,National Institute for Health and Care Research (NIHR),"UK government funder of clinical, public health, social care and translational research.",Funding Organisation
4,UK Defence Innovation (UKDI),Supports exploitable innovation for UK defence and security.,Funding Organisation
5,Medical Research Council (MRC),Supports excellent science and training scientists to improve health in the UK and globally.,Funding Organisation
6,Economic and Social Research Council (ESRC),"Funds research benefiting economy, society, public policy, health and quality of life.",Funding Organisation
7,NC3RS,"Supports research communities to replace, reduce and refine animal use in medical testing.",Funding Organisation
8,The Health Foundation (HF),"Supports research, analysis and practical solutions for improving healthcare.",Charity Funder
9,The British Heart Foundation (BHF),Provides grants and support for cardiovascular researchers.,Charity Funder


In [90]:
trust_df = pd.DataFrame(trust_data)

trust_df

,Funding Name,Description,Type
0,Innovate UK,"Innovate UK is part of UK Research and Innovation, a non-departmental public body funded by a grant-in-aid from the UK government.",Funding Organisation
1,Biotechnology and Biological Sciences Research Council (BBSRC),BBSRC is part of UK Research and Innovation and funds scientific research institutes and university research departments in the UK.,Funding Organisation
2,BBSRC standard research grant,"Research grants supporting biological research including plants, microbes, animals and technology underpinning biological research.",Grant
3,National Institute for Health and Care Research (NIHR),"UK government funder of clinical, public health, social care and translational research.",Funding Organisation
4,UK Defence Innovation (UKDI),Supports exploitable innovation for UK defence and security.,Funding Organisation
5,Medical Research Council (MRC),Supports excellent science and training scientists to improve health in the UK and globally.,Funding Organisation
6,Economic and Social Research Council (ESRC),"Funds research benefiting economy, society, public policy, health and quality of life.",Funding Organisation
7,NC3RS,"Supports research communities to replace, reduce and refine animal use in medical testing.",Funding Organisation
8,The Health Foundation (HF),"Supports research, analysis and practical solutions for improving healthcare.",Charity Funder
9,The British Heart Foundation (BHF),Provides grants and support for cardiovascular researchers.,Charity Funder


In [93]:
trust_df["Eligibility"] = "Not stated"
trust_df["Deadline"] = "Not stated"
trust_df["States/Country/Region Covered"] = "United Kingdom"
trust_df["Funding Amount"] = "Not stated"
trust_df["Website"] = url
trust_df["Category"] = "Health Innovation"
trust_df["Status"] = "Active"

In [94]:
trust_df.columns

Index(['Funding Name', 'Description', 'Type', 'Eligibility', 'Deadline',
       'States/Country/Region Covered', 'Funding Amount', 'Website',
       'Category', 'Status'],
      dtype='object')

In [95]:
trust_df[[
    "Funding Name",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Website",
    "Category",
    "Status"
]]

,Funding Name,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,Innovate UK,Funding Organisation,Not stated,Not stated,United Kingdom,Not stated,https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Active
1,Biotechnology and Biological Sciences Research Council (BBSRC),Funding Organisation,Not stated,Not stated,United Kingdom,Not stated,https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Active
2,BBSRC standard research grant,Grant,Not stated,Not stated,United Kingdom,Not stated,https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Active
3,National Institute for Health and Care Research (NIHR),Funding Organisation,Not stated,Not stated,United Kingdom,Not stated,https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Active
4,UK Defence Innovation (UKDI),Funding Organisation,Not stated,Not stated,United Kingdom,Not stated,https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Active
5,Medical Research Council (MRC),Funding Organisation,Not stated,Not stated,United Kingdom,Not stated,https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Active
6,Economic and Social Research Council (ESRC),Funding Organisation,Not stated,Not stated,United Kingdom,Not stated,https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Active
7,NC3RS,Funding Organisation,Not stated,Not stated,United Kingdom,Not stated,https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Active
8,The Health Foundation (HF),Charity Funder,Not stated,Not stated,United Kingdom,Not stated,https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Active
9,The British Heart Foundation (BHF),Charity Funder,Not stated,Not stated,United Kingdom,Not stated,https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Active


In [96]:
trust_df["Deadline"] = "Multiple funding calls"
trust_df["Funding Amount"] = "Multiple grants available"
trust_df["Status"] = "Active"

In [97]:
eligibility_map = {
    "Innovate UK": "UK registered businesses, innovators and organisations developing innovative solutions",
    "Biotechnology and Biological Sciences Research Council (BBSRC)": "Researchers, universities and eligible research organisations",
    "BBSRC standard research grant": "Researchers and eligible research organisations conducting biological research",
    "National Institute for Health and Care Research (NIHR)": "Researchers, healthcare professionals and research organisations",
    "UK Defence Innovation (UKDI)": "UK innovators and organisations working on defence and security innovation",
    "Medical Research Council (MRC)": "Researchers and research organisations conducting health and biomedical research",
    "Economic and Social Research Council (ESRC)": "Researchers and organisations conducting social science research",
    "NC3RS": "Researchers and organisations involved in medical research and animal testing alternatives",
    "The Health Foundation (HF)": "Health and care researchers, organisations and innovators",
    "The British Heart Foundation (BHF)": "Clinical and non-clinical cardiovascular researchers",
    "Association of Medical Research Charities (AMRC)": "Medical research charities and organisations"
}

trust_df["Eligibility"] = trust_df["Funding Name"].map(eligibility_map)

In [98]:
region_map = {
    "Innovate UK": "United Kingdom",
    "Biotechnology and Biological Sciences Research Council (BBSRC)": "United Kingdom",
    "BBSRC standard research grant": "United Kingdom",
    "National Institute for Health and Care Research (NIHR)": "United Kingdom",
    "UK Defence Innovation (UKDI)": "United Kingdom",
    "Medical Research Council (MRC)": "United Kingdom, International",
    "Economic and Social Research Council (ESRC)": "United Kingdom",
    "NC3RS": "United Kingdom, Global",
    "The Health Foundation (HF)": "United Kingdom",
    "The British Heart Foundation (BHF)": "United Kingdom",
    "Association of Medical Research Charities (AMRC)": "United Kingdom"
}

trust_df["States/Country/Region Covered"] = trust_df["Funding Name"].map(region_map)

In [99]:
trust_df[[
    "Funding Name",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Status"
]]

,Funding Name,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Status
0,Innovate UK,Funding Organisation,"UK registered businesses, innovators and organisations developing innovative solutions",Multiple funding calls,United Kingdom,Multiple grants available,Active
1,Biotechnology and Biological Sciences Research Council (BBSRC),Funding Organisation,"Researchers, universities and eligible research organisations",Multiple funding calls,United Kingdom,Multiple grants available,Active
2,BBSRC standard research grant,Grant,Researchers and eligible research organisations conducting biological research,Multiple funding calls,United Kingdom,Multiple grants available,Active
3,National Institute for Health and Care Research (NIHR),Funding Organisation,"Researchers, healthcare professionals and research organisations",Multiple funding calls,United Kingdom,Multiple grants available,Active
4,UK Defence Innovation (UKDI),Funding Organisation,UK innovators and organisations working on defence and security innovation,Multiple funding calls,United Kingdom,Multiple grants available,Active
5,Medical Research Council (MRC),Funding Organisation,Researchers and research organisations conducting health and biomedical research,Multiple funding calls,"United Kingdom, International",Multiple grants available,Active
6,Economic and Social Research Council (ESRC),Funding Organisation,Researchers and organisations conducting social science research,Multiple funding calls,United Kingdom,Multiple grants available,Active
7,NC3RS,Funding Organisation,Researchers and organisations involved in medical research and animal testing alternatives,Multiple funding calls,"United Kingdom, Global",Multiple grants available,Active
8,The Health Foundation (HF),Charity Funder,"Health and care researchers, organisations and innovators",Multiple funding calls,United Kingdom,Multiple grants available,Active
9,The British Heart Foundation (BHF),Charity Funder,Clinical and non-clinical cardiovascular researchers,Multiple funding calls,United Kingdom,Multiple grants available,Active


In [100]:
# Update Deadline and Funding Amount
trust_df["Deadline"] = "Varies by funding call"
trust_df["Funding Amount"] = "Varies by programme/grant"
trust_df["Status"] = "Active"


# Update Eligibility
eligibility_map = {
    "Innovate UK": "UK registered businesses, innovators, and organisations developing innovative products/services",
    "Biotechnology and Biological Sciences Research Council (BBSRC)": "Researchers and eligible research organisations conducting biological and bioscience research",
    "BBSRC standard research grant": "Researchers based at eligible UK research organisations",
    "National Institute for Health and Care Research (NIHR)": "Researchers, healthcare professionals, universities, NHS organisations and research organisations",
    "UK Defence Innovation (UKDI)": "UK innovators and organisations developing defence and security innovation",
    "Medical Research Council (MRC)": "Researchers based at eligible organisations and research teams working in biomedical research",
    "Economic and Social Research Council (ESRC)": "Researchers and eligible research organisations working in social science research",
    "NC3RS": "Researchers and eligible research establishments working on replacing, reducing and refining animal research",
    "The Health Foundation (HF)": "Health and care researchers, organisations and innovators improving healthcare",
    "The British Heart Foundation (BHF)": "Cardiovascular researchers and eligible research institutions",
    "Association of Medical Research Charities (AMRC)": "Medical research charities and organisations involved in health research"
}

trust_df["Eligibility"] = trust_df["Funding Name"].map(eligibility_map)


# Update Regions
region_map = {
    "Innovate UK": "United Kingdom",
    "Biotechnology and Biological Sciences Research Council (BBSRC)": "United Kingdom",
    "BBSRC standard research grant": "United Kingdom",
    "National Institute for Health and Care Research (NIHR)": "United Kingdom",
    "UK Defence Innovation (UKDI)": "United Kingdom",
    "Medical Research Council (MRC)": "United Kingdom, International",
    "Economic and Social Research Council (ESRC)": "United Kingdom, International collaborations",
    "NC3RS": "United Kingdom, Global research community",
    "The Health Foundation (HF)": "United Kingdom",
    "The British Heart Foundation (BHF)": "United Kingdom",
    "Association of Medical Research Charities (AMRC)": "United Kingdom"
}

trust_df["States/Country/Region Covered"] = trust_df["Funding Name"].map(region_map)


# Special case: BBSRC Standard Research Grant
trust_df.loc[
    trust_df["Funding Name"] == "BBSRC standard research grant",
    "Deadline"
] = "Open / varies by opportunity"

trust_df.loc[
    trust_df["Funding Name"] == "BBSRC standard research grant",
    "Funding Amount"
] = "Up to £2 million"

trust_df.loc[
    trust_df["Funding Name"] == "BBSRC standard research grant",
    "Status"
] = "Open"

In [101]:
trust_df[[
    "Funding Name",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Status"
]]

,Funding Name,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Status
0,Innovate UK,Funding Organisation,"UK registered businesses, innovators, and organisations developing innovative products/services",Varies by funding call,United Kingdom,Varies by programme/grant,Active
1,Biotechnology and Biological Sciences Research Council (BBSRC),Funding Organisation,Researchers and eligible research organisations conducting biological and bioscience research,Varies by funding call,United Kingdom,Varies by programme/grant,Active
2,BBSRC standard research grant,Grant,Researchers based at eligible UK research organisations,Open / varies by opportunity,United Kingdom,Up to £2 million,Open
3,National Institute for Health and Care Research (NIHR),Funding Organisation,"Researchers, healthcare professionals, universities, NHS organisations and research organisations",Varies by funding call,United Kingdom,Varies by programme/grant,Active
4,UK Defence Innovation (UKDI),Funding Organisation,UK innovators and organisations developing defence and security innovation,Varies by funding call,United Kingdom,Varies by programme/grant,Active
5,Medical Research Council (MRC),Funding Organisation,Researchers based at eligible organisations and research teams working in biomedical research,Varies by funding call,"United Kingdom, International",Varies by programme/grant,Active
6,Economic and Social Research Council (ESRC),Funding Organisation,Researchers and eligible research organisations working in social science research,Varies by funding call,"United Kingdom, International collaborations",Varies by programme/grant,Active
7,NC3RS,Funding Organisation,"Researchers and eligible research establishments working on replacing, reducing and refining animal research",Varies by funding call,"United Kingdom, Global research community",Varies by programme/grant,Active
8,The Health Foundation (HF),Charity Funder,"Health and care researchers, organisations and innovators improving healthcare",Varies by funding call,United Kingdom,Varies by programme/grant,Active
9,The British Heart Foundation (BHF),Charity Funder,Cardiovascular researchers and eligible research institutions,Varies by funding call,United Kingdom,Varies by programme/grant,Active


In [102]:
health_df = pd.concat([df, trust_df], ignore_index=True)

In [103]:
health_df.shape

(17, 10)

In [104]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

health_df

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,"Horizon Europe 2026-2027: Health, Cluster 1","The indicative amount of EU funding contributions for projects in this cluster is mostly between €1.5m and €10m. This can cover up to 100% of costs depending on the call. The European Commission has published all of the 2026-27 Horizon Europe Work Programmes, which list the funding opportunities under the 2026-27 call topics.",Funding Opportunity,Not stated,"various (inc. September 2026, April 2027, and September 2027.",Europe,"€1.5m, €10m",https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Upcoming
1,UK-Switzerland CR&D Round 3,UK registered businesses can apply for a share of up to £3 million for innovative projects in specific technology areas. You must collaborate with at least one Swiss implementation partner applying under the equivalent Swiss Innosuisse programme.,Funding Opportunity,UK registered businesses can apply for a share of up to £3 million for innovative projects in specific technology areas. You must collaborate with at least one Swiss implementation partner applying under the equivalent Swiss Innosuisse programme,"03 September 2026, at 1100am","Switzerland, United Kingdom",£3 million,https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Upcoming
2,Digital Catapult: Digital Twin Adoption Accelerator 2026,"The Digital Twin Adoption Accelerator Programme 2026 is a nine-month programme that supports partnerships between UK technology SMEs and industry adopters to accelerate the adoption of digital twin technologies. Successful projects will receive up to £100,000 in Innovate UK grant funding, alongside technical expertise, mentoring and access to specialist facilities to help accelerate the development and deployment of their solutions.",Accelerator,The Digital Twin Adoption Accelerator Programme 2026 is a nine-month programme that supports partnerships between UK technology SMEs and industry adopters to accelerate the adoption of digital twin technologies,"06 September 2026, at 1159pm",United Kingdom,"£100,000",https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Upcoming
3,EIT Health Innovation Uptake Call 2026,"EIT Health is calling for mature digital, data-driven and AI-powered healthcare solutions ready to scale across European markets: up to €650k per project. Maximum funding per project: €650k (up to 8 projects). Funding covers up to 50% of total project costs (co-funding required). SMEs leading projects receive €400k to €500k within the grant.",Call for Proposals,SMEs leading projects receive €400k to €500k within the grant,"16 September 2026, at 300pm",Europe,"€650k, €650k, €400k, €500k",https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Upcoming
4,IHI (Innovative Health Initiative) 13th Call for Proposals,"Topics on accelerating healthcare innovation through networks, reducing animal use in drug safety studies, and decoding the immuno-science of diseases affected by ageing and the immune system. Maximum contribution of €9m-€35m depending on call. A percentage of costs must be provided by contributions (in-kind or financial) from private members which are members of IHI JU, their constituent or affiliated entities, and contributing partners.",Call for Proposals,Not stated,"08 October 2026, at 500pm (GMT+1) Brussels time",Europe,"€9m, €35m",https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Upcoming
5,UKRI Translation: MRC Impact Acceleration Awards,"This funding opportunity was formerly known as ‘Gap Fund: single-step support for medical product development’. £50k–£300k for a single high-risk translational step (data generation) on a new/repurposed medicine, device or diagnostic. Led by MRC-eligible r

In [105]:
extra_health = pd.DataFrame([
    {
        "Funding Name": "Wellcome Discovery Awards",
        "Type": "Grant",
        "Eligibility": "Researchers and research teams based at eligible organisations conducting health-related research",
        "Deadline": "Varies by funding round",
        "States/Country/Region Covered": "UK, Republic of Ireland, Low- and Middle-Income Countries",
        "Funding Amount": "Varies by award",
        "Website": "Wellcome funding opportunities",
        "Category": "Health Innovation",
        "Status": "Active"
    },
    {
        "Funding Name": "Wellcome Career Development Awards",
        "Type": "Grant",
        "Eligibility": "Researchers developing independent research careers through health-related research",
        "Deadline": "Varies by funding round",
        "States/Country/Region Covered": "UK, Republic of Ireland, Low- and Middle-Income Countries",
        "Funding Amount": "Varies by award",
        "Website": "Wellcome funding opportunities",
        "Category": "Health Innovation",
        "Status": "Active"
    },
    {
        "Funding Name": "Gates Foundation Global Health Grants",
        "Type": "Grant",
        "Eligibility": "Organisations and partners working on global health challenges; some opportunities are invitation-based",
        "Deadline": "Varies by opportunity",
        "States/Country/Region Covered": "Global, with focus on Low- and Middle-Income Countries",
        "Funding Amount": "Varies by grant",
        "Website": "Gates Foundation Grants",
        "Category": "Health Innovation",
        "Status": "Active"
    },
    {
        "Funding Name": "Grand Challenges Global Health Grants",
        "Type": "Grant",
        "Eligibility": "Researchers, innovators and organisations developing solutions to global health challenges",
        "Deadline": "Varies by challenge",
        "States/Country/Region Covered": "Global",
        "Funding Amount": "Varies by challenge",
        "Website": "Grand Challenges",
        "Category": "Health Innovation",
        "Status": "Active"
    },
    {
        "Funding Name": "European Innovation Council (EIC) Accelerator",
        "Type": "Grant/Investment",
        "Eligibility": "Startups and SMEs developing breakthrough innovations with commercial potential",
        "Deadline": "Multiple deadlines depending on call",
        "States/Country/Region Covered": "European Union and Horizon Europe associated countries",
        "Funding Amount": "Grant and equity support; varies by project",
        "Website": "European Innovation Council",
        "Category": "Health Innovation",
        "Status": "Active"
    },
    {
        "Funding Name": "Global Health Innovative Technology (GHIT) Fund",
        "Type": "Grant",
        "Eligibility": "Organisations developing health technologies including drugs, vaccines and diagnostics through eligible partnerships",
        "Deadline": "Varies by funding opportunity",
        "States/Country/Region Covered": "Global, especially Low- and Middle-Income Countries",
        "Funding Amount": "Varies by project",
        "Website": "GHIT Fund",
        "Category": "Health Innovation",
        "Status": "Active"
    }
])

In [106]:
health_df = pd.concat([health_df, extra_health], ignore_index=True)

In [107]:
health_df.shape

(23, 10)

In [108]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

health_df

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,"Horizon Europe 2026-2027: Health, Cluster 1","The indicative amount of EU funding contributions for projects in this cluster is mostly between €1.5m and €10m. This can cover up to 100% of costs depending on the call. The European Commission has published all of the 2026-27 Horizon Europe Work Programmes, which list the funding opportunities under the 2026-27 call topics.",Funding Opportunity,Not stated,"various (inc. September 2026, April 2027, and September 2027.",Europe,"€1.5m, €10m",https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Upcoming
1,UK-Switzerland CR&D Round 3,UK registered businesses can apply for a share of up to £3 million for innovative projects in specific technology areas. You must collaborate with at least one Swiss implementation partner applying under the equivalent Swiss Innosuisse programme.,Funding Opportunity,UK registered businesses can apply for a share of up to £3 million for innovative projects in specific technology areas. You must collaborate with at least one Swiss implementation partner applying under the equivalent Swiss Innosuisse programme,"03 September 2026, at 1100am","Switzerland, United Kingdom",£3 million,https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Upcoming
2,Digital Catapult: Digital Twin Adoption Accelerator 2026,"The Digital Twin Adoption Accelerator Programme 2026 is a nine-month programme that supports partnerships between UK technology SMEs and industry adopters to accelerate the adoption of digital twin technologies. Successful projects will receive up to £100,000 in Innovate UK grant funding, alongside technical expertise, mentoring and access to specialist facilities to help accelerate the development and deployment of their solutions.",Accelerator,The Digital Twin Adoption Accelerator Programme 2026 is a nine-month programme that supports partnerships between UK technology SMEs and industry adopters to accelerate the adoption of digital twin technologies,"06 September 2026, at 1159pm",United Kingdom,"£100,000",https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Upcoming
3,EIT Health Innovation Uptake Call 2026,"EIT Health is calling for mature digital, data-driven and AI-powered healthcare solutions ready to scale across European markets: up to €650k per project. Maximum funding per project: €650k (up to 8 projects). Funding covers up to 50% of total project costs (co-funding required). SMEs leading projects receive €400k to €500k within the grant.",Call for Proposals,SMEs leading projects receive €400k to €500k within the grant,"16 September 2026, at 300pm",Europe,"€650k, €650k, €400k, €500k",https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Upcoming
4,IHI (Innovative Health Initiative) 13th Call for Proposals,"Topics on accelerating healthcare innovation through networks, reducing animal use in drug safety studies, and decoding the immuno-science of diseases affected by ageing and the immune system. Maximum contribution of €9m-€35m depending on call. A percentage of costs must be provided by contributions (in-kind or financial) from private members which are members of IHI JU, their constituent or affiliated entities, and contributing partners.",Call for Proposals,Not stated,"08 October 2026, at 500pm (GMT+1) Brussels time",Europe,"€9m, €35m",https://healthinnovationnetwork.com/news/health-innovation-funding-opportunities/,Health Innovation,Upcoming
5,UKRI Translation: MRC Impact Acceleration Awards,"This funding opportunity was formerly known as ‘Gap Fund: single-step support for medical product development’. £50k–£300k for a single high-risk translational step (data generation) on a new/repurposed medicine, device or diagnostic. Led by MRC-eligible r

In [109]:
descriptions = {
    "Wellcome Discovery Awards": "Supports bold, innovative research ideas that improve understanding of life, health and wellbeing.",
    "Wellcome Career Development Awards": "Supports researchers developing independent careers while conducting research that improves human health.",
    "Gates Foundation Global Health Grants": "Supports projects addressing major global health challenges including infectious diseases, health systems and health technologies.",
    "Grand Challenges Global Health Grants": "Supports innovators developing transformative solutions to global health problems.",
    "European Innovation Council (EIC) Accelerator": "Supports startups and SMEs developing breakthrough innovations with potential for global impact.",
    "Global Health Innovative Technology (GHIT) Fund": "Supports development of health technologies including medicines, vaccines and diagnostics for diseases affecting low- and middle-income countries."
}

health_df["Description"] = health_df.apply(
    lambda row: descriptions.get(row["Funding Name"], row["Description"]),
    axis=1
)

In [110]:
website_updates = {
    "Horizon Europe 2026-2027: Health, Cluster 1": "European Commission - Horizon Europe",
    "UK-Switzerland CR&D Round 3": "Innovate UK Funding Competitions",
    "Digital Catapult: Digital Twin Adoption Accelerator 2026": "Digital Catapult Opportunities",
    "EIT Health Innovation Uptake Call 2026": "EIT Health Innovation Uptake Call",
    "IHI (Innovative Health Initiative) 13th Call for Proposals": "Innovative Health Initiative (IHI)",
    "UKRI Translation: MRC Impact Acceleration Awards": "UKRI / Medical Research Council Funding",
    "Innovate UK": "Innovate UK Funding Competitions",
    "Biotechnology and Biological Sciences Research Council (BBSRC)": "BBSRC Funding Opportunities",
    "BBSRC standard research grant": "BBSRC Standard Research Grant",
    "National Institute for Health and Care Research (NIHR)": "NIHR Funding Opportunities",
    "UK Defence Innovation (UKDI)": "UK Defence Innovation",
    "Medical Research Council (MRC)": "Medical Research Council Funding",
    "Economic and Social Research Council (ESRC)": "ESRC Funding Opportunities",
    "NC3RS": "NC3Rs Funding Opportunities",
    "The Health Foundation (HF)": "The Health Foundation Funding",
    "The British Heart Foundation (BHF)": "British Heart Foundation Research Grants",
    "Association of Medical Research Charities (AMRC)": "AMRC Member Charities",
    "Wellcome Discovery Awards": "Wellcome Discovery Awards",
    "Wellcome Career Development Awards": "Wellcome Research Funding",
    "Gates Foundation Global Health Grants": "Gates Foundation Funding",
    "Grand Challenges Global Health Grants": "Grand Challenges Funding Opportunities",
    "European Innovation Council (EIC) Accelerator": "European Innovation Council Funding",
    "Global Health Innovative Technology (GHIT) Fund": "GHIT Fund Funding Opportunities"
}

health_df["Website"] = health_df.apply(
    lambda row: website_updates.get(row["Funding Name"], row["Website"]),
    axis=1
)

In [111]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

health_df

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,"Horizon Europe 2026-2027: Health, Cluster 1","The indicative amount of EU funding contributions for projects in this cluster is mostly between €1.5m and €10m. This can cover up to 100% of costs depending on the call. The European Commission has published all of the 2026-27 Horizon Europe Work Programmes, which list the funding opportunities under the 2026-27 call topics.",Funding Opportunity,Not stated,"various (inc. September 2026, April 2027, and September 2027.",Europe,"€1.5m, €10m",European Commission - Horizon Europe,Health Innovation,Upcoming
1,UK-Switzerland CR&D Round 3,UK registered businesses can apply for a share of up to £3 million for innovative projects in specific technology areas. You must collaborate with at least one Swiss implementation partner applying under the equivalent Swiss Innosuisse programme.,Funding Opportunity,UK registered businesses can apply for a share of up to £3 million for innovative projects in specific technology areas. You must collaborate with at least one Swiss implementation partner applying under the equivalent Swiss Innosuisse programme,"03 September 2026, at 1100am","Switzerland, United Kingdom",£3 million,Innovate UK Funding Competitions,Health Innovation,Upcoming
2,Digital Catapult: Digital Twin Adoption Accelerator 2026,"The Digital Twin Adoption Accelerator Programme 2026 is a nine-month programme that supports partnerships between UK technology SMEs and industry adopters to accelerate the adoption of digital twin technologies. Successful projects will receive up to £100,000 in Innovate UK grant funding, alongside technical expertise, mentoring and access to specialist facilities to help accelerate the development and deployment of their solutions.",Accelerator,The Digital Twin Adoption Accelerator Programme 2026 is a nine-month programme that supports partnerships between UK technology SMEs and industry adopters to accelerate the adoption of digital twin technologies,"06 September 2026, at 1159pm",United Kingdom,"£100,000",Digital Catapult Opportunities,Health Innovation,Upcoming
3,EIT Health Innovation Uptake Call 2026,"EIT Health is calling for mature digital, data-driven and AI-powered healthcare solutions ready to scale across European markets: up to €650k per project. Maximum funding per project: €650k (up to 8 projects). Funding covers up to 50% of total project costs (co-funding required). SMEs leading projects receive €400k to €500k within the grant.",Call for Proposals,SMEs leading projects receive €400k to €500k within the grant,"16 September 2026, at 300pm",Europe,"€650k, €650k, €400k, €500k",EIT Health Innovation Uptake Call,Health Innovation,Upcoming
4,IHI (Innovative Health Initiative) 13th Call for Proposals,"Topics on accelerating healthcare innovation through networks, reducing animal use in drug safety studies, and decoding the immuno-science of diseases affected by ageing and the immune system. Maximum contribution of €9m-€35m depending on call. A percentage of costs must be provided by contributions (in-kind or financial) from private members which are members of IHI JU, their constituent or affiliated entities, and contributing partners.",Call for Proposals,Not stated,"08 October 2026, at 500pm (GMT+1) Brussels time",Europe,"€9m, €35m",Innovative Health Initiative (IHI),Health Innovation,Upcoming
5,UKRI Translation: MRC Impact Acceleration Awards,"This funding opportunity was formerly known as ‘Gap Fund: single-step support for medical product development’. £50k–£300k for a single high-risk translational step (data generation) on a new/repurposed medicine, device or diagnostic. Led by MRC-eligible research organisations; health SMEs typically participate as commercial/development partners rather than lead applicants.",Award,Led by MRC-eligible research organisations; health SMEs typically participate as commercial/development partners

In [112]:
website_urls = {
    "Horizon Europe 2026-2027: Health, Cluster 1": "https://commission.europa.eu/funding-and-tenders/find-funding/eu-funding-programmes/horizon-europe_en",

    "UK-Switzerland CR&D Round 3": "https://www.ukri.org/opportunity/",

    "Digital Catapult: Digital Twin Adoption Accelerator 2026": "https://www.digicatapult.org.uk/apply/opportunities/opportunity/digital-twin-adoption-accelerator-2026/",

    "EIT Health Innovation Uptake Call 2026": "https://eithealth.eu/opportunity/innovation-uptake/",

    "IHI (Innovative Health Initiative) 13th Call for Proposals": "https://www.ihi.europa.eu/apply-funding",

    "UKRI Translation: MRC Impact Acceleration Awards": "https://www.ukri.org/opportunity/",

    "Innovate UK": "https://www.ukri.org/councils/innovate-uk/",

    "Biotechnology and Biological Sciences Research Council (BBSRC)": "https://www.ukri.org/councils/bbsrc/",

    "BBSRC standard research grant": "https://www.ukri.org/opportunity/bbsrc-standard-research-grant/",

    "National Institute for Health and Care Research (NIHR)": "https://www.nihr.ac.uk/funding/",

    "UK Defence Innovation (UKDI)": "https://www.gov.uk/government/organisations/uk-defence-innovation",

    "Medical Research Council (MRC)": "https://www.ukri.org/councils/mrc/",

    "Economic and Social Research Council (ESRC)": "https://www.ukri.org/councils/esrc/",

    "NC3RS": "https://www.nc3rs.org.uk/funding",

    "The Health Foundation (HF)": "https://www.health.org.uk/funding",

    "The British Heart Foundation (BHF)": "https://www.bhf.org.uk/researchers",

    "Association of Medical Research Charities (AMRC)": "https://www.amrc.org.uk/",

    "Wellcome Discovery Awards": "https://wellcome.org/research-funding/schemes/wellcome-discovery-awards",

    "Wellcome Career Development Awards": "https://wellcome.org/research-funding",

    "Gates Foundation Global Health Grants": "https://www.gatesfoundation.org/about/how-we-work/grant-opportunities",

    "Grand Challenges Global Health Grants": "https://grandchallenges.org/grants",

    "European Innovation Council (EIC) Accelerator": "https://eic.ec.europa.eu/eic-funding-opportunities/eic-accelerator_en",

    "Global Health Innovative Technology (GHIT) Fund": "https://www.ghitfund.org/applyforfunding/opportunities"
}


health_df["Website"] = health_df["Funding Name"].map(website_urls)

In [113]:
health_df[["Funding Name", "Website"]]

,Funding Name,Website
0,"Horizon Europe 2026-2027: Health, Cluster 1",https://commission.europa.eu/funding-and-tenders/find-funding/eu-funding-programmes/horizon-europe_en
1,UK-Switzerland CR&D Round 3,https://www.ukri.org/opportunity/
2,Digital Catapult: Digital Twin Adoption Accelerator 2026,https://www.digicatapult.org.uk/apply/opportunities/opportunity/digital-twin-adoption-accelerator-2026/
3,EIT Health Innovation Uptake Call 2026,https://eithealth.eu/opportunity/innovation-uptake/
4,IHI (Innovative Health Initiative) 13th Call for Proposals,https://www.ihi.europa.eu/apply-funding
5,UKRI Translation: MRC Impact Acceleration Awards,https://www.ukri.org/opportunity/
6,Innovate UK,https://www.ukri.org/councils/innovate-uk/
7,Biotechnology and Biological Sciences Research Council (BBSRC),https://www.ukri.org/councils/bbsrc/
8,BBSRC standard research grant,https://www.ukri.org/opportunity/bbsrc-standard-research-grant/
9,National Institute for Health and Care Research (NIHR),https://www.nihr.ac.uk/funding/


In [114]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

health_df

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,"Horizon Europe 2026-2027: Health, Cluster 1","The indicative amount of EU funding contributions for projects in this cluster is mostly between €1.5m and €10m. This can cover up to 100% of costs depending on the call. The European Commission has published all of the 2026-27 Horizon Europe Work Programmes, which list the funding opportunities under the 2026-27 call topics.",Funding Opportunity,Not stated,"various (inc. September 2026, April 2027, and September 2027.",Europe,"€1.5m, €10m",https://commission.europa.eu/funding-and-tenders/find-funding/eu-funding-programmes/horizon-europe_en,Health Innovation,Upcoming
1,UK-Switzerland CR&D Round 3,UK registered businesses can apply for a share of up to £3 million for innovative projects in specific technology areas. You must collaborate with at least one Swiss implementation partner applying under the equivalent Swiss Innosuisse programme.,Funding Opportunity,UK registered businesses can apply for a share of up to £3 million for innovative projects in specific technology areas. You must collaborate with at least one Swiss implementation partner applying under the equivalent Swiss Innosuisse programme,"03 September 2026, at 1100am","Switzerland, United Kingdom",£3 million,https://www.ukri.org/opportunity/,Health Innovation,Upcoming
2,Digital Catapult: Digital Twin Adoption Accelerator 2026,"The Digital Twin Adoption Accelerator Programme 2026 is a nine-month programme that supports partnerships between UK technology SMEs and industry adopters to accelerate the adoption of digital twin technologies. Successful projects will receive up to £100,000 in Innovate UK grant funding, alongside technical expertise, mentoring and access to specialist facilities to help accelerate the development and deployment of their solutions.",Accelerator,The Digital Twin Adoption Accelerator Programme 2026 is a nine-month programme that supports partnerships between UK technology SMEs and industry adopters to accelerate the adoption of digital twin technologies,"06 September 2026, at 1159pm",United Kingdom,"£100,000",https://www.digicatapult.org.uk/apply/opportunities/opportunity/digital-twin-adoption-accelerator-2026/,Health Innovation,Upcoming
3,EIT Health Innovation Uptake Call 2026,"EIT Health is calling for mature digital, data-driven and AI-powered healthcare solutions ready to scale across European markets: up to €650k per project. Maximum funding per project: €650k (up to 8 projects). Funding covers up to 50% of total project costs (co-funding required). SMEs leading projects receive €400k to €500k within the grant.",Call for Proposals,SMEs leading projects receive €400k to €500k within the grant,"16 September 2026, at 300pm",Europe,"€650k, €650k, €400k, €500k",https://eithealth.eu/opportunity/innovation-uptake/,Health Innovation,Upcoming
4,IHI (Innovative Health Initiative) 13th Call for Proposals,"Topics on accelerating healthcare innovation through networks, reducing animal use in drug safety studies, and decoding the immuno-science of diseases affected by ageing and the immune system. Maximum contribution of €9m-€35m depending on call. A percentage of costs must be provided by contributions (in-kind or financial) from private members which are members of IHI JU, their constituent or affiliated entities, and contributing partners.",Call for Proposals,Not stated,"08 October 2026, at 500pm (GMT+1) Brussels time",Europe,"€9m, €35m",https://www.ihi.europa.eu/apply-funding,Health Innovation,Upcoming
5,UKRI Translation: MRC Impact Acceleration Awards,"This funding opportunity was formerly known as ‘Gap Fund: single-step support for medical product development’. £50k–£300k for a single high-risk translational step (data generation) on a new/repurposed medicine, device or diagnostic. Led by MRC-eligible research organisations; health SMEs typically participate as commercial/develop

In [115]:
# Fix verified missing/incomplete information in health_df

updates = {
    "Horizon Europe 2026-2027: Health, Cluster 1": {
        "Type": "Grant",
        "Eligibility": "Researchers, universities, research organisations, companies and other eligible entities in EU Member States and Horizon Europe associated countries",
    },

    "UK-Switzerland CR&D Round 3": {
        "Type": "Grant",
        "Eligibility": "UK registered businesses collaborating with at least one Swiss implementation partner under the equivalent Swiss Innosuisse programme",
    },

    "Digital Catapult: Digital Twin Adoption Accelerator 2026": {
        "Eligibility": "UK technology SMEs partnering with industry adopters to develop and deploy digital twin technologies",
    },

    "EIT Health Innovation Uptake Call 2026": {
        "Eligibility": "Digital health SMEs and organisations developing mature data-driven and AI-powered healthcare solutions ready for scaling",
        "Funding Amount": "Up to €650,000 per project; SME-led projects may receive €400,000–€500,000",
    },

    "IHI (Innovative Health Initiative) 13th Call for Proposals": {
        "Eligibility": "Healthcare companies, academic institutions, research organisations, patient organisations and other eligible IHI partners",
    },

    "UKRI Translation: MRC Impact Acceleration Awards": {
        "Eligibility": "MRC-eligible research organisations leading translational research projects; SMEs may participate as development partners",
    }
}


for name, values in updates.items():
    for column, value in values.items():
        health_df.loc[
            health_df["Funding Name"] == name,
            column
        ] = value

In [116]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

health_df

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,"Horizon Europe 2026-2027: Health, Cluster 1","The indicative amount of EU funding contributions for projects in this cluster is mostly between €1.5m and €10m. This can cover up to 100% of costs depending on the call. The European Commission has published all of the 2026-27 Horizon Europe Work Programmes, which list the funding opportunities under the 2026-27 call topics.",Grant,"Researchers, universities, research organisations, companies and other eligible entities in EU Member States and Horizon Europe associated countries","various (inc. September 2026, April 2027, and September 2027.",Europe,"€1.5m, €10m",https://commission.europa.eu/funding-and-tenders/find-funding/eu-funding-programmes/horizon-europe_en,Health Innovation,Upcoming
1,UK-Switzerland CR&D Round 3,UK registered businesses can apply for a share of up to £3 million for innovative projects in specific technology areas. You must collaborate with at least one Swiss implementation partner applying under the equivalent Swiss Innosuisse programme.,Grant,UK registered businesses collaborating with at least one Swiss implementation partner under the equivalent Swiss Innosuisse programme,"03 September 2026, at 1100am","Switzerland, United Kingdom",£3 million,https://www.ukri.org/opportunity/,Health Innovation,Upcoming
2,Digital Catapult: Digital Twin Adoption Accelerator 2026,"The Digital Twin Adoption Accelerator Programme 2026 is a nine-month programme that supports partnerships between UK technology SMEs and industry adopters to accelerate the adoption of digital twin technologies. Successful projects will receive up to £100,000 in Innovate UK grant funding, alongside technical expertise, mentoring and access to specialist facilities to help accelerate the development and deployment of their solutions.",Accelerator,UK technology SMEs partnering with industry adopters to develop and deploy digital twin technologies,"06 September 2026, at 1159pm",United Kingdom,"£100,000",https://www.digicatapult.org.uk/apply/opportunities/opportunity/digital-twin-adoption-accelerator-2026/,Health Innovation,Upcoming
3,EIT Health Innovation Uptake Call 2026,"EIT Health is calling for mature digital, data-driven and AI-powered healthcare solutions ready to scale across European markets: up to €650k per project. Maximum funding per project: €650k (up to 8 projects). Funding covers up to 50% of total project costs (co-funding required). SMEs leading projects receive €400k to €500k within the grant.",Call for Proposals,Digital health SMEs and organisations developing mature data-driven and AI-powered healthcare solutions ready for scaling,"16 September 2026, at 300pm",Europe,"Up to €650,000 per project; SME-led projects may receive €400,000–€500,000",https://eithealth.eu/opportunity/innovation-uptake/,Health Innovation,Upcoming
4,IHI (Innovative Health Initiative) 13th Call for Proposals,"Topics on accelerating healthcare innovation through networks, reducing animal use in drug safety studies, and decoding the immuno-science of diseases affected by ageing and the immune system. Maximum contribution of €9m-€35m depending on call. A percentage of costs must be provided by contributions (in-kind or financial) from private members which are members of IHI JU, their constituent or affiliated entities, and contributing partners.",Call for Proposals,"Healthcare companies, academic institutions, research organisations, patient organisations and other eligible IHI partners","08 October 2026, at 500pm (GMT+1) Brussels time",Europe,"€9m, €35m",https://www.ihi.europa.eu/apply-funding,Health Innovation,Upcoming
5,UKRI Translation: MRC Impact Acceleration Awards,"This funding opportunity was formerly known as ‘Gap Fund: single-step support for medical product development’. £50k–£300k for a single high-risk translational step (data generation) on a new/repurposed medicine, device or

In [117]:
health_df.shape

(23, 10)

In [118]:
health_df.isna().sum()

Funding Name                     0
Description                      0
Type                             0
Eligibility                      0
Deadline                         0
States/Country/Region Covered    0
Funding Amount                   0
Website                          0
Category                         0
Status                           0
dtype: int64

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

In [3]:
url = "https://www.lloyds.com/lloydslab"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

print(response.status_code)

200


In [4]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)

Welcome to Lloyd’s Lab - Lloyd's


In [5]:
headings = soup.find_all(["h1", "h2", "h3"])

for i, h in enumerate(headings):
    text = h.get_text(strip=True)
    if text:
        print(i, ":", text)

0 : About Lloyd's
1 : In the spotlight
2 : Our company
3 : Our people
4 : Our impact
5 : Investor relations
6 : Policyholders
7 : Market resources
8 : Our featured resources
9 : Market communications
10 : Oversight
11 : Reporting
12 : Regulatory
13 : Market Directory
14 : Services
15 : Tools
16 : Insights
17 : Featured insights
18 : News
19 : Risk Insights
20 : Innovation
21 : Education
22 : Welcome to Lloyd’s Lab
23 : Our mission statement
24 : About the Lloyd's Lab
26 : Programmes & innovation tools
27 : Accelerator alumni
28 : Events
29 : FAQs
30 : Our guide for InsurTech Start-ups
31 : Get in touch


In [6]:
links = soup.find_all("a", href=True)

for i, link in enumerate(links[:50]):   # first 50 links
    print(i, link.get_text(strip=True), "->", link["href"])

0 Skip to main content -> #mainContent
1  -> /
2 Contact -> /contact-us
3 Investors -> /about-lloyds/investor-relations
4 Careers -> /about-lloyds/careers
5 Locations -> /lloyds-around-the-world
6 Login -> https://ldc.lloyds.com/my-account
7 Explore Lloyd's -> /about-lloyds
8 Full Year Results 2025We have announced our Full Year Results for 2025. The Lloyd’s market produced strong results with profit before tax of £10.6bn.View the results -> /about-lloyds/investor-relations/financial-results/full-year-results-2025
9 Join the Lloyd’s marketJoin the best minds in the industry. Access the expertise, knowledge and insights to protect and develop your business.How to join the market -> /about-lloyds/join-lloyds-market
10 Understanding the Lloyd's market -> /about-lloyds/our-market
11 Our strategy -> /about-lloyds/our-purpose/2025-strategy
12 Our history -> /about-lloyds/history
13 The Lloyd's building -> /about-lloyds/the-lloyds-building
14 Corporation governance -> /about-lloyds/the-corpor

In [7]:
for link in links:
    text = link.get_text(" ", strip=True)
    href = link["href"]

    if any(word in text.lower() for word in [
        "programme", "programmes", "cohort", "accelerator",
        "apply", "innovation", "lab"
    ]):
        print(text)
        print(href)
        print("-" * 80)

Lloyd's Lab - the heart of innovation for insurance An award winning space dedicated to accelerating and fostering new products and solutions fit for the needs of our customers around the world. Explore Lloyd's Lab
/insights/lloyds-lab
--------------------------------------------------------------------------------
Lloyd's Lab
/insights/lloyds-lab
--------------------------------------------------------------------------------
About the Lloyd's Lab
#about
--------------------------------------------------------------------------------
Lloyd's Lab
/insights/lloyds-lab
--------------------------------------------------------------------------------


In [8]:
url = "https://startupmapafrica.com/startups/sector/healthtech"

response = requests.get(url, headers={"User-Agent":"Mozilla/5.0"})
print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")
print(soup.title.text)

200
19 Top Healthtech Startups in Africa (2026) – Tech Companies Database


In [9]:
cards = soup.find_all(["article", "div"])

print(len(cards))

197


In [10]:
print(soup.prettify()[:3000])

<!-- app/views/layouts/application.html.erb -->
<!DOCTYPE html>
<html lang="en">
 <head>
  <!-- Google tag (gtag.js) -->
  <script async="" src="https://www.googletagmanager.com/gtag/js?id=G-H8W80REDM2">
  </script>
  <script>
   window.dataLayer = window.dataLayer || [];
      function gtag(){dataLayer.push(arguments);}
      gtag('js', new Date());

      gtag('config', 'G-H8W80REDM2');
  </script>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <title>
   19 Top Healthtech Startups in Africa (2026) – Tech Companies Database
  </title>
  <meta content="Discover the top Healthtech startups in Africa. Browse the most active african Healthtech companies by stage and fundraising status — updated 2026." name="description">
   <meta content="19 Top Healthtech Startups in Africa (2026) – Tech Companies Database · Startup Map Africa" property="og:title">
    <meta content="Discover the top Healthtech startups in Africa. Browse the most acti

In [11]:
print(soup.title.text)

19 Top Healthtech Startups in Africa (2026) – Tech Companies Database


In [12]:
text = soup.get_text(" ", strip=True)

print(text[:1500])

19 Top Healthtech Startups in Africa (2026) – Tech Companies Database Funding Events Startups Pitch Deck Analysis How It Works Pricing Log In Sign Up Healthtech Startups in Africa List Your Startup Download Full Dataset Startup Map Africa tracks the most active Healthtech startups and tech companies across the continent.
      Filter by country, stage, and fundraising status — or explore the full Africa startups database . Top Startups in Africa to Watch A curated selection of African tech companies solving critical infrastructure, financial, and logistical challenges across the continent. Everything Data Kenya Edtech & Analytics A data community bridging data literacy across Africa through mentorship, training, partnerships and events. Nova Fleet Kenya Logistics Smart fleet management for owners and drivers: tracking, alerts, leases, and payments in one dashboard. Justxpend Technologies Nigeria Fintech Spend crypto as fiat, send professional invoices, and gift friends — all directly f

In [13]:
print(len(soup.find_all("article")))
print(len(soup.find_all("div")))

0
197


In [14]:
print("HealthTech" in response.text)

True


In [15]:
for i, div in enumerate(soup.find_all("div")[:30]):
    text = div.get_text(" ", strip=True)

    if len(text) > 50:
        print(f"\n------ DIV {i} ------")
        print(text[:400])


------ DIV 0 ------
Funding Events Startups Pitch Deck Analysis How It Works Pricing Log In Sign Up

------ DIV 1 ------
Funding Events Startups Pitch Deck Analysis How It Works Pricing Log In Sign Up

------ DIV 2 ------
Healthtech Startups in Africa List Your Startup Download Full Dataset Startup Map Africa tracks the most active Healthtech startups and tech companies across the continent.
      Filter by country, stage, and fundraising status — or explore the full Africa startups database . Top Startups in Africa to Watch A curated selection of African tech companies solving critical infrastructure, financial, a

------ DIV 3 ------
Healthtech Startups in Africa List Your Startup Download Full Dataset

------ DIV 6 ------
Everything Data Kenya Edtech & Analytics A data community bridging data literacy across Africa through mentorship, training, partnerships and events. Nova Fleet Kenya Logistics Smart fleet management for owners and drivers: tracking, alerts, leases, and payments i

In [16]:
articles = soup.find_all("article")

print("Articles:", len(articles))

for i, article in enumerate(articles[:5]):
    print("\n----------------")
    print(article.get_text(" ", strip=True)[:500])

Articles: 0


In [17]:
from collections import Counter

classes = []

for div in soup.find_all("div"):
    if div.get("class"):
        classes.extend(div.get("class"))

Counter(classes).most_common(30)

[('d-flex', 54),
 ('gap-2', 31),
 ('justify-content-between', 24),
 ('mb-3', 23),
 ('border-0', 23),
 ('flex-wrap', 22),
 ('card', 22),
 ('shadow-sm', 22),
 ('card-body', 22),
 ('col-12', 21),
 ('small', 21),
 ('col-sm-6', 20),
 ('align-items-center', 14),
 ('text-muted', 12),
 ('col-md-4', 12),
 ('col-lg-4', 8),
 ('featured-card', 8),
 ('h-100', 8),
 ('p-3', 8),
 ('mb-2', 8),
 ('lh-tight', 8),
 ('fw-bold', 8),
 ('collapse', 6),
 ('col-6', 6),
 ('row', 4),
 ('col-md-2', 4),
 ('accordion-item', 4),
 ('accordion-collapse', 4),
 ('accordion-body', 4),
 ('container', 3)]

In [18]:
cards = soup.find_all("div", class_="card")

print(len(cards))

22


In [19]:
print(cards[0].prettify())

<div class="card border-0 shadow-sm mb-3 sticky-filters bg-white">
 <!-- Header row: toggle + chips + count -->
 <div class="card-body py-2">
  <div class="d-flex flex-wrap align-items-center justify-content-between gap-2">
   <button aria-controls="filtersCollapse" aria-expanded="false" class="btn btn-outline-secondary btn-sm" data-bs-target="#filtersCollapse" data-bs-toggle="collapse" type="button">
    <i class="bi bi-sliders me-1">
    </i>
    Filters
   </button>
   <!-- Chips always visible (if any) even when collapsed) -->
   <div class="d-flex flex-wrap gap-2">
   </div>
   <span class="text-muted small ms-auto">
    19 results
   </span>
  </div>
 </div>
 <!-- Collapsible body -->
 <div class="collapse" id="filtersCollapse">
  <div class="card-body pt-0">
   <form accept-charset="UTF-8" action="/startups" class="row g-3 align-items-end" id="startups-filters" method="get">
    <div class="col-12 col-md-3">
     <label class="form-label" for="q">
      Search
     </label>
    

In [20]:
for tag in cards[0].find_all():
    print(tag.name, tag.get("class"))

div ['card-body', 'py-2']
div ['d-flex', 'flex-wrap', 'align-items-center', 'justify-content-between', 'gap-2']
button ['btn', 'btn-outline-secondary', 'btn-sm']
i ['bi', 'bi-sliders', 'me-1']
div ['d-flex', 'flex-wrap', 'gap-2']
span ['text-muted', 'small', 'ms-auto']
div ['collapse']
div ['card-body', 'pt-0']
form ['row', 'g-3', 'align-items-end']
div ['col-12', 'col-md-3']
label ['form-label']
input ['form-control']
div ['col-6', 'col-md-2']
label ['form-label']
select ['form-select']
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
option None
opt

In [21]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://startupmapafrica.com/?utm_source=chatgpt.com"

response = requests.get(url)

print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")


# Find startup cards
cards = soup.find_all("div", class_="card")

print("Number of cards:", len(cards))


data = []

for card in cards:
    text = card.get_text(" ", strip=True)

    # skip filter cards
    if "Filters" in text or "19 results" in text:
        continue

    data.append({
        "Raw Text": text
    })


df = pd.DataFrame(data)

df.head()

200
Number of cards: 0


""


In [23]:
print(response.text[:1000])

<!-- app/views/layouts/application.html.erb -->
<!DOCTYPE html>
<html lang="en">
  <head>
     <!-- Google tag (gtag.js) -->
    <script async src="https://www.googletagmanager.com/gtag/js?id=G-H8W80REDM2"></script>
    <script>
      window.dataLayer = window.dataLayer || [];
      function gtag(){dataLayer.push(arguments);}
      gtag('js', new Date());

      gtag('config', 'G-H8W80REDM2');
    </script>
    
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">

    <title>Funding for African Startups — Grants, Accelerators &amp; Investors | Startup Map Africa</title>
      <meta name="description" content="Find funding open to your African startup, draft the application, and get your pitch deck scored. 61 opportunities across 8 countries, updated weekly.">
      <link rel="canonical" href="https://startupmapafrica.com/">
    

    <meta name="csrf-param" content="authenticity_token" />
<meta name="csrf-token" content="wvrj21X_djZxDqCLH

In [24]:
import requests
from bs4 import BeautifulSoup

url = "https://startupmapafrica.com/"

response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

for a in soup.find_all("a", href=True):
    text = a.get_text(" ", strip=True)
    
    if "health" in text.lower() or "startup" in text.lower():
        print(text, "---->", a["href"])

Startups ----> /startups
Startup World Cup 2026 ----> /funding/startup-world-cup-2026
Healthtech ----> /funding/sector/healthtech
Healthtech ----> /funding/healthtech/nigeria
Healthtech ----> /funding/healthtech/south-africa
Healthtech ----> /funding/healthtech/kenya
Healthtech ----> /funding/healthtech/ghana
Healthtech ----> /funding/healthtech/egypt
Healthtech ----> /funding/healthtech/rwanda
Startups by sector and city ----> /startups


In [25]:
import requests
from bs4 import BeautifulSoup

url = "https://startupmapafrica.com/funding/sector/healthtech"

response = requests.get(url)

print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)
print("Healthtech Startups" in response.text)

200
Healthtech Funding in Africa (2026) — Grants, Accelerators & Investors
False


In [26]:
import requests
from bs4 import BeautifulSoup

url = "https://startupmapafrica.com/startups"

response = requests.get(url)

print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)

print("AI" in response.text)

200
African Startups Directory (2026) — Tech Startups in Africa by Country, Sector & Stage
True


In [27]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://startupmapafrica.com/startups"

response = requests.get(url)

soup = BeautifulSoup(response.text, "html.parser")


# Find all tables
tables = soup.find_all("table")

print("Number of tables:", len(tables))

Number of tables: 2


In [28]:
table = tables[0]

rows = []

for tr in table.find_all("tr"):
    cells = [td.get_text(" ", strip=True) for td in tr.find_all(["td","th"])]
    
    if cells:
        rows.append(cells)

for row in rows[:5]:
    print(row)

['Name', 'One-liner', 'Country', 'Sector', 'Stage', 'Raising']
['024global', '024global is developing a profit engine for agricultural value chain actors to cut loss via end-to-end intergration.', 'Kenya', 'Agriculture, Aquaculture & Agritech', 'Seed', 'Actively Raising']
['10mg Health', '10mg Health builds embedded credit infrastructure that enables clinics to buy medicines now and pay later across Africa', 'Nigeria', 'Fintech', 'Seed', 'Actively Raising']
['10mg Health', '10mg Health is a Nigerian HealthTech startup providing AI-powered, collateral-free healthcare financing to clinics and pharmacies in Africa.', 'Nigeria', 'Healthtech', 'Seed', 'Open to funding']
['20thrive', '—', 'South Africa', '—', '—', 'Not Raising']


In [29]:
df = pd.DataFrame(rows[1:], columns=rows[0])

df.head()

,Name,One-liner,Country,Sector,Stage,Raising
0,024global,024global is developing a profit engine for ag...,Kenya,"Agriculture, Aquaculture & Agritech",Seed,Actively Raising
1,10mg Health,10mg Health builds embedded credit infrastruct...,Nigeria,Fintech,Seed,Actively Raising
2,10mg Health,10mg Health is a Nigerian HealthTech startup p...,Nigeria,Healthtech,Seed,Open to funding
3,20thrive,—,South Africa,—,—,Not Raising
4,25x50,25x50 offers a custom AI Chief Storytelling Of...,United States,Marketing\nResearch & Consulting,Idea,Actively Raising


In [30]:
ai_df = df[
    df["Sector"].str.contains(
        "AI|Artificial Intelligence|Analytics|Machine Learning",
        case=False,
        na=False
    )
]

ai_df

,Name,One-liner,Country,Sector,Stage,Raising


In [31]:
import requests
from bs4 import BeautifulSoup

url = "https://startupmapafrica.com/startups"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)

# find cards
cards = soup.find_all("div", class_="card")

print("Number of cards:", len(cards))

# preview first 3
for i, card in enumerate(cards[:3]):
    print("\nCARD", i)
    print(card.get_text(" ", strip=True)[:500])

200
African Startups Directory (2026) — Tech Startups in Africa by Country, Sector & Stage
Number of cards: 12

CARD 0
Filters Showing 10 of 1,043 — search to find more Search Country All Africa Algeria Angola Antarctica Bangladesh Benin Bosnia and Herzegovina Botswana Cameroon canada Canada China Congo, Democratic Republic Côte d'Ivoire Côte d’Ivoire Democratic Republic of Congo Egypt Estonia Eswatini Ethiopia France Gabon Germany Ghana Guinea India Ireland Israel Italy Japan Kazakhstan KE Kenya Lesotho Liberia Malawi Mali Malta Mauritania Mauritius Morocco Mozambique Namibia Netherlands New Zealand Nigeria Oth

CARD 1
024global Actively Raising 024global is developing a profit engine for agricultural value chain actors to cut loss via end-to-end intergration. Kenya Agriculture, Aquaculture & Agritech Seed

CARD 2
10mg Health Actively Raising 10mg Health builds embedded credit infrastructure that enables clinics to buy medicines now and pay later across Africa Nigeria Fintech Seed


In [32]:
startup_cards = cards[1:]  # remove filter card

data = []

for card in startup_cards:
    text = card.get_text(" ", strip=True)
    data.append(text)

for i, item in enumerate(data):
    print(i, ":", item[:300])

0 : 024global Actively Raising 024global is developing a profit engine for agricultural value chain actors to cut loss via end-to-end intergration. Kenya Agriculture, Aquaculture & Agritech Seed
1 : 10mg Health Actively Raising 10mg Health builds embedded credit infrastructure that enables clinics to buy medicines now and pay later across Africa Nigeria Fintech Seed
2 : 10mg Health Open to funding 10mg Health is a Nigerian HealthTech startup providing AI-powered, collateral-free healthcare financing to clinics and pharmacies in Africa. Nigeria Healthtech Seed
3 : 20thrive Not Raising — South Africa — —
4 : 25x50 Actively Raising 25x50 offers a custom AI Chief Storytelling Officer for businesses across Africa struggling to break through media bias. United States Marketing
Research & Consulting Idea
5 : 3rees Actively Raising Instant access to productivity tools, paid for on your terms. Kenya Gaming
E-commerce
Consumer Electronics & Appliances Series A
6 : 3S Money Not Raising 3S Money i

In [33]:
print(startup_cards[0].prettify()[:3000])

<div class="card border-0 shadow-sm mb-3">
 <div class="card-body">
  <div class="d-flex justify-content-between">
   <h2 class="h6 mb-1 line-clamp-1">
    <a class="text-decoration-none" href="/startups/024global">
     024global
    </a>
   </h2>
   <span class="badge bg-success-subtle text-success border">
    Actively Raising
   </span>
  </div>
  <p class="mb-2 line-clamp-2">
   024global is developing a profit engine for agricultural value chain actors to cut loss via end-to-end intergration.
  </p>
  <div class="d-flex flex-wrap gap-2 small">
   <span class="badge text-bg-light">
    Kenya
   </span>
   <span class="badge text-bg-light">
    Agriculture, Aquaculture &amp; Agritech
   </span>
   <span class="badge text-bg-light">
    Seed
   </span>
  </div>
 </div>
</div>



In [34]:
import pandas as pd

startup_cards = cards[1:]  # remove filter card

ai_data = []

for card in startup_cards:
    try:
        name = card.find("h2").get_text(strip=True)
    except:
        name = "Not stated"

    try:
        description = card.find("p").get_text(" ", strip=True)
    except:
        description = "Not stated"

    try:
        status = card.find("span", class_="badge").get_text(strip=True)
    except:
        status = "Not stated"

    badges = card.find_all("span", class_="badge")

    country = badges[1].get_text(strip=True) if len(badges) > 1 else "Not stated"
    sector = badges[2].get_text(strip=True) if len(badges) > 2 else "Not stated"
    stage = badges[3].get_text(strip=True) if len(badges) > 3 else "Not stated"

    link = card.find("a")["href"] if card.find("a") else ""

    ai_data.append({
        "Funding Name": name,
        "Description": description,
        "Type": "Startup",
        "Eligibility": "Not stated",
        "Deadline": "Not stated",
        "States/Country/Region Covered": country,
        "Funding Amount": "Not stated",
        "Website": "https://startupmapafrica.com" + link,
        "Category": sector,
        "Status": status,
        "Stage": stage
    })


ai_df = pd.DataFrame(ai_data)

ai_df

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status,Stage
0,024global,024global is developing a profit engine for ag...,Startup,Not stated,Not stated,Kenya,Not stated,https://startupmapafrica.com/startups/024global,"Agriculture, Aquaculture & Agritech",Actively Raising,Seed
1,10mg Health,10mg Health builds embedded credit infrastruct...,Startup,Not stated,Not stated,Nigeria,Not stated,https://startupmapafrica.com/startups/10mg-health,Fintech,Actively Raising,Seed
2,10mg Health,10mg Health is a Nigerian HealthTech startup p...,Startup,Not stated,Not stated,Nigeria,Not stated,https://startupmapafrica.com/startups/10mg-hea...,Healthtech,Open to funding,Seed
3,20thrive,—,Startup,Not stated,Not stated,South Africa,Not stated,https://startupmapafrica.com/startups/20thrive,—,Not Raising,—
4,25x50,25x50 offers a custom AI Chief Storytelling Of...,Startup,Not stated,Not stated,United States,Not stated,https://startupmapafrica.com/startups/25x50,Marketing\nResearch & Consulting,Actively Raising,Idea
5,3rees,"Instant access to productivity tools, paid for...",Startup,Not stated,Not stated,Kenya,Not stated,https://startupmapafrica.com/startups/3rees,Gaming\nE-commerce\nConsumer Electronics & App...,Actively Raising,Series A
6,3S Money,3S Money is a UK-headquartered fintech platfor...,Startup,Not stated,Not stated,United Kingdom,Not stated,https://startupmapafrica.com/startups/3s-money,Fintech,Not Raising,Series B
7,99.co,99.co is a Singapore-based real estate technol...,Startup,Not stated,Not stated,Singapore,Not stated,https://startupmapafrica.com/startups/99-co,PropTech,Actively Raising,Series C
8,A2V Solutions,A2V Solutions is a South African IT services p...,Startup,Not stated,Not stated,South Africa,Not stated,https://startupmapafrica.com/startups/a2v-solu...,Information Technology,Not Raising,—
9,Accounting.com,Accounting.com is a US-based leading source of...,Startup,Not stated,Not stated,United States,Not stated,https://startupmapafrica.com/startups/accounti...,Education,Not Raising,—


In [35]:
ai_df.shape

(11, 11)

In [36]:
import pandas as pd

startup_cards = cards[1:]

ai_data = []

for card in startup_cards:

    name_tag = card.find("h2")
    name = name_tag.get_text(strip=True) if name_tag else "Not stated"

    desc_tag = card.find("p")
    description = desc_tag.get_text(" ", strip=True) if desc_tag else "Not stated"

    badges = [
        b.get_text(" ", strip=True)
        for b in card.find_all("span", class_="badge")
    ]

    status = badges[0] if len(badges) > 0 else "Not stated"
    country = badges[1] if len(badges) > 1 else "Not stated"
    category = badges[2] if len(badges) > 2 else "Not stated"
    stage = badges[3] if len(badges) > 3 else "Not stated"

    link_tag = card.find("a")

    website = (
        "https://startupmapafrica.com" + link_tag["href"]
        if link_tag and link_tag.get("href")
        else "Not stated"
    )

    ai_data.append({
        "Funding Name": name,
        "Description": description,
        "Type": "Startup",
        "Eligibility": "Not stated",
        "Deadline": "Not stated",
        "States/Country/Region Covered": country,
        "Funding Amount": "Not stated",
        "Website": website,
        "Category": category,
        "Status": status,
        "Stage": stage
    })


ai_df = pd.DataFrame(ai_data)

ai_df

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status,Stage
0,024global,024global is developing a profit engine for ag...,Startup,Not stated,Not stated,Kenya,Not stated,https://startupmapafrica.com/startups/024global,"Agriculture, Aquaculture & Agritech",Actively Raising,Seed
1,10mg Health,10mg Health builds embedded credit infrastruct...,Startup,Not stated,Not stated,Nigeria,Not stated,https://startupmapafrica.com/startups/10mg-health,Fintech,Actively Raising,Seed
2,10mg Health,10mg Health is a Nigerian HealthTech startup p...,Startup,Not stated,Not stated,Nigeria,Not stated,https://startupmapafrica.com/startups/10mg-hea...,Healthtech,Open to funding,Seed
3,20thrive,—,Startup,Not stated,Not stated,South Africa,Not stated,https://startupmapafrica.com/startups/20thrive,—,Not Raising,—
4,25x50,25x50 offers a custom AI Chief Storytelling Of...,Startup,Not stated,Not stated,United States,Not stated,https://startupmapafrica.com/startups/25x50,Marketing\nResearch & Consulting,Actively Raising,Idea
5,3rees,"Instant access to productivity tools, paid for...",Startup,Not stated,Not stated,Kenya,Not stated,https://startupmapafrica.com/startups/3rees,Gaming\nE-commerce\nConsumer Electronics & App...,Actively Raising,Series A
6,3S Money,3S Money is a UK-headquartered fintech platfor...,Startup,Not stated,Not stated,United Kingdom,Not stated,https://startupmapafrica.com/startups/3s-money,Fintech,Not Raising,Series B
7,99.co,99.co is a Singapore-based real estate technol...,Startup,Not stated,Not stated,Singapore,Not stated,https://startupmapafrica.com/startups/99-co,PropTech,Actively Raising,Series C
8,A2V Solutions,A2V Solutions is a South African IT services p...,Startup,Not stated,Not stated,South Africa,Not stated,https://startupmapafrica.com/startups/a2v-solu...,Information Technology,Not Raising,—
9,Accounting.com,Accounting.com is a US-based leading source of...,Startup,Not stated,Not stated,United States,Not stated,https://startupmapafrica.com/startups/accounti...,Education,Not Raising,—


In [37]:
ai_keywords = [
    "AI",
    "Artificial Intelligence",
    "Machine Learning",
    "Data",
    "Analytics",
    "Deep Learning"
]


ai_only = ai_df[
    ai_df["Description"].str.contains(
        "|".join(ai_keywords),
        case=False,
        na=False
    )
    |
    ai_df["Category"].str.contains(
        "|".join(ai_keywords),
        case=False,
        na=False
    )
]


ai_only

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status,Stage
0,024global,024global is developing a profit engine for ag...,Startup,Not stated,Not stated,Kenya,Not stated,https://startupmapafrica.com/startups/024global,"Agriculture, Aquaculture & Agritech",Actively Raising,Seed
2,10mg Health,10mg Health is a Nigerian HealthTech startup p...,Startup,Not stated,Not stated,Nigeria,Not stated,https://startupmapafrica.com/startups/10mg-hea...,Healthtech,Open to funding,Seed
4,25x50,25x50 offers a custom AI Chief Storytelling Of...,Startup,Not stated,Not stated,United States,Not stated,https://startupmapafrica.com/startups/25x50,Marketing\nResearch & Consulting,Actively Raising,Idea
5,3rees,"Instant access to productivity tools, paid for...",Startup,Not stated,Not stated,Kenya,Not stated,https://startupmapafrica.com/startups/3rees,Gaming\nE-commerce\nConsumer Electronics & App...,Actively Raising,Series A


In [38]:
ai_keywords = [
    "AI",
    "Artificial Intelligence",
    "Machine Learning",
    "Deep Learning",
    "Data & AI",
    "AI & Analytics",
    "Analytics",
    "Robotics"
]

ai_only = ai_df[
    ai_df["Category"].str.contains(
        "|".join(ai_keywords),
        case=False,
        na=False
    )
].copy()

ai_only

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status,Stage


In [39]:
ai_only[["Funding Name", "Category", "Description"]]

,Funding Name,Category,Description


In [40]:
# find all links containing AI-related words

for a in soup.find_all("a", href=True):
    text = a.get_text(" ", strip=True)
    href = a["href"]

    if "ai" in text.lower() or "artificial" in text.lower() or "machine" in text.lower():
        print(text, "---->", href)

Hadaa Home & Garden / Real Estate · United States Live profile Hadaa: AI Landscape Design — 22 garden photos in 60 seconds View the live page → ----> /startups/hadaa
AI startups in Africa ----> /startups?sector=AI


In [41]:
url = "https://startupmapafrica.com/funding/sector/ai"

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")

print(response.status_code)
print(soup.title.text)

200
AI Funding in Africa (2026) — Grants, Accelerators & Investors


In [42]:
import requests
from bs4 import BeautifulSoup

url = "https://startupmapafrica.com/funding/sector/ai"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)


# check cards
cards = soup.find_all("div", class_="card")

print("Number of cards:", len(cards))


for i, card in enumerate(cards[:5]):
    print("\nCARD", i)
    print(card.get_text(" ", strip=True)[:500])

200
AI Funding in Africa (2026) — Grants, Accelerators & Investors
Number of cards: 2

CARD 0
AI funding by type AI Grants AI Accelerators AI Investors

CARD 1
Funding for other sectors Fintech funding Healthtech funding Logistics funding Climate Tech funding E-commerce funding Agritech funding Edtech funding SaaS funding Proptech funding Energy funding Mobility funding Insurtech funding


In [43]:
for a in soup.find_all("a", href=True):
    text = a.get_text(" ", strip=True)
    href = a["href"]

    if "ai" in href.lower() or "ai" in text.lower():
        print(text, "---->", href)

AI Funding in Africa (2026) — Grants, Accelerators & Investors ----> /users/sign_up?next=funding
AI Grants ----> /funding/grants/ai
AI Accelerators ----> /funding/accelerators/ai
AI Investors ----> /funding/investors/ai


In [44]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

headers = {
    "User-Agent": "Mozilla/5.0"
}

urls = {
    "AI Grants": "https://startupmapafrica.com/funding/grants/ai",
    "AI Accelerators": "https://startupmapafrica.com/funding/accelerators/ai",
    "AI Investors": "https://startupmapafrica.com/funding/investors/ai"
}


ai_funding = []

for category, url in urls.items():

    response = requests.get(url, headers=headers)

    print(category, response.status_code)

    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.find_all("div", class_="card")

    print("Cards:", len(cards))

    for card in cards:

        # skip filter/navigation cards
        if not card.find("h2"):
            continue

        name = card.find("h2").get_text(strip=True)

        desc_tag = card.find("p")
        description = (
            desc_tag.get_text(" ", strip=True)
            if desc_tag else "Not stated"
        )

        badges = [
            b.get_text(" ", strip=True)
            for b in card.find_all("span", class_="badge")
        ]

        status = badges[0] if len(badges) > 0 else "Not stated"
        region = badges[1] if len(badges) > 1 else "Not stated"

        link = card.find("a")

        website = (
            "https://startupmapafrica.com" + link["href"]
            if link and link.get("href")
            else "Not stated"
        )

        ai_funding.append({
            "Funding Name": name,
            "Description": description,
            "Type": category.replace("AI ", ""),
            "Eligibility": "Not stated",
            "Deadline": "Not stated",
            "States/Country/Region Covered": region,
            "Funding Amount": "Not stated",
            "Website": website,
            "Category": "AI Innovation",
            "Status": status
        })


ai_df = pd.DataFrame(ai_funding)

ai_df

AI Grants 200
Cards: 2
AI Accelerators 200
Cards: 2
AI Investors 200
Cards: 2


""


In [45]:
ai_df.shape

(0, 0)

In [46]:
ai_df.head()

""


In [47]:
url = "https://startupmapafrica.com/funding/grants/ai"

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)

for i, card in enumerate(soup.find_all("div", class_="card")):
    print("\nCARD", i)
    print(card.get_text(" ", strip=True)[:1000])

Best AI Grants for Startups (2026) — Programs & Eligibility

CARD 0
Other AI funding Accelerators Investors

CARD 1
Grants in other sectors Healthtech Grants Climate Tech Grants Agritech Grants SaaS Grants Proptech Grants Energy Grants Cleantech Grants


In [48]:
from bs4 import BeautifulSoup
import requests

url = "https://startupmapafrica.com/funding/grants/ai"

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")

for a in soup.find_all("a", href=True):
    text = a.get_text(" ", strip=True)
    href = a["href"]

    if text:
        print(text, "---->", href)

Funding ----> /funding
Events ----> /events
Startups ----> /startups
Pitch Deck Analysis ----> /pitch-deck-analysis
How It Works ----> /#how-it-works
Pricing ----> /pricing
Log In ----> /users/sign_in
Sign Up ----> /users/sign_up
Best AI Grants for Startups (2026) — Programs & Eligibility ----> /users/sign_up?next=funding
Grants ----> #grants
Accelerators ----> #accelerators
Investors ----> #investors
Investor directory → ----> /investors
Small Business Innovation Research (SBIR) and Small Business Technology Transfer (STTR) Programs (America's Seed Fund) ----> /funding/small-business-innovation-research-sbir-and-small-business-technology-transfer-sttr-programs-america-s-seed-fund
Autodesk Foundation Funding & Impact Support ----> /funding/autodesk-foundation-funding
Accelerators ----> /funding/accelerators/ai
Investors ----> /funding/investors/ai
Healthtech Grants ----> /funding/grants/healthtech
Climate Tech Grants ----> /funding/grants/climate-tech
Agritech Grants ----> /funding/gra

In [49]:
url = "https://www.nvidia.com/en-us/startups/"

response = requests.get(url, headers=headers)

print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)

print(soup.get_text(" ", strip=True)[:1000])

200
Inception Program for Startups | NVIDIA
Inception Program for Startups | NVIDIA NVIDIA Home NVIDIA Home Menu Menu icon Menu Menu icon Close Close icon Close Close icon Close Close icon Caret down icon Accordion is closed, click to open. Caret down icon Accordion is closed, click to open. Caret up icon Accordion is open, click to close. Caret right icon Click to expand Caret right icon Click to expand Caret right icon Click to expand menu. Caret left icon Click to collapse menu. Caret left icon Click to collapse menu. Caret left icon Click to collapse menu. Shopping Cart Click to see cart items Search icon Click to search Visit your regional NVIDIA website for local content, pricing, and where to buy partners specific to your country. Argentina Australia België (Belgium) Belgique (Belgium) Brasil (Brazil) Canada Česká Republika (Czech Republic) Chile Colombia Danmark (Denmark) Deutschland (Germany) España (Spain) France India Italia (Italy) México (Mexico) Middle East Nederland (Net

In [50]:
import pandas as pd

ai_grants = [
    {
        "Funding Name": "AI Fund in Collaboration with Google – NCAIR",
        "Description": "Initiative supporting Nigerian startups building AI-powered solutions through grant funding, Google AI tools, mentorship and networking opportunities.",
        "Type": "Grant",
        "Eligibility": "Companies headquartered in Nigeria with at least one Nigerian founder, building AI solutions with a live product and potential to scale.",
        "Deadline": "Closed (Applications closed September 2024)",
        "States/Country/Region Covered": "Nigeria",
        "Funding Amount": "Up to ₦10 million per startup (₦100 million total fund)",
        "Website": "https://ncair.nitda.gov.ng/aifund/",
        "Category": "AI Innovation",
        "Status": "Closed"
    },
    {
        "Funding Name": "OpenAI AI and Mental Health Research Grants",
        "Description": "Grant programme supporting research exploring the intersection of artificial intelligence and mental health.",
        "Type": "Grant",
        "Eligibility": "Researchers and organisations submitting proposals related to AI and mental health research.",
        "Deadline": "Closed (19 December 2025)",
        "States/Country/Region Covered": "Global",
        "Funding Amount": "Up to $2 million total programme funding",
        "Website": "https://grants.openai.com/",
        "Category": "AI Innovation",
        "Status": "Closed"
    },
    {
        "Funding Name": "OpenAI People-First AI Fund",
        "Description": "Grant initiative supporting nonprofit and community organisations using AI to improve education, economic opportunity, healthcare and community research.",
        "Type": "Grant",
        "Eligibility": "Eligible nonprofit and community organisations using AI for public benefit initiatives.",
        "Deadline": "Closed (2025 funding round)",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "$50 million commitment",
        "Website": "https://openai.com/index/people-first-ai-fund/",
        "Category": "AI Innovation",
        "Status": "Closed"
    }
]

ai_grants_df = pd.DataFrame(ai_grants)

ai_grants_df

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,AI Fund in Collaboration with Google – NCAIR,Initiative supporting Nigerian startups buildi...,Grant,Companies headquartered in Nigeria with at lea...,Closed (Applications closed September 2024),Nigeria,Up to ₦10 million per startup (₦100 million to...,https://ncair.nitda.gov.ng/aifund/,AI Innovation,Closed
1,OpenAI AI and Mental Health Research Grants,Grant programme supporting research exploring ...,Grant,Researchers and organisations submitting propo...,Closed (19 December 2025),Global,Up to $2 million total programme funding,https://grants.openai.com/,AI Innovation,Closed
2,OpenAI People-First AI Fund,Grant initiative supporting nonprofit and comm...,Grant,Eligible nonprofit and community organisations...,Closed (2025 funding round),United States,$50 million commitment,https://openai.com/index/people-first-ai-fund/,AI Innovation,Closed


In [51]:
ai_grants_df.shape

(3, 10)

In [52]:
import pandas as pd

ai_grants = [
    {
        "Funding Name": "AI Fund in Collaboration with Google – NCAIR",
        "Description": "Grant programme supporting Nigerian startups developing artificial intelligence solutions through funding, technical support and ecosystem opportunities.",
        "Type": "Grant",
        "Eligibility": "Nigerian startups developing AI-based solutions with eligible business registration and scalable products.",
        "Deadline": "Closed (previous round)",
        "States/Country/Region Covered": "Nigeria",
        "Funding Amount": "Up to ₦10 million per startup",
        "Website": "https://ncair.nitda.gov.ng/aifund/",
        "Category": "AI Innovation",
        "Status": "Closed"
    },

    {
        "Funding Name": "AI Upskill Accelerator Pilot Program",
        "Description": "U.S. Economic Development Administration funding supporting AI workforce training programmes and partnerships that expand AI skills.",
        "Type": "Grant",
        "Eligibility": "Eligible U.S. entities implementing industry-led AI workforce training programmes.",
        "Deadline": "10 July 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Approximately $25 million programme; awards $1 million–$8 million",
        "Website": "https://www.eda.gov/funding/funding-opportunities/ai-upskill-accelerator-pilot-program",
        "Category": "AI Innovation",
        "Status": "Open"
    },

    {
        "Funding Name": "The Genesis Mission: Transforming Science and Energy with AI",
        "Description": "U.S. Department of Energy funding supporting interdisciplinary teams using AI models and frameworks to accelerate scientific discovery.",
        "Type": "Research Grant",
        "Eligibility": "Universities, research organisations, companies and eligible entities conducting AI-related scientific research.",
        "Deadline": "17 December 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "$293.76 million total; awards $500,000–$16 million",
        "Website": "https://www.grants.gov/search-results-detail/361526",
        "Category": "AI Innovation",
        "Status": "Open"
    },

    {
        "Funding Name": "Innovate UK Frontier Artificial Intelligence Discovery",
        "Description": "Grant competition supporting feasibility studies for frontier AI, machine learning and foundation model development.",
        "Type": "Grant",
        "Eligibility": "UK businesses, research organisations, charities, public sector organisations and eligible innovation organisations.",
        "Deadline": "10 June 2026",
        "States/Country/Region Covered": "United Kingdom",
        "Funding Amount": "Up to £2.5 million total funding",
        "Website": "https://www.ukri.org/opportunity/frontier-artificial-intelligence-discovery/",
        "Category": "AI Innovation",
        "Status": "Closed"
    },

    {
        "Funding Name": "Future Computing Paradigms Network Plus",
        "Description": "Research funding supporting UK collaboration and research networks exploring future computing technologies including AI-related areas.",
        "Type": "Research Grant",
        "Eligibility": "Researchers based at UK research organisations eligible for EPSRC funding.",
        "Deadline": "13 October 2026",
        "States/Country/Region Covered": "United Kingdom",
        "Funding Amount": "£2.8 million total funding",
        "Website": "https://www.ukri.org/opportunity/",
        "Category": "AI Innovation",
        "Status": "Upcoming"
    },

    {
        "Funding Name": "AI Pathways To The Future",
        "Description": "Grant programme supporting education and cooperation initiatives focused on artificial intelligence development.",
        "Type": "Grant",
        "Eligibility": "Eligible organisations applying under the U.S. Department of State programme requirements.",
        "Deadline": "9 August 2026",
        "States/Country/Region Covered": "Indonesia",
        "Funding Amount": "Not stated",
        "Website": "https://www.grants.gov/search-results-detail/363061",
        "Category": "AI Innovation",
        "Status": "Open"
    }
]


ai_grants_df = pd.DataFrame(ai_grants)

ai_grants_df

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,AI Fund in Collaboration with Google – NCAIR,Grant programme supporting Nigerian startups d...,Grant,Nigerian startups developing AI-based solution...,Closed (previous round),Nigeria,Up to ₦10 million per startup,https://ncair.nitda.gov.ng/aifund/,AI Innovation,Closed
1,AI Upskill Accelerator Pilot Program,U.S. Economic Development Administration fundi...,Grant,Eligible U.S. entities implementing industry-l...,10 July 2026,United States,Approximately $25 million programme; awards $1...,https://www.eda.gov/funding/funding-opportunit...,AI Innovation,Open
2,The Genesis Mission: Transforming Science and ...,U.S. Department of Energy funding supporting i...,Research Grant,"Universities, research organisations, companie...",17 December 2026,United States,"$293.76 million total; awards $500,000–$16 mil...",https://www.grants.gov/search-results-detail/3...,AI Innovation,Open
3,Innovate UK Frontier Artificial Intelligence D...,Grant competition supporting feasibility studi...,Grant,"UK businesses, research organisations, chariti...",10 June 2026,United Kingdom,Up to £2.5 million total funding,https://www.ukri.org/opportunity/frontier-arti...,AI Innovation,Closed
4,Future Computing Paradigms Network Plus,Research funding supporting UK collaboration a...,Research Grant,Researchers based at UK research organisations...,13 October 2026,United Kingdom,£2.8 million total funding,https://www.ukri.org/opportunity/,AI Innovation,Upcoming
5,AI Pathways To The Future,Grant programme supporting education and coope...,Grant,Eligible organisations applying under the U.S....,9 August 2026,Indonesia,Not stated,https://www.grants.gov/search-results-detail/3...,AI Innovation,Open


In [53]:
ai_grants_df.shape

(6, 10)

In [54]:
more_ai_grants = [
    {
        "Funding Name": "Horizon Europe Digital Calls - Trustworthy Artificial Intelligence Services",
        "Description": "Horizon Europe funding supporting AI, data services, robotics and digital technologies under Cluster 4 Digital, Industry and Space.",
        "Type": "Grant",
        "Eligibility": "Research organisations, universities, companies and eligible international partners under Horizon Europe rules.",
        "Deadline": "15 April 2026",
        "States/Country/Region Covered": "European Union and Horizon Europe Associated Countries",
        "Funding Amount": "€221.8 million total across Digital call topics",
        "Website": "https://ec.europa.eu/info/funding-tenders/opportunities/portal/",
        "Category": "Artificial Intelligence",
        "Status": "Closed"
    },

    {
        "Funding Name": "Horizon Europe Next-Generation AI Agents for Real-World Applications",
        "Description": "Research and innovation funding supporting next-generation AI agents, improving autonomy, reliability and real-world AI applications.",
        "Type": "Research Grant",
        "Eligibility": "Universities, research organisations, companies and eligible Horizon Europe applicants.",
        "Deadline": "15 April 2026",
        "States/Country/Region Covered": "Europe",
        "Funding Amount": "€38 million budget",
        "Website": "https://ec.europa.eu/info/funding-tenders/opportunities/portal/",
        "Category": "Artificial Intelligence",
        "Status": "Closed"
    },

    {
        "Funding Name": "Horizon Europe Efficient and Compliant Access and Use of Data (AI, Data and Robotics Partnership)",
        "Description": "Innovation funding supporting AI-driven compliance technologies, data systems and trustworthy artificial intelligence solutions.",
        "Type": "Innovation Grant",
        "Eligibility": "Research institutions, companies and eligible Horizon Europe participants.",
        "Deadline": "15 April 2026",
        "States/Country/Region Covered": "Europe",
        "Funding Amount": "€46.5 million budget",
        "Website": "https://ec.europa.eu/info/funding-tenders/opportunities/portal/",
        "Category": "Artificial Intelligence",
        "Status": "Closed"
    },

    {
        "Funding Name": "European Innovation Council (EIC) Advanced Innovation Challenges - Physical AI",
        "Description": "Horizon Europe challenge funding supporting breakthrough innovations including AI-powered robotics and embodied intelligence.",
        "Type": "Grant",
        "Eligibility": "Startups, SMEs and research teams developing breakthrough technologies.",
        "Deadline": "Varies by challenge",
        "States/Country/Region Covered": "European Union and Horizon Europe Associated Countries",
        "Funding Amount": "Varies by challenge",
        "Website": "https://eic.ec.europa.eu/eic-funding-opportunities_en",
        "Category": "Artificial Intelligence",
        "Status": "Upcoming"
    },

    {
        "Funding Name": "National Science Foundation (NSF) Artificial Intelligence Research Funding",
        "Description": "NSF supports research projects advancing artificial intelligence, machine learning and AI applications across scientific fields.",
        "Type": "Research Grant",
        "Eligibility": "Eligible U.S. universities, research organisations and researchers.",
        "Deadline": "Varies by programme",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by NSF programme",
        "Website": "https://www.nsf.gov/funding",
        "Category": "Artificial Intelligence",
        "Status": "Active"
    },

    {
        "Funding Name": "DARPA Artificial Intelligence Research Programs",
        "Description": "DARPA funds advanced research projects developing new artificial intelligence capabilities for national security applications.",
        "Type": "Research Grant",
        "Eligibility": "Universities, companies, research institutions and eligible organisations.",
        "Deadline": "Varies by programme",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by programme",
        "Website": "https://www.darpa.mil/work-with-us",
        "Category": "Artificial Intelligence",
        "Status": "Active"
    },

    {
        "Funding Name": "Google.org Artificial Intelligence Grants",
        "Description": "Google.org provides philanthropic grants supporting organisations using AI to address social challenges.",
        "Type": "Grant",
        "Eligibility": "Nonprofits, social impact organisations and eligible initiatives.",
        "Deadline": "Varies by opportunity",
        "States/Country/Region Covered": "Global",
        "Funding Amount": "Varies by grant",
        "Website": "https://www.google.org/programs/",
        "Category": "Artificial Intelligence",
        "Status": "Active"
    }
]


more_ai_df = pd.DataFrame(more_ai_grants)

more_ai_df

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,Horizon Europe Digital Calls - Trustworthy Art...,"Horizon Europe funding supporting AI, data ser...",Grant,"Research organisations, universities, companie...",15 April 2026,European Union and Horizon Europe Associated C...,€221.8 million total across Digital call topics,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
1,Horizon Europe Next-Generation AI Agents for R...,Research and innovation funding supporting nex...,Research Grant,"Universities, research organisations, companie...",15 April 2026,Europe,€38 million budget,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
2,Horizon Europe Efficient and Compliant Access ...,Innovation funding supporting AI-driven compli...,Innovation Grant,"Research institutions, companies and eligible ...",15 April 2026,Europe,€46.5 million budget,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
3,European Innovation Council (EIC) Advanced Inn...,Horizon Europe challenge funding supporting br...,Grant,"Startups, SMEs and research teams developing b...",Varies by challenge,European Union and Horizon Europe Associated C...,Varies by challenge,https://eic.ec.europa.eu/eic-funding-opportuni...,Artificial Intelligence,Upcoming
4,National Science Foundation (NSF) Artificial I...,NSF supports research projects advancing artif...,Research Grant,"Eligible U.S. universities, research organisat...",Varies by programme,United States,Varies by NSF programme,https://www.nsf.gov/funding,Artificial Intelligence,Active
5,DARPA Artificial Intelligence Research Programs,DARPA funds advanced research projects develop...,Research Grant,"Universities, companies, research institutions...",Varies by programme,United States,Varies by programme,https://www.darpa.mil/work-with-us,Artificial Intelligence,Active
6,Google.org Artificial Intelligence Grants,Google.org provides philanthropic grants suppo...,Grant,"Nonprofits, social impact organisations and el...",Varies by opportunity,Global,Varies by grant,https://www.google.org/programs/,Artificial Intelligence,Active


In [55]:
ai_grants_final = pd.concat(
    [ai_grants_df, more_ai_df],
    ignore_index=True
)

ai_grants_final.shape

(13, 10)

In [56]:
ai_grants_batch3 = [
    {
        "Funding Name": "NSF AI Datasets for AI-Enabled Scientific Discovery",
        "Description": "Funding supporting creation, management and use of datasets that enable artificial intelligence-driven scientific discovery.",
        "Type": "Research Grant",
        "Eligibility": "Eligible US universities, research institutions and researchers.",
        "Deadline": "04 November 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by NSF award",
        "Website": "https://www.nsf.gov/funding",
        "Category": "Artificial Intelligence",
        "Status": "Open"
    },

    {
        "Funding Name": "NSF Computer and Information Science and Engineering (CISE) AI Research Funding",
        "Description": "NSF funding supporting research in artificial intelligence, machine learning, computing systems and information technologies.",
        "Type": "Research Grant",
        "Eligibility": "US-based eligible academic institutions and research organisations.",
        "Deadline": "Varies by programme",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by programme",
        "Website": "https://www.nsf.gov/funding",
        "Category": "Artificial Intelligence",
        "Status": "Active"
    },

    {
        "Funding Name": "UKRI Advancing Sustainable AI Technology",
        "Description": "Funding to develop innovative sustainable AI technologies and tools that support future AI systems.",
        "Type": "Grant",
        "Eligibility": "UK research organisations and eligible innovation partners.",
        "Deadline": "Expected future call",
        "States/Country/Region Covered": "United Kingdom",
        "Funding Amount": "£10 million total fund; maximum £500,000 award",
        "Website": "https://www.ukri.org/apply-for-funding/future-research-and-innovation-funding-opportunities/",
        "Category": "Artificial Intelligence",
        "Status": "Upcoming"
    },

    {
        "Funding Name": "UKRI Next Generation AI: Explainable AI",
        "Description": "Research funding supporting fundamental, high-risk, high-reward research into explainable artificial intelligence.",
        "Type": "Research Grant",
        "Eligibility": "Researchers based at eligible UK research organisations.",
        "Deadline": "Expected future call",
        "States/Country/Region Covered": "United Kingdom",
        "Funding Amount": "£10 million total fund; maximum £500,000 award",
        "Website": "https://www.ukri.org/apply-for-funding/future-research-and-innovation-funding-opportunities/",
        "Category": "Artificial Intelligence",
        "Status": "Upcoming"
    },

    {
        "Funding Name": "UKRI Future Computing Paradigms Network Plus",
        "Description": "Funding supporting research networks exploring future computing approaches including emerging AI technologies.",
        "Type": "Grant",
        "Eligibility": "Researchers based at UK research organisations eligible for EPSRC funding.",
        "Deadline": "13 October 2026",
        "States/Country/Region Covered": "United Kingdom",
        "Funding Amount": "£2.8 million total funding",
        "Website": "https://www.ukri.org/opportunity/future-computing-paradigms-network-plus/",
        "Category": "Artificial Intelligence",
        "Status": "Upcoming"
    },

    {
        "Funding Name": "DARPA Artificial Intelligence Exploration Program",
        "Description": "Research funding supporting innovative artificial intelligence concepts and breakthrough AI technologies.",
        "Type": "Research Grant",
        "Eligibility": "Universities, companies, research institutions and eligible organisations.",
        "Deadline": "Varies by opportunity",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by programme",
        "Website": "https://www.darpa.mil/work-with-us",
        "Category": "Artificial Intelligence",
        "Status": "Active"
    },

    {
        "Funding Name": "NIH Artificial Intelligence and Machine Learning Research Grants",
        "Description": "Health research grants supporting artificial intelligence and machine learning applications in biomedical research.",
        "Type": "Research Grant",
        "Eligibility": "Eligible universities, research institutions and biomedical researchers.",
        "Deadline": "Varies by funding announcement",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by NIH programme",
        "Website": "https://grants.nih.gov/",
        "Category": "Artificial Intelligence",
        "Status": "Active"
    }
]


ai_grants_batch3_df = pd.DataFrame(ai_grants_batch3)

ai_grants_batch3_df

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,NSF AI Datasets for AI-Enabled Scientific Disc...,"Funding supporting creation, management and us...",Research Grant,"Eligible US universities, research institution...",04 November 2026,United States,Varies by NSF award,https://www.nsf.gov/funding,Artificial Intelligence,Open
1,NSF Computer and Information Science and Engin...,NSF funding supporting research in artificial ...,Research Grant,US-based eligible academic institutions and re...,Varies by programme,United States,Varies by programme,https://www.nsf.gov/funding,Artificial Intelligence,Active
2,UKRI Advancing Sustainable AI Technology,Funding to develop innovative sustainable AI t...,Grant,UK research organisations and eligible innovat...,Expected future call,United Kingdom,"£10 million total fund; maximum £500,000 award",https://www.ukri.org/apply-for-funding/future-...,Artificial Intelligence,Upcoming
3,UKRI Next Generation AI: Explainable AI,"Research funding supporting fundamental, high-...",Research Grant,Researchers based at eligible UK research orga...,Expected future call,United Kingdom,"£10 million total fund; maximum £500,000 award",https://www.ukri.org/apply-for-funding/future-...,Artificial Intelligence,Upcoming
4,UKRI Future Computing Paradigms Network Plus,Funding supporting research networks exploring...,Grant,Researchers based at UK research organisations...,13 October 2026,United Kingdom,£2.8 million total funding,https://www.ukri.org/opportunity/future-comput...,Artificial Intelligence,Upcoming
5,DARPA Artificial Intelligence Exploration Program,Research funding supporting innovative artific...,Research Grant,"Universities, companies, research institutions...",Varies by opportunity,United States,Varies by programme,https://www.darpa.mil/work-with-us,Artificial Intelligence,Active
6,NIH Artificial Intelligence and Machine Learni...,Health research grants supporting artificial i...,Research Grant,"Eligible universities, research institutions a...",Varies by funding announcement,United States,Varies by NIH programme,https://grants.nih.gov/,Artificial Intelligence,Active


In [57]:
ai_grants_final = pd.concat(
    [ai_grants_final, ai_grants_batch3_df],
    ignore_index=True
)

ai_grants_final.shape

(20, 10)

In [58]:
ai_grants_batch4 = [
    {
        "Funding Name": "Horizon Europe: Trustworthy Artificial Intelligence and Data Services Call",
        "Description": "Horizon Europe funding supporting trustworthy AI services, innovative data services, robotics and strategic digital technologies.",
        "Type": "Grant",
        "Eligibility": "Universities, research organisations, companies and eligible Horizon Europe participants.",
        "Deadline": "15 April 2026",
        "States/Country/Region Covered": "European Union and Horizon Europe Associated Countries",
        "Funding Amount": "€221.8 million allocated across related AI and digital topics",
        "Website": "https://ec.europa.eu/info/funding-tenders/opportunities/portal/",
        "Category": "Artificial Intelligence",
        "Status": "Closed"
    },

    {
        "Funding Name": "Horizon Europe: Next-Generation AI Agents for Real-World Applications",
        "Description": "Research and innovation funding supporting next-generation AI agents and real-world artificial intelligence applications.",
        "Type": "Research Grant",
        "Eligibility": "Research institutions, universities, companies and eligible Horizon Europe applicants.",
        "Deadline": "15 April 2026",
        "States/Country/Region Covered": "European Union and Associated Countries",
        "Funding Amount": "€38 million indicative budget",
        "Website": "https://ec.europa.eu/info/funding-tenders/opportunities/portal/",
        "Category": "Artificial Intelligence",
        "Status": "Closed"
    },

    {
        "Funding Name": "Horizon Europe: Apply AI - Science for AI Pillar of RAISE",
        "Description": "Research and Innovation Action supporting advanced AI research capabilities and AI science networks.",
        "Type": "Research Grant",
        "Eligibility": "Eligible research organisations and institutions under Horizon Europe rules.",
        "Deadline": "15 April 2026",
        "States/Country/Region Covered": "European Union and Horizon Europe Associated Countries",
        "Funding Amount": "€17 million budget",
        "Website": "https://ec.europa.eu/info/funding-tenders/opportunities/portal/",
        "Category": "Artificial Intelligence",
        "Status": "Closed"
    },

    {
        "Funding Name": "NSF Artificial Intelligence Research Funding",
        "Description": "National Science Foundation funding supporting artificial intelligence research, machine learning, computing and AI applications.",
        "Type": "Research Grant",
        "Eligibility": "Eligible US universities, research institutions and researchers.",
        "Deadline": "Varies by programme",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by NSF programme",
        "Website": "https://www.nsf.gov/funding",
        "Category": "Artificial Intelligence",
        "Status": "Active"
    },

    {
        "Funding Name": "DARPA Artificial Intelligence Research Programs",
        "Description": "Government research funding supporting advanced artificial intelligence technologies and breakthrough AI capabilities.",
        "Type": "Research Grant",
        "Eligibility": "Universities, companies, research organisations and eligible institutions.",
        "Deadline": "Varies by programme",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by programme",
        "Website": "https://www.darpa.mil/work-with-us",
        "Category": "Artificial Intelligence",
        "Status": "Active"
    }
]


ai_grants_batch4_df = pd.DataFrame(ai_grants_batch4)

ai_grants_final = pd.concat(
    [ai_grants_final, ai_grants_batch4_df],
    ignore_index=True
)

ai_grants_final.shape

(25, 10)

In [59]:
ai_grants_df.columns

Index(['Funding Name', 'Description', 'Type', 'Eligibility', 'Deadline',
       'States/Country/Region Covered', 'Funding Amount', 'Website',
       'Category', 'Status'],
      dtype='object')

In [60]:
ai_grants_final = pd.concat(
    [
        ai_grants_df,
        more_ai_df,
        ai_grants_batch3_df,
        ai_grants_batch4_df
    ],
    ignore_index=True
)

ai_grants_final

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,AI Fund in Collaboration with Google – NCAIR,Grant programme supporting Nigerian startups d...,Grant,Nigerian startups developing AI-based solution...,Closed (previous round),Nigeria,Up to ₦10 million per startup,https://ncair.nitda.gov.ng/aifund/,AI Innovation,Closed
1,AI Upskill Accelerator Pilot Program,U.S. Economic Development Administration fundi...,Grant,Eligible U.S. entities implementing industry-l...,10 July 2026,United States,Approximately $25 million programme; awards $1...,https://www.eda.gov/funding/funding-opportunit...,AI Innovation,Open
2,The Genesis Mission: Transforming Science and ...,U.S. Department of Energy funding supporting i...,Research Grant,"Universities, research organisations, companie...",17 December 2026,United States,"$293.76 million total; awards $500,000–$16 mil...",https://www.grants.gov/search-results-detail/3...,AI Innovation,Open
3,Innovate UK Frontier Artificial Intelligence D...,Grant competition supporting feasibility studi...,Grant,"UK businesses, research organisations, chariti...",10 June 2026,United Kingdom,Up to £2.5 million total funding,https://www.ukri.org/opportunity/frontier-arti...,AI Innovation,Closed
4,Future Computing Paradigms Network Plus,Research funding supporting UK collaboration a...,Research Grant,Researchers based at UK research organisations...,13 October 2026,United Kingdom,£2.8 million total funding,https://www.ukri.org/opportunity/,AI Innovation,Upcoming
5,AI Pathways To The Future,Grant programme supporting education and coope...,Grant,Eligible organisations applying under the U.S....,9 August 2026,Indonesia,Not stated,https://www.grants.gov/search-results-detail/3...,AI Innovation,Open
6,Horizon Europe Digital Calls - Trustworthy Art...,"Horizon Europe funding supporting AI, data ser...",Grant,"Research organisations, universities, companie...",15 April 2026,European Union and Horizon Europe Associated C...,€221.8 million total across Digital call topics,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
7,Horizon Europe Next-Generation AI Agents for R...,Research and innovation funding supporting nex...,Research Grant,"Universities, research organisations, companie...",15 April 2026,Europe,€38 million budget,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
8,Horizon Europe Efficient and Compliant Access ...,Innovation funding supporting AI-driven compli...,Innovation Grant,"Research institutions, companies and eligible ...",15 April 2026,Europe,€46.5 million budget,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
9,European Innovation Council (EIC) Advanced Inn...,Horizon Europe challenge funding supporting br...,Grant,"Startups, SMEs and research teams developing b...",Varies by challenge,European Union and Horizon Europe Associated C...,Varies by challenge,https://eic.ec.europa.eu/eic-funding-opportuni...,Artificial Intelligence,Upcoming


In [61]:
ai_grants_final[["Funding Name","Website","Status"]]

,Funding Name,Website,Status
0,AI Fund in Collaboration with Google – NCAIR,https://ncair.nitda.gov.ng/aifund/,Closed
1,AI Upskill Accelerator Pilot Program,https://www.eda.gov/funding/funding-opportunit...,Open
2,The Genesis Mission: Transforming Science and ...,https://www.grants.gov/search-results-detail/3...,Open
3,Innovate UK Frontier Artificial Intelligence D...,https://www.ukri.org/opportunity/frontier-arti...,Closed
4,Future Computing Paradigms Network Plus,https://www.ukri.org/opportunity/,Upcoming
5,AI Pathways To The Future,https://www.grants.gov/search-results-detail/3...,Open
6,Horizon Europe Digital Calls - Trustworthy Art...,https://ec.europa.eu/info/funding-tenders/oppo...,Closed
7,Horizon Europe Next-Generation AI Agents for R...,https://ec.europa.eu/info/funding-tenders/oppo...,Closed
8,Horizon Europe Efficient and Compliant Access ...,https://ec.europa.eu/info/funding-tenders/oppo...,Closed
9,European Innovation Council (EIC) Advanced Inn...,https://eic.ec.europa.eu/eic-funding-opportuni...,Upcoming


In [62]:
# Rename Type where necessary

ai_grants_final.loc[
    ai_grants_final["Funding Name"].str.contains(
        "NSF|DARPA|NIH|Google.org",
        case=False,
        na=False
    ),
    "Type"
] = "Funding Programme"


# Remove only exact duplicates
ai_grants_final = ai_grants_final.drop_duplicates(
    subset=["Funding Name"],
    keep="first"
)


# Check size
ai_grants_final.shape

(24, 10)

In [63]:
ai_grants_final = pd.concat(
    [
        ai_grants_df,
        more_ai_df,
        ai_grants_batch3_df,
        ai_grants_batch4_df
    ],
    ignore_index=True
)

ai_grants_final

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,AI Fund in Collaboration with Google – NCAIR,Grant programme supporting Nigerian startups d...,Grant,Nigerian startups developing AI-based solution...,Closed (previous round),Nigeria,Up to ₦10 million per startup,https://ncair.nitda.gov.ng/aifund/,AI Innovation,Closed
1,AI Upskill Accelerator Pilot Program,U.S. Economic Development Administration fundi...,Grant,Eligible U.S. entities implementing industry-l...,10 July 2026,United States,Approximately $25 million programme; awards $1...,https://www.eda.gov/funding/funding-opportunit...,AI Innovation,Open
2,The Genesis Mission: Transforming Science and ...,U.S. Department of Energy funding supporting i...,Research Grant,"Universities, research organisations, companie...",17 December 2026,United States,"$293.76 million total; awards $500,000–$16 mil...",https://www.grants.gov/search-results-detail/3...,AI Innovation,Open
3,Innovate UK Frontier Artificial Intelligence D...,Grant competition supporting feasibility studi...,Grant,"UK businesses, research organisations, chariti...",10 June 2026,United Kingdom,Up to £2.5 million total funding,https://www.ukri.org/opportunity/frontier-arti...,AI Innovation,Closed
4,Future Computing Paradigms Network Plus,Research funding supporting UK collaboration a...,Research Grant,Researchers based at UK research organisations...,13 October 2026,United Kingdom,£2.8 million total funding,https://www.ukri.org/opportunity/,AI Innovation,Upcoming
5,AI Pathways To The Future,Grant programme supporting education and coope...,Grant,Eligible organisations applying under the U.S....,9 August 2026,Indonesia,Not stated,https://www.grants.gov/search-results-detail/3...,AI Innovation,Open
6,Horizon Europe Digital Calls - Trustworthy Art...,"Horizon Europe funding supporting AI, data ser...",Grant,"Research organisations, universities, companie...",15 April 2026,European Union and Horizon Europe Associated C...,€221.8 million total across Digital call topics,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
7,Horizon Europe Next-Generation AI Agents for R...,Research and innovation funding supporting nex...,Research Grant,"Universities, research organisations, companie...",15 April 2026,Europe,€38 million budget,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
8,Horizon Europe Efficient and Compliant Access ...,Innovation funding supporting AI-driven compli...,Innovation Grant,"Research institutions, companies and eligible ...",15 April 2026,Europe,€46.5 million budget,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
9,European Innovation Council (EIC) Advanced Inn...,Horizon Europe challenge funding supporting br...,Grant,"Startups, SMEs and research teams developing b...",Varies by challenge,European Union and Horizon Europe Associated C...,Varies by challenge,https://eic.ec.europa.eu/eic-funding-opportuni...,Artificial Intelligence,Upcoming


In [64]:
ai_grants_final.loc[
    ai_grants_final["Funding Name"].str.contains(
        "NSF|DARPA|NIH|Google.org",
        case=False,
        na=False
    ),
    "Type"
] = "Funding Programme"

In [65]:
ai_grants_final = ai_grants_final.drop_duplicates(
    subset=["Funding Name"]
)

In [66]:
ai_grants_final.shape

(24, 10)

In [67]:
ai_grants_final = pd.concat(
    [
        ai_grants_df,
        more_ai_df,
        ai_grants_batch3_df,
        ai_grants_batch4_df
    ],
    ignore_index=True
)

ai_grants_final

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,AI Fund in Collaboration with Google – NCAIR,Grant programme supporting Nigerian startups d...,Grant,Nigerian startups developing AI-based solution...,Closed (previous round),Nigeria,Up to ₦10 million per startup,https://ncair.nitda.gov.ng/aifund/,AI Innovation,Closed
1,AI Upskill Accelerator Pilot Program,U.S. Economic Development Administration fundi...,Grant,Eligible U.S. entities implementing industry-l...,10 July 2026,United States,Approximately $25 million programme; awards $1...,https://www.eda.gov/funding/funding-opportunit...,AI Innovation,Open
2,The Genesis Mission: Transforming Science and ...,U.S. Department of Energy funding supporting i...,Research Grant,"Universities, research organisations, companie...",17 December 2026,United States,"$293.76 million total; awards $500,000–$16 mil...",https://www.grants.gov/search-results-detail/3...,AI Innovation,Open
3,Innovate UK Frontier Artificial Intelligence D...,Grant competition supporting feasibility studi...,Grant,"UK businesses, research organisations, chariti...",10 June 2026,United Kingdom,Up to £2.5 million total funding,https://www.ukri.org/opportunity/frontier-arti...,AI Innovation,Closed
4,Future Computing Paradigms Network Plus,Research funding supporting UK collaboration a...,Research Grant,Researchers based at UK research organisations...,13 October 2026,United Kingdom,£2.8 million total funding,https://www.ukri.org/opportunity/,AI Innovation,Upcoming
5,AI Pathways To The Future,Grant programme supporting education and coope...,Grant,Eligible organisations applying under the U.S....,9 August 2026,Indonesia,Not stated,https://www.grants.gov/search-results-detail/3...,AI Innovation,Open
6,Horizon Europe Digital Calls - Trustworthy Art...,"Horizon Europe funding supporting AI, data ser...",Grant,"Research organisations, universities, companie...",15 April 2026,European Union and Horizon Europe Associated C...,€221.8 million total across Digital call topics,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
7,Horizon Europe Next-Generation AI Agents for R...,Research and innovation funding supporting nex...,Research Grant,"Universities, research organisations, companie...",15 April 2026,Europe,€38 million budget,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
8,Horizon Europe Efficient and Compliant Access ...,Innovation funding supporting AI-driven compli...,Innovation Grant,"Research institutions, companies and eligible ...",15 April 2026,Europe,€46.5 million budget,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
9,European Innovation Council (EIC) Advanced Inn...,Horizon Europe challenge funding supporting br...,Grant,"Startups, SMEs and research teams developing b...",Varies by challenge,European Union and Horizon Europe Associated C...,Varies by challenge,https://eic.ec.europa.eu/eic-funding-opportuni...,Artificial Intelligence,Upcoming


In [68]:
# Copy dataframe first
ai_final = ai_df.copy()

# Remove exact duplicates by funding name
ai_final = ai_final.drop_duplicates(subset=["Funding Name"], keep="first")


# Remove duplicate/overlapping entries
remove_names = [
    "Horizon Europe: Trustworthy Artificial Intelligence",
    "Horizon Europe: Next-Generation AI Agents for Research",
    "NSF Artificial Intelligence Research Funding",
    "DARPA Artificial Intelligence Research Programs"
]

ai_final = ai_final[
    ~ai_final["Funding Name"].isin(remove_names)
]


# Fix Type column for umbrella funding programmes

programme_keywords = [
    "National Science Foundation",
    "NSF",
    "DARPA",
    "NIH",
    "Google.org"
]

mask = ai_final["Funding Name"].str.contains(
    "|".join(programme_keywords),
    case=False,
    na=False
)

ai_final.loc[mask, "Type"] = "Funding Programme"


# Fix Horizon Europe entries
horizon_mask = ai_final["Funding Name"].str.contains(
    "Horizon Europe",
    case=False,
    na=False
)

ai_final.loc[horizon_mask, "Type"] = "Funding Opportunity"


# Fix Google.org status
google_mask = ai_final["Funding Name"].str.contains(
    "Google.org",
    case=False,
    na=False
)

ai_final.loc[google_mask, "Status"] = "Varies by opportunity"


# Fix broad UKRI future calls
ukri_future_mask = ai_final["Funding Name"].str.contains(
    "Advancing Sustainable AI|Next Generation AI",
    case=False,
    na=False
)

ai_final.loc[ukri_future_mask, "Status"] = "Upcoming"


# Replace missing values
ai_final = ai_final.fillna("Not stated")


# Reset index
ai_final = ai_final.reset_index(drop=True)


# Check final size
print("Final shape:", ai_final.shape)


# Save
ai_final.to_csv(
    "verified_ai_funding_final.csv",
    index=False
)

print("Saved successfully!")

KeyError: 'Funding Name'

In [69]:
print(ai_df.shape)
print(ai_df.columns.tolist())

(0, 0)
[]


In [70]:
# Show all dataframe variables currently in memory
for name, obj in globals().items():
    if str(type(obj)) == "<class 'pandas.core.frame.DataFrame'>":
        print(name, obj.shape, obj.columns.tolist())

RuntimeError: dictionary changed size during iteration

In [2]:
import pandas as pd

# Safely find all DataFrames currently in memory
dataframes = []

for name in list(globals().keys()):
    obj = globals()[name]
    if isinstance(obj, pd.DataFrame):
        dataframes.append((name, obj.shape, list(obj.columns)))

for item in dataframes:
    print(item)

In [3]:
import pandas as pd

data = [
    [
        "AI Fund in Collaboration with Google – NCAIR",
        "Grant programme supporting Nigerian startups developing AI-based solutions.",
        "Grant",
        "Nigerian startups developing AI-based solutions",
        "Closed (previous round)",
        "Nigeria",
        "Up to ₦10 million per startup",
        "https://ncair.nitda.gov.ng/aifund/",
        "AI Innovation",
        "Closed"
    ],
    [
        "AI Upskill Accelerator Pilot Program",
        "U.S. Economic Development Administration funding supporting AI workforce and industry upskilling.",
        "Grant Programme",
        "Eligible U.S. entities implementing eligible industry-led AI upskilling activities",
        "10 July 2026",
        "United States",
        "Varies by programme",
        "https://www.eda.gov/funding/funding-opportunities",
        "AI Innovation",
        "Open"
    ],
    [
        "The Genesis Mission: Transforming Science and Engineering Through AI",
        "U.S. Department of Energy funding supporting scientific research and innovation using artificial intelligence.",
        "Research Grant",
        "Eligible universities, research organisations, companies and other eligible applicants",
        "17 December 2026",
        "United States",
        "$500,000–$16 million",
        "https://www.grants.gov/",
        "AI Innovation",
        "Open"
    ],
    [
        "Innovate UK Frontier Artificial Intelligence Discovery",
        "Funding supporting feasibility and discovery work involving frontier artificial intelligence.",
        "Grant",
        "Eligible UK businesses, research organisations and other eligible organisations",
        "10 June 2026",
        "United Kingdom",
        "Up to £2.5 million total funding",
        "https://www.ukri.org/opportunity/",
        "AI Innovation",
        "Closed"
    ],
    [
        "Future Computing Paradigms Network Plus",
        "UK research funding supporting collaboration and research into future computing paradigms.",
        "Research Grant",
        "Researchers based at eligible UK research organisations",
        "13 October 2026",
        "United Kingdom",
        "£2.8 million total funding",
        "https://www.ukri.org/opportunity/future-computing-paradigms-network-plus/",
        "AI Innovation",
        "Upcoming"
    ],
    [
        "AI Pathways To The Future",
        "Grant programme supporting AI-related education and cooperation.",
        "Grant",
        "Eligible organisations applying under the relevant U.S. programme requirements",
        "9 August 2026",
        "Indonesia",
        "Not stated",
        "https://www.grants.gov/",
        "AI Innovation",
        "Open"
    ],
    [
        "Horizon Europe Digital Calls – Trustworthy Artificial Intelligence",
        "Horizon Europe funding supporting trustworthy artificial intelligence, data and digital technologies.",
        "Funding Opportunity",
        "Eligible research organisations, universities, companies and consortium partners",
        "15 April 2026",
        "European Union and Horizon Europe Associated Countries",
        "€221.8 million across related Digital call topics",
        "https://ec.europa.eu/info/funding-tenders/opportunities/portal/",
        "Artificial Intelligence",
        "Closed"
    ],
    [
        "Horizon Europe Next-Generation AI Agents for Robotics",
        "Research and innovation funding supporting next-generation artificial intelligence agents and robotics.",
        "Funding Opportunity",
        "Eligible research organisations, universities, companies and consortium partners",
        "15 April 2026",
        "Europe",
        "€38 million",
        "https://ec.europa.eu/info/funding-tenders/opportunities/portal/",
        "Artificial Intelligence",
        "Closed"
    ],
    [
        "Horizon Europe Efficient and Compliant Access to AI",
        "Innovation funding supporting efficient and compliant access to artificial intelligence technologies.",
        "Funding Opportunity",
        "Eligible research institutions, companies and consortium partners",
        "15 April 2026",
        "Europe",
        "€46.5 million",
        "https://ec.europa.eu/info/funding-tenders/opportunities/portal/",
        "Artificial Intelligence",
        "Closed"
    ],
    [
        "European Innovation Council (EIC) Advanced Innovation Challenges",
        "Horizon Europe challenge funding supporting breakthrough innovation and advanced technologies.",
        "Grant",
        "Startups, SMEs and eligible research teams",
        "Varies by challenge",
        "European Union and Horizon Europe Associated Countries",
        "Varies by challenge",
        "https://eic.ec.europa.eu/eic-funding-opportunities_en",
        "Artificial Intelligence",
        "Upcoming"
    ],
    [
        "National Science Foundation (NSF) Artificial Intelligence Research Funding",
        "NSF funding supporting research and innovation involving artificial intelligence.",
        "Funding Programme",
        "Eligible U.S. universities, research organisations and institutions",
        "Varies by programme",
        "United States",
        "Varies by NSF programme",
        "https://www.nsf.gov/funding",
        "Artificial Intelligence",
        "Active"
    ],
    [
        "DARPA Artificial Intelligence Research Programs",
        "DARPA supports advanced research into artificial intelligence and related technologies.",
        "Funding Programme",
        "Universities, companies and eligible research institutions",
        "Varies by programme",
        "United States",
        "Varies by programme",
        "https://www.darpa.mil/work-with-us",
        "Artificial Intelligence",
        "Active"
    ],
    [
        "Google.org Artificial Intelligence Grants",
        "Google.org provides philanthropic funding for projects using AI to address social challenges.",
        "Funding Programme",
        "Nonprofits, social impact organisations and eligible partners",
        "Varies by opportunity",
        "Global",
        "Varies by grant",
        "https://www.google.org/programs/",
        "Artificial Intelligence",
        "Varies by opportunity"
    ],
    [
        "NSF AI Datasets for AI-Enabled Scientific Discovery",
        "Funding supporting creation, management and use of datasets for AI-enabled scientific discovery.",
        "Research Grant",
        "Eligible U.S. universities and research institutions",
        "4 November 2026",
        "United States",
        "Varies by NSF award",
        "https://www.nsf.gov/funding",
        "Artificial Intelligence",
        "Open"
    ],
    [
        "NSF Computer and Information Science and Engineering",
        "NSF funding supporting computer and information science research, including artificial intelligence.",
        "Funding Programme",
        "U.S.-based eligible academic institutions and research organisations",
        "Varies by programme",
        "United States",
        "Varies by programme",
        "https://www.nsf.gov/funding",
        "Artificial Intelligence",
        "Active"
    ],
    [
        "UKRI Advancing Sustainable AI Technology",
        "Funding supporting research and development of innovative sustainable AI technologies.",
        "Grant",
        "Eligible UK research organisations and innovators",
        "Expected future call",
        "United Kingdom",
        "Up to £500,000 per award",
        "https://www.ukri.org/apply-for-funding/",
        "Artificial Intelligence",
        "Upcoming"
    ],
    [
        "UKRI Next Generation AI: Explainable AI",
        "Research funding supporting fundamental research into explainable and next-generation AI.",
        "Research Grant",
        "Researchers based at eligible UK research organisations",
        "Expected future call",
        "United Kingdom",
        "Up to £500,000 per award",
        "https://www.ukri.org/apply-for-funding/",
        "Artificial Intelligence",
        "Upcoming"
    ],
    [
        "NIH Artificial Intelligence and Machine Learning Funding",
        "Health research funding supporting applications of artificial intelligence and machine learning.",
        "Funding Programme",
        "Eligible universities, research institutions and investigators",
        "Varies by funding announcement",
        "United States",
        "Varies by NIH programme",
        "https://grants.nih.gov/",
        "Artificial Intelligence",
        "Active"
    ],
    [
        "Horizon Europe: Apply AI – Science for AI Pillar",
        "Research and innovation funding supporting advanced artificial intelligence research.",
        "Funding Opportunity",
        "Eligible research organisations and institutions",
        "15 April 2026",
        "European Union and Horizon Europe Associated Countries",
        "€17 million",
        "https://ec.europa.eu/info/funding-tenders/opportunities/portal/",
        "Artificial Intelligence",
        "Closed"
    ]
]

columns = [
    "Funding Name",
    "Description",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Website",
    "Category",
    "Status"
]

ai_final = pd.DataFrame(data, columns=columns)

# Remove duplicate names just in case
ai_final = ai_final.drop_duplicates(subset="Funding Name").reset_index(drop=True)

# Display
print("FINAL AI DATASET")
print("Rows:", len(ai_final))
print("Columns:", len(ai_final.columns))

display(ai_final)

# Save
ai_final.to_csv("verified_ai_funding_final.csv", index=False)

print("\nSaved as: verified_ai_funding_final.csv")

FINAL AI DATASET
Rows: 19
Columns: 10


,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,AI Fund in Collaboration with Google – NCAIR,Grant programme supporting Nigerian startups d...,Grant,Nigerian startups developing AI-based solutions,Closed (previous round),Nigeria,Up to ₦10 million per startup,https://ncair.nitda.gov.ng/aifund/,AI Innovation,Closed
1,AI Upskill Accelerator Pilot Program,U.S. Economic Development Administration fundi...,Grant Programme,Eligible U.S. entities implementing eligible i...,10 July 2026,United States,Varies by programme,https://www.eda.gov/funding/funding-opportunities,AI Innovation,Open
2,The Genesis Mission: Transforming Science and ...,U.S. Department of Energy funding supporting s...,Research Grant,"Eligible universities, research organisations,...",17 December 2026,United States,"$500,000–$16 million",https://www.grants.gov/,AI Innovation,Open
3,Innovate UK Frontier Artificial Intelligence D...,Funding supporting feasibility and discovery w...,Grant,"Eligible UK businesses, research organisations...",10 June 2026,United Kingdom,Up to £2.5 million total funding,https://www.ukri.org/opportunity/,AI Innovation,Closed
4,Future Computing Paradigms Network Plus,UK research funding supporting collaboration a...,Research Grant,Researchers based at eligible UK research orga...,13 October 2026,United Kingdom,£2.8 million total funding,https://www.ukri.org/opportunity/future-comput...,AI Innovation,Upcoming
5,AI Pathways To The Future,Grant programme supporting AI-related educatio...,Grant,Eligible organisations applying under the rele...,9 August 2026,Indonesia,Not stated,https://www.grants.gov/,AI Innovation,Open
6,Horizon Europe Digital Calls – Trustworthy Art...,Horizon Europe funding supporting trustworthy ...,Funding Opportunity,"Eligible research organisations, universities,...",15 April 2026,European Union and Horizon Europe Associated C...,€221.8 million across related Digital call topics,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
7,Horizon Europe Next-Generation AI Agents for R...,Research and innovation funding supporting nex...,Funding Opportunity,"Eligible research organisations, universities,...",15 April 2026,Europe,€38 million,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
8,Horizon Europe Efficient and Compliant Access ...,Innovation funding supporting efficient and co...,Funding Opportunity,"Eligible research institutions, companies and ...",15 April 2026,Europe,€46.5 million,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
9,European Innovation Council (EIC) Advanced Inn...,Horizon Europe challenge funding supporting br...,Grant,"Startups, SMEs and eligible research teams",Varies by challenge,European Union and Horizon Europe Associated C...,Varies by challenge,https://eic.ec.europa.eu/eic-funding-opportuni...,Artificial Intelligence,Upcoming



Saved as: verified_ai_funding_final.csv


In [4]:
import pandas as pd
import requests
from io import BytesIO

# Official NAICOM insurer database
url = "https://portal.naicom.gov.ng/Download/AllInsurers.xlsx"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

print("Status code:", response.status_code)
print("File size:", len(response.content), "bytes")

# Read the official Excel file
insurance_raw = pd.read_excel(BytesIO(response.content))

print("\nShape:", insurance_raw.shape)
print("\nColumns:")
print(insurance_raw.columns.tolist())

print("\nFirst 10 rows:")
display(insurance_raw.head(10))

Status code: 200
File size: 20262 bytes


C:\Users\user\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:232: UserWarning: Workbook contains no stylesheet, using openpyxl's defaults
  warn("Workbook contains no stylesheet, using openpyxl's defaults")



Shape: (82, 37)

Columns:
['CompanyID', 'CompanyRegistrationType', 'CompanyRegistrationNumber', 'CompanyRegistrationDate', 'CompanyRegistrationRenewalDate', 'CompanyName', 'CompanyAcronyms', 'CompanyCode', 'CompanyWebsite', 'CompanyAddressLine', 'CompanyAddressCityLGA', 'CompanyAddressState', 'CompanyAddressPostcode', 'CompanyPhone', 'CompanyFax', 'CompanyEmail', 'CompanyContactNameTitle', 'CompanyContactNameFirst', 'CompanyContactNameMiddle', 'CompanyContactNameLast', 'CompanyContactAddressLine', 'CompanyContactAddressCityLGA', 'CompanyContactAddressState', 'CompanyContactAddressPostcode', 'CompanyContactPhone', 'CompanyContactFax', 'CompanyContactEmail', 'CompanyCEONameTitle', 'CompanyCEONameFirst', 'CompanyCEONameMiddle', 'CompanyCEONameLast', 'CompanyDescription', 'CompanyStatus', 'LastModificationDate', 'CreationDate', 'LastModificationPerson', 'CreationPerson']

First 10 rows:


,CompanyID,CompanyRegistrationType,CompanyRegistrationNumber,CompanyRegistrationDate,CompanyRegistrationRenewalDate,CompanyName,CompanyAcronyms,CompanyCode,CompanyWebsite,CompanyAddressLine,...,CompanyCEONameTitle,CompanyCEONameFirst,CompanyCEONameMiddle,CompanyCEONameLast,CompanyDescription,CompanyStatus,LastModificationDate,CreationDate,LastModificationPerson,CreationPerson
0,1,G,1,1/1/1900 12:00:00 AM,NaN,Alliance & General Insurance Company Ltd,AGIC,RIC-001,NaN,A&G Tower. 12 Abibu Oki street Off Marina,...,Mr,Felicia,Abiola (Resigned),Bolajoko-David,NaN,0,NaN,NaN,NaN,NaN
1,3,L,2,1/1/1900 12:00:00 AM,NaN,African Alliance Insurance Plc,AAIP,RIC-002,NaN,"112, Broad Street P.O.Box 2276",...,Mrs,Joyce,Ojemudia,NaN,NaN,0,NaN,NaN,NaN,NaN
2,4,G,3,1/1/1900 12:00:00 AM,NaN,NSIA Insurance Company Ltd,ADIC,RIC-003,NaN,"3, Elsie Femi Pearse Street",...,Mr,Moruf,.,Apampa,NaN,0,NaN,NaN,NaN,NaN
3,5,C,4,1/1/1900 12:00:00 AM,NaN,AIICO Insurance PLC,AIIC,RIC-004,NaN,Abuja,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
4,6,G,5,1/1/1900 12:00:00 AM,NaN,Anchor Insurance Company Ltd,AICL,RIC-005,NaN,"Plot 13A, Ayo Jagun Street",...,Mr,Ebosa,Austin,Osegha,NaN,0,NaN,NaN,NaN,NaN
5,7,L,6,1/1/1900 12:00:00 AM,NaN,Capital Express Assurance Limited,CEAL,RIC-006,NaN,Capital Express House13 Bishop Kale Close Behi...,...,Mrs,Foluke,Adebola,Odulake,NaN,0,NaN,NaN,NaN,NaN
6,8,G,7,1/1/1900 12:00:00 AM,NaN,Consolidated Hallmark Insurance Plc,CHIP,RIC-007,NaN,Plot 33D Bishop Aboyade Cole Street P.O.Box 74013,...,Mrs.,Mary,NaN,Adeyanju,NaN,0,NaN,NaN,NaN,NaN
7,9,C,8,1/1/1900 12:00:00 AM,NaN,Cornerstone Insurance Plc,COIP,RIC-008,NaN,"21, Water Corporation Drive,Off Ligali Ayorind...",...,Mr,Stephen,.,Alangbo,NaN,0,NaN,NaN,NaN,NaN
8,15,L,51,1/1/1900 12:00:00 AM,NaN,Sanlam Allianz Life Insurance Limited,SALAL,RIC-051,NaN,"9/11, Marcarthy Street",...,Mr.,Babatunde,NaN,Mimiko,NaN,0,NaN,NaN,NaN,NaN
9,16,G,47,1/1/1900 12:00:00 AM,NaN,FIN Insurance Company Limited,FCLT,RIC-047,NaN,No. 34 Gana Street,...,Mr.,Bashi,.,Binji,NaN,0,NaN,NaN,NaN,NaN


In [5]:
insurance_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82 entries, 0 to 81
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   CompanyID                       82 non-null     int64  
 1   CompanyRegistrationType         82 non-null     object 
 2   CompanyRegistrationNumber       82 non-null     int64  
 3   CompanyRegistrationDate         82 non-null     object 
 4   CompanyRegistrationRenewalDate  10 non-null     object 
 5   CompanyName                     82 non-null     object 
 6   CompanyAcronyms                 82 non-null     object 
 7   CompanyCode                     82 non-null     object 
 8   CompanyWebsite                  0 non-null      float64
 9   CompanyAddressLine              80 non-null     object 
 10  CompanyAddressCityLGA           68 non-null     object 
 11  CompanyAddressState             80 non-null     object 
 12  CompanyAddressPostcode          70 non

In [6]:
display(insurance_raw)

,CompanyID,CompanyRegistrationType,CompanyRegistrationNumber,CompanyRegistrationDate,CompanyRegistrationRenewalDate,CompanyName,CompanyAcronyms,CompanyCode,CompanyWebsite,CompanyAddressLine,...,CompanyCEONameTitle,CompanyCEONameFirst,CompanyCEONameMiddle,CompanyCEONameLast,CompanyDescription,CompanyStatus,LastModificationDate,CreationDate,LastModificationPerson,CreationPerson
0,1,G,1,1/1/1900 12:00:00 AM,NaN,Alliance & General Insurance Company Ltd,AGIC,RIC-001,NaN,A&G Tower. 12 Abibu Oki street Off Marina,...,Mr,Felicia,Abiola (Resigned),Bolajoko-David,NaN,0,NaN,NaN,NaN,NaN
1,3,L,2,1/1/1900 12:00:00 AM,NaN,African Alliance Insurance Plc,AAIP,RIC-002,NaN,"112, Broad Street P.O.Box 2276",...,Mrs,Joyce,Ojemudia,NaN,NaN,0,NaN,NaN,NaN,NaN
2,4,G,3,1/1/1900 12:00:00 AM,NaN,NSIA Insurance Company Ltd,ADIC,RIC-003,NaN,"3, Elsie Femi Pearse Street",...,Mr,Moruf,.,Apampa,NaN,0,NaN,NaN,NaN,NaN
3,5,C,4,1/1/1900 12:00:00 AM,NaN,AIICO Insurance PLC,AIIC,RIC-004,NaN,Abuja,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
4,6,G,5,1/1/1900 12:00:00 AM,NaN,Anchor Insurance Company Ltd,AICL,RIC-005,NaN,"Plot 13A, Ayo Jagun Street",...,Mr,Ebosa,Austin,Osegha,NaN,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,152,C,12,1/1/1900 12:00:00 AM,1/1/1900 12:00:00 AM,NASSURE MICROINSURANCE LIMITED,NAML,RMC-012,NaN,"4ED Building, 47 Marina Lagos",...,Mr,Babatunde,Olarinde,Oshadiya,NaN,0,NaN,NaN,NaN,NaN
78,154,G,98,11/27/2024 12:00:00 AM,11/27/2034 12:00:00 AM,NPF Insurance Company Ltd,NPF,RIC-098,NaN,"Behind Luis Edet House, Force Headquarters, Sh...",...,Mr,Temitayo,NaN,Oke,NaN,0,NaN,NaN,NaN,NaN
79,156,G,13,3/20/2025 12:00:00 AM,NaN,Vertex Microinsurance Company Ltd,VERT,RMC-013,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
80,159,L,99,1/1/1900 12:00:00 AM,1/1/1900 12:00:00 AM,CHI LIFE ASSURANCE LTD,CHIL,RIC-99,NaN,NaN,...,Mrs,Ose,NaN,Oluyanwo,NaN,0,NaN,NaN,NaN,NaN


In [7]:
import pandas as pd
import numpy as np

# Make a copy so the original NAICOM data stays untouched
insurance_df = insurance_raw.copy()

# ---------------------------------------------------------
# 1. Create the final columns you requested
# ---------------------------------------------------------

insurance_final = pd.DataFrame()

insurance_final["Name"] = insurance_df["CompanyName"]

# NAICOM is an insurance regulator, not an HMO regulator.
# Therefore we do NOT invent HMO IDs.
insurance_final["HMO ID"] = "Not applicable"

insurance_final["Website"] = insurance_df["CompanyWebsite"]

insurance_final["Address"] = (
    insurance_df["CompanyAddressLine"]
    .fillna("")
    .astype(str)
    .replace("nan", "", regex=False)
)

# Add city/LGA and state where available
insurance_final["Address"] = (
    insurance_final["Address"]
    + insurance_df["CompanyAddressCityLGA"].fillna("").astype(str).replace("nan", "", regex=False)
    .apply(lambda x: ", " + x if x else "")
    + insurance_df["CompanyAddressState"].fillna("").astype(str).replace("nan", "", regex=False)
    .apply(lambda x: ", " + x if x else "")
)

insurance_final["Email"] = insurance_df["CompanyEmail"]

insurance_final["Contact Number"] = insurance_df["CompanyPhone"]

insurance_final["Country"] = "Nigeria"

# Official regulator
insurance_final["Source"] = "NAICOM"

insurance_final["Source URL"] = (
    "https://portal.naicom.gov.ng/Download/AllInsurers.xlsx"
)

insurance_final["State"] = insurance_df["CompanyAddressState"]

insurance_final["Region"] = "Nigeria"

# Coverage information is NOT contained in this NAICOM file,
# so we explicitly mark it instead of guessing.
insurance_final["Coverage State"] = "Not stated"

insurance_final["Coverage Type"] = insurance_df["CompanyRegistrationType"].map({
    "G": "General Insurance",
    "L": "Life Insurance",
    "C": "Composite Insurance"
}).fillna("Insurance")

insurance_final["Services Rendered"] = "Insurance services"

insurance_final["Target Customers"] = "Not stated"


# ---------------------------------------------------------
# 2. Replace missing values consistently
# ---------------------------------------------------------

insurance_final = insurance_final.replace(
    [np.nan, "nan", "NaN", ""],
    "Not stated"
)


# ---------------------------------------------------------
# 3. Remove exact duplicate companies
# ---------------------------------------------------------

insurance_final = insurance_final.drop_duplicates(
    subset=["Name"],
    keep="first"
).reset_index(drop=True)


# ---------------------------------------------------------
# 4. Add an internal verification field
# ---------------------------------------------------------

insurance_final["Verification"] = "Verified via official NAICOM dataset"


# ---------------------------------------------------------
# 5. Show result
# ---------------------------------------------------------

print("TOTAL INSURANCE COMPANIES:", len(insurance_final))
print("\nColumns:")
print(insurance_final.columns.tolist())

display(insurance_final.head(10))

TOTAL INSURANCE COMPANIES: 82

Columns:
['Name', 'HMO ID', 'Website', 'Address', 'Email', 'Contact Number', 'Country', 'Source', 'Source URL', 'State', 'Region', 'Coverage State', 'Coverage Type', 'Services Rendered', 'Target Customers', 'Verification']


,Name,HMO ID,Website,Address,Email,Contact Number,Country,Source,Source URL,State,Region,Coverage State,Coverage Type,Services Rendered,Target Customers,Verification
0,Alliance & General Insurance Company Ltd,Not applicable,Not stated,"A&G Tower. 12 Abibu Oki street Off Marina, Lag...",fdavid@aginsuranceplc.com,08033154124,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,General Insurance,Insurance services,Not stated,Verified via official NAICOM dataset
1,African Alliance Insurance Plc,Not applicable,Not stated,"112, Broad Street P.O.Box 2276, Lagos, Lagos",joyce.ojemudia@africanallianceplc.com,08033074307,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,Life Insurance,Insurance services,Not stated,Verified via official NAICOM dataset
2,NSIA Insurance Company Ltd,Not applicable,Not stated,"3, Elsie Femi Pearse Street, Victoria Island, ...",moruf.apampa@nsiainsurance.com,08033465581,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,General Insurance,Insurance services,Not stated,Verified via official NAICOM dataset
3,AIICO Insurance PLC,Not applicable,Not stated,"Abuja, Abuja, FCT",Not stated,080,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,FCT,Nigeria,Not stated,Composite Insurance,Insurance services,Not stated,Verified via official NAICOM dataset
4,Anchor Insurance Company Ltd,Not applicable,Not stated,"Plot 13A, Ayo Jagun Street, Lekki, Lagos",ebosa.a@anchorinsuranceng.com,08053006000,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,General Insurance,Insurance services,Not stated,Verified via official NAICOM dataset
5,Capital Express Assurance Limited,Not applicable,Not stated,Capital Express House13 Bishop Kale Close Behi...,bodukale@capitexpressassurance.com,08033671785,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,Life Insurance,Insurance services,Not stated,Verified via official NAICOM dataset
6,Consolidated Hallmark Insurance Plc,Not applicable,Not stated,Plot 33D Bishop Aboyade Cole Street P.O.Box 74...,maryadeyanju@chiplc.com,08027754441,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,General Insurance,Insurance services,Not stated,Verified via official NAICOM dataset
7,Cornerstone Insurance Plc,Not applicable,Not stated,"21, Water Corporation Drive,Off Ligali Ayorind...",salangbo@cornerstone.com.ng,08033058530,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,Composite Insurance,Insurance services,Not stated,Verified via official NAICOM dataset
8,Sanlam Allianz Life Insurance Limited,Not applicable,Not stated,"9/11, Marcarthy Street, Marina, Lagos",tunde.mimiko@sanlam.com.ng,08023131702,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,Life Insurance,Insurance services,Not stated,Verified via official NAICOM dataset
9,FIN Insurance Company Limited,Not applicable,Not stated,"No. 34 Gana Street, Abuja, FCT",bbinji@finsurance.com.ng,08033112919,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,FCT,Nigeria,Not stated,General Insurance,Insurance services,Not stated,Verified via official NAICOM dataset


In [8]:
# Remove HMO ID and Verification so Insurance matches the common dataset structure

insurance_final = insurance_final.drop(
    columns=["HMO ID", "Verification"],
    errors="ignore"
)

# Make sure the column order is exactly the same
common_columns = [
    "Name",
    "Website",
    "Address",
    "Email",
    "Contact Number",
    "Country",
    "Source",
    "Source URL",
    "State",
    "Region",
    "Coverage State",
    "Coverage Type",
    "Services Rendered",
    "Target Customers"
]

insurance_final = insurance_final[common_columns]

print("Shape:", insurance_final.shape)
print("\nColumns:")
print(insurance_final.columns.tolist())

display(insurance_final.head(10))

Shape: (82, 14)

Columns:
['Name', 'Website', 'Address', 'Email', 'Contact Number', 'Country', 'Source', 'Source URL', 'State', 'Region', 'Coverage State', 'Coverage Type', 'Services Rendered', 'Target Customers']


,Name,Website,Address,Email,Contact Number,Country,Source,Source URL,State,Region,Coverage State,Coverage Type,Services Rendered,Target Customers
0,Alliance & General Insurance Company Ltd,Not stated,"A&G Tower. 12 Abibu Oki street Off Marina, Lag...",fdavid@aginsuranceplc.com,08033154124,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,General Insurance,Insurance services,Not stated
1,African Alliance Insurance Plc,Not stated,"112, Broad Street P.O.Box 2276, Lagos, Lagos",joyce.ojemudia@africanallianceplc.com,08033074307,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,Life Insurance,Insurance services,Not stated
2,NSIA Insurance Company Ltd,Not stated,"3, Elsie Femi Pearse Street, Victoria Island, ...",moruf.apampa@nsiainsurance.com,08033465581,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,General Insurance,Insurance services,Not stated
3,AIICO Insurance PLC,Not stated,"Abuja, Abuja, FCT",Not stated,080,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,FCT,Nigeria,Not stated,Composite Insurance,Insurance services,Not stated
4,Anchor Insurance Company Ltd,Not stated,"Plot 13A, Ayo Jagun Street, Lekki, Lagos",ebosa.a@anchorinsuranceng.com,08053006000,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,General Insurance,Insurance services,Not stated
5,Capital Express Assurance Limited,Not stated,Capital Express House13 Bishop Kale Close Behi...,bodukale@capitexpressassurance.com,08033671785,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,Life Insurance,Insurance services,Not stated
6,Consolidated Hallmark Insurance Plc,Not stated,Plot 33D Bishop Aboyade Cole Street P.O.Box 74...,maryadeyanju@chiplc.com,08027754441,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,General Insurance,Insurance services,Not stated
7,Cornerstone Insurance Plc,Not stated,"21, Water Corporation Drive,Off Ligali Ayorind...",salangbo@cornerstone.com.ng,08033058530,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,Composite Insurance,Insurance services,Not stated
8,Sanlam Allianz Life Insurance Limited,Not stated,"9/11, Marcarthy Street, Marina, Lagos",tunde.mimiko@sanlam.com.ng,08023131702,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,Lagos,Nigeria,Not stated,Life Insurance,Insurance services,Not stated
9,FIN Insurance Company Limited,Not stated,"No. 34 Gana Street, Abuja, FCT",bbinji@finsurance.com.ng,08033112919,Nigeria,NAICOM,https://portal.naicom.gov.ng/Download/AllInsur...,FCT,Nigeria,Not stated,General Insurance,Insurance services,Not stated


In [9]:
import pandas as pd

insurance_funding = pd.DataFrame(columns=[
    "Funding Name",
    "Description",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Website",
    "Category",
    "Status"
])

print("Insurance funding dataframe created.")
print(insurance_funding.shape)
print(insurance_funding.columns.tolist())

Insurance funding dataframe created.
(0, 10)
['Funding Name', 'Description', 'Type', 'Eligibility', 'Deadline', 'States/Country/Region Covered', 'Funding Amount', 'Website', 'Category', 'Status']


In [10]:
insurance_funding = pd.DataFrame([
    {
        "Funding Name": "Inclusive Insurance Innovation Challenge Nigeria",
        "Description": "Challenge supporting innovative insurance solutions for financially underserved populations in Nigeria, including climate risk, health insurance and MSME resilience.",
        "Type": "Innovation Challenge / Grant",
        "Eligibility": "Innovators, startups, insurtechs, fintechs, insurance companies and community-based organisations registered in Nigeria and meeting the stated requirements.",
        "Deadline": "27 October 2025",
        "States/Country/Region Covered": "Nigeria",
        "Funding Amount": "₦30 million per winning proposal",
        "Website": "https://www.undp.org/nigeria/news/inclusive-insurance-innovation-challenge-nigeria",
        "Category": "Insurance Innovation",
        "Status": "Closed"
    },
    {
        "Funding Name": "Insurance Innovation Challenge Fund",
        "Description": "UNDP and ICMIF Foundation fund supporting innovative mutual and cooperative insurance initiatives that improve financial resilience and expand affordable insurance for vulnerable populations in developing countries.",
        "Type": "Innovation Challenge Fund",
        "Eligibility": "Mutual and cooperative insurers and eligible organisations developing or scaling inclusive insurance solutions in developing countries.",
        "Deadline": "Previous application round closed",
        "States/Country/Region Covered": "Developing countries",
        "Funding Amount": "Fund initially launched with US$600,000",
        "Website": "https://icmiffoundation.org/insurance-innovation-challenge/",
        "Category": "Insurance Innovation",
        "Status": "Closed"
    },
    {
        "Funding Name": "Insurance Innovation Challenge – Bangladesh",
        "Description": "UNDP and Bangladesh SME Foundation initiative supporting innovative, inclusive insurance solutions designed to improve resilience among cottage, micro and small enterprises exposed to climate and other risks.",
        "Type": "Innovation Challenge",
        "Eligibility": "Eligible innovators and organisations developing regulator-approved inclusive insurance solutions for cottage, micro and small enterprises in Bangladesh.",
        "Deadline": "Not stated",
        "States/Country/Region Covered": "Bangladesh",
        "Funding Amount": "Not stated",
        "Website": "https://www.undp.org/bangladesh/press-releases/insurance-innovation-challenge-launched-small-businesses",
        "Category": "Insurance Innovation",
        "Status": "Active / Programme launched 2026"
    }
])

print("Records:", len(insurance_funding))
display(insurance_funding)

Records: 3


,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,Inclusive Insurance Innovation Challenge Nigeria,Challenge supporting innovative insurance solu...,Innovation Challenge / Grant,"Innovators, startups, insurtechs, fintechs, in...",27 October 2025,Nigeria,₦30 million per winning proposal,https://www.undp.org/nigeria/news/inclusive-in...,Insurance Innovation,Closed
1,Insurance Innovation Challenge Fund,UNDP and ICMIF Foundation fund supporting inno...,Innovation Challenge Fund,Mutual and cooperative insurers and eligible o...,Previous application round closed,Developing countries,"Fund initially launched with US$600,000",https://icmiffoundation.org/insurance-innovati...,Insurance Innovation,Closed
2,Insurance Innovation Challenge – Bangladesh,UNDP and Bangladesh SME Foundation initiative ...,Innovation Challenge,Eligible innovators and organisations developi...,Not stated,Bangladesh,Not stated,https://www.undp.org/bangladesh/press-releases...,Insurance Innovation,Active / Programme launched 2026


In [12]:
import requests
from bs4 import BeautifulSoup

url = "https://www.developmentaid.org/grants/search"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/151.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers, timeout=30)

print("Status:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

print("Title:", soup.title.get_text(strip=True) if soup.title else "No title")

print("Page length:", len(response.text))

# See whether grant information is actually present in the HTML
text = soup.get_text(" ", strip=True)

print("\nContains insurance:", "insurance" in text.lower())
print("Contains grants:", "grants" in text.lower())

# Preview relevant links
links = []

for a in soup.find_all("a", href=True):
    name = a.get_text(" ", strip=True)
    href = a["href"]

    if name and ("grant" in name.lower() or "insurance" in name.lower()):
        links.append((name, href))

print("\nRelevant links found:", len(links))

for x in links[:20]:
    print(x)

Status: 200
Title: DevelopmentAid
Page length: 10874

Contains insurance: False
Contains grants: False

Relevant links found: 0


In [14]:
import requests
from bs4 import BeautifulSoup

url = "https://www.fundsforngos.org/tag/insurance/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

try:
    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    print("Status:", response.status_code)

    soup = BeautifulSoup(response.text, "html.parser")

    print(
        "Title:",
        soup.title.get_text(" ", strip=True)
        if soup.title else "No title"
    )

    # Find all links
    links = []

    for a in soup.find_all("a", href=True):
        title = a.get_text(" ", strip=True)
        href = a["href"]

        if title and href.startswith("http"):
            links.append((title, href))

    # Remove duplicates
    links = list(dict.fromkeys(links))

    print("Links found:", len(links))

    for title, href in links[:30]:
        print(title[:120], "---->", href)

except Exception as e:
    print("ERROR:", type(e).__name__)
    print(e)

Status: 403
Title: Just a moment...
Links found: 0


In [15]:
import requests
from bs4 import BeautifulSoup

url = "https://www.undp.org/stories/insuring-future"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=30)

print("Status:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

print("Title:", soup.title.get_text(" ", strip=True))

# Find all links containing insurance/challenge/funding
results = []

for a in soup.find_all("a", href=True):
    text = a.get_text(" ", strip=True)
    href = a["href"]

    if any(word in (text + " " + href).lower()
           for word in ["insurance", "challenge", "fund", "grant"]):
        results.append((text, href))

results = list(dict.fromkeys(results))

print("Relevant links:", len(results))

for text, href in results[:50]:
    print(text[:150], "---->", href)

Status: 403
Title: Access Denied
Relevant links: 0


In [16]:
import pandas as pd

insurance_data = [
    {
        "Funding Name": "UNDP Insurance Innovation Challenge Fund",
        "Description": "Funding to scale innovative and inclusive insurance solutions for underserved populations in developing countries.",
        "Type": "Innovation Grant",
        "Eligibility": "Eligible organisations developing or scaling inclusive insurance solutions in participating developing countries.",
        "Deadline": "Varies by country challenge",
        "States/Country/Region Covered": "Developing countries / participating UNDP IRFF countries",
        "Funding Amount": "Varies by challenge",
        "Website": "https://irff.undp.org/innovation",
        "Category": "Insurance",
        "Status": "Active programme"
    },

    {
        "Funding Name": "Inclusive Insurance Innovation Challenge – Nigeria",
        "Description": "Challenge supporting innovative insurance solutions for financially underserved populations in Nigeria, including climate, health and MSME risk protection.",
        "Type": "Challenge Grant / Prize",
        "Eligibility": "Innovators, startups, insurance companies, fintechs and community-based organisations with eligible insurance solutions and Nigerian registration.",
        "Deadline": "27 October 2025",
        "States/Country/Region Covered": "Nigeria",
        "Funding Amount": "₦30 million per winner; 3 winners",
        "Website": "https://www.undp.org/nigeria/news/inclusive-insurance-innovation-challenge-nigeria",
        "Category": "Inclusive Insurance",
        "Status": "Closed"
    },

    {
        "Funding Name": "Insurance Innovation Challenge – Bangladesh",
        "Description": "Initiative supporting innovative, inclusive and climate-resilient insurance solutions for cottage, micro and small enterprises.",
        "Type": "Innovation Challenge",
        "Eligibility": "Eligible organisations developing insurance solutions for small and vulnerable businesses in Bangladesh.",
        "Deadline": "Not stated",
        "States/Country/Region Covered": "Bangladesh",
        "Funding Amount": "Not stated",
        "Website": "https://www.undp.org/bangladesh/press-releases/insurance-innovation-challenge-launched-small-businesses",
        "Category": "Inclusive Insurance",
        "Status": "Active / Programme launched 2026"
    },

    {
        "Funding Name": "UNDP Pakistan Insurance Innovation Challenge",
        "Description": "Insurance innovation programme supporting context-relevant insurance products and financial protection for vulnerable populations.",
        "Type": "Innovation Challenge",
        "Eligibility": "Eligible organisations developing inclusive insurance solutions in Pakistan.",
        "Deadline": "Not stated",
        "States/Country/Region Covered": "Pakistan",
        "Funding Amount": "Not stated",
        "Website": "https://www.undp.org/pakistan/projects/pakistan-insurance-and-risk-finance-facility",
        "Category": "Inclusive Insurance",
        "Status": "Upcoming / Programme"
    }
]

insurance_df = pd.DataFrame(insurance_data)

print("Shape:", insurance_df.shape)
display(insurance_df)

Shape: (4, 10)


,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,UNDP Insurance Innovation Challenge Fund,Funding to scale innovative and inclusive insu...,Innovation Grant,Eligible organisations developing or scaling i...,Varies by country challenge,Developing countries / participating UNDP IRFF...,Varies by challenge,https://irff.undp.org/innovation,Insurance,Active programme
1,Inclusive Insurance Innovation Challenge – Nig...,Challenge supporting innovative insurance solu...,Challenge Grant / Prize,"Innovators, startups, insurance companies, fin...",27 October 2025,Nigeria,₦30 million per winner; 3 winners,https://www.undp.org/nigeria/news/inclusive-in...,Inclusive Insurance,Closed
2,Insurance Innovation Challenge – Bangladesh,"Initiative supporting innovative, inclusive an...",Innovation Challenge,Eligible organisations developing insurance so...,Not stated,Bangladesh,Not stated,https://www.undp.org/bangladesh/press-releases...,Inclusive Insurance,Active / Programme launched 2026
3,UNDP Pakistan Insurance Innovation Challenge,Insurance innovation programme supporting cont...,Innovation Challenge,Eligible organisations developing inclusive in...,Not stated,Pakistan,Not stated,https://www.undp.org/pakistan/projects/pakista...,Inclusive Insurance,Upcoming / Programme


In [17]:
import pandas as pd

# ============================================================
# VERIFIED INSURANCE FUNDING / CHALLENGE OPPORTUNITIES
# Sources restricted to official organisations
# ============================================================

insurance_data = [

    {
        "Funding Name": "UNDP Inclusive Insurance Innovation Challenge – Nigeria",
        "Description": "Challenge supporting innovative and inclusive insurance solutions for underserved Nigerian populations, including climate, health and MSME insurance.",
        "Type": "Grant / Prize",
        "Eligibility": "Nigerian innovators, startups, insurtechs, fintechs, insurance companies and community-based organisations with a viable insurance solution.",
        "Deadline": "27 October 2025",
        "States/Country/Region Covered": "Nigeria",
        "Funding Amount": "₦30 million per winner; 3 winners",
        "Website": "https://www.undp.org/nigeria/news/inclusive-insurance-innovation-challenge-nigeria",
        "Category": "Insurance",
        "Status": "Closed"
    },

    {
        "Funding Name": "UNDP Inclusive Insurance Challenge Fund – Ethiopia",
        "Description": "Challenge fund supporting proof-of-concept and scaling of innovative inclusive insurance products for underserved communities in Ethiopia.",
        "Type": "Grant / Prize",
        "Eligibility": "Licensed Ethiopian insurance entities, or other organisations partnering with licensed insurance entities.",
        "Deadline": "Closed",
        "States/Country/Region Covered": "Ethiopia",
        "Funding Amount": "$40,000 first-place prize",
        "Website": "https://www.undp.org/ethiopia/news/call-applications-inclusive-insurance-challenge-fund",
        "Category": "Insurance",
        "Status": "Closed"
    },

    {
        "Funding Name": "UNDP Inclusive Insurance Innovation Challenge – Ghana",
        "Description": "Insurance innovation challenge supporting climate and agricultural insurance, women's financial protection, digital insurance access and inclusive insurance.",
        "Type": "Grant / Prize",
        "Eligibility": "Innovators and organisations developing practical and scalable inclusive insurance solutions in Ghana.",
        "Deadline": "Closed",
        "States/Country/Region Covered": "Ghana",
        "Funding Amount": "Not publicly stated",
        "Website": "https://www.undp.org/ghana/press-releases/undp-and-nic-launch-inclusive-insurance-innovation-challenge-safeguard-ghanas-vulnerable-communities",
        "Category": "Insurance",
        "Status": "Closed"
    },

    {
        "Funding Name": "UNDP Insurance Innovation Challenge – Bangladesh",
        "Description": "Challenge supporting innovative insurance solutions that strengthen the resilience of cottage, micro, small and medium enterprises against climate and other risks.",
        "Type": "Grant / Challenge",
        "Eligibility": "Eligible organisations developing insurance solutions for CMSEs in Bangladesh.",
        "Deadline": "Not stated",
        "States/Country/Region Covered": "Bangladesh",
        "Funding Amount": "Not stated",
        "Website": "https://www.undp.org/bangladesh/press-releases/insurance-innovation-challenge-launched-small-businesses",
        "Category": "Insurance",
        "Status": "Open / Programme Launched"
    },

    {
        "Funding Name": "UNDP–ICMIF Insurance Innovation Challenge",
        "Description": "Supports mutual and cooperative insurers to scale innovative, affordable and inclusive insurance products serving underserved households and MSMEs.",
        "Type": "Grant",
        "Eligibility": "Mutual and cooperative insurers in developing economies.",
        "Deadline": "28 April (first round)",
        "States/Country/Region Covered": "Developing economies",
        "Funding Amount": "Up to US$100,000 over two years",
        "Website": "https://icmiffoundation.org/insurance-innovation-challenge/",
        "Category": "Insurance",
        "Status": "Closed – First Round"
    },

    {
        "Funding Name": "UNDP–Generali Insurance Innovation Challenge – Thailand",
        "Description": "Insurance Innovation Challenge focused on developing resilience-oriented insurance solutions for micro, small and medium enterprises.",
        "Type": "Grant / Challenge",
        "Eligibility": "Eligible insurance-sector and innovation partners developing insurance solutions for MSMEs.",
        "Deadline": "Closed",
        "States/Country/Region Covered": "Thailand",
        "Funding Amount": "Not stated",
        "Website": "https://www.undp.org/sites/g/files/zskgke326/files/2025-08/114625199_annex-2-undp_generali-terms-of-reference-iicf-call-for-proposal.pdf",
        "Category": "Insurance",
        "Status": "Closed"
    },

    {
        "Funding Name": "UNDP Insurance Innovation Challenge – Pakistan",
        "Description": "Insurance innovation programme supporting context-relevant insurance products and solutions for vulnerable populations.",
        "Type": "Prize / Challenge",
        "Eligibility": "Eligible organisations developing inclusive insurance solutions in Pakistan.",
        "Deadline": "Closed",
        "States/Country/Region Covered": "Pakistan",
        "Funding Amount": "Up to US$40,000",
        "Website": "https://www.undp.org/pakistan/press-releases/kashf-foundation-wins-undp-pakistans-insurance-innovation-challenge",
        "Category": "Insurance",
        "Status": "Closed / Awarded"
    },

    {
        "Funding Name": "UNDP Insurance Innovation Challenge – Tanzania",
        "Description": "Challenge supporting innovative insurance solutions for underserved populations and livelihoods, including climate and agricultural risks.",
        "Type": "Grant / Prize",
        "Eligibility": "Eligible innovators and insurance-sector organisations developing inclusive insurance solutions in Tanzania.",
        "Deadline": "Closed",
        "States/Country/Region Covered": "Tanzania",
        "Funding Amount": "Not stated",
        "Website": "https://www.undp.org/tanzania",
        "Category": "Insurance",
        "Status": "Closed"
    },

    {
        "Funding Name": "UNDP Insurance Innovation Challenge – Senegal",
        "Description": "Insurance innovation programme supporting agricultural and climate-risk insurance solutions, including parametric insurance.",
        "Type": "Challenge / Funding",
        "Eligibility": "Eligible insurance-sector innovators and organisations.",
        "Deadline": "Closed",
        "States/Country/Region Covered": "Senegal",
        "Funding Amount": "Not stated",
        "Website": "https://sdgfinance.undp.org/sites/default/files/2026-03/Global-Insurance-Innovators-Community-Parametric-Insurance-for-Climate-Action.pdf.pdf",
        "Category": "Insurance",
        "Status": "Closed / Implemented"
    },

    {
        "Funding Name": "UNDP Insurance Innovation Programme – Viet Nam",
        "Description": "Insurance innovation programme supporting digital and parametric insurance solutions for vulnerable agricultural communities.",
        "Type": "Challenge / Funding",
        "Eligibility": "Eligible insurance-sector innovators and organisations.",
        "Deadline": "Closed",
        "States/Country/Region Covered": "Viet Nam",
        "Funding Amount": "Not stated",
        "Website": "https://sdgfinance.undp.org/sites/default/files/2026-03/Global-Insurance-Innovators-Community-Parametric-Insurance-for-Climate-Action.pdf.pdf",
        "Category": "Insurance",
        "Status": "Closed / Implemented"
    },

    {
        "Funding Name": "UNDP Insurance Innovation Programme – Nepal",
        "Description": "Insurance innovation programme developing insurance solutions for vulnerable communities, including technology-enabled livestock protection.",
        "Type": "Challenge / Funding",
        "Eligibility": "Eligible insurance-sector partners and innovators.",
        "Deadline": "Programme ongoing",
        "States/Country/Region Covered": "Nepal",
        "Funding Amount": "Not stated",
        "Website": "https://www.undp.org/stories/insuring-future",
        "Category": "Insurance",
        "Status": "Ongoing"
    },

    {
        "Funding Name": "UNDP Insurance Innovation Programme – Argentina",
        "Description": "Insurance innovation work developing protection solutions for vulnerable ecosystems and communities, including innovative biodiversity insurance.",
        "Type": "Challenge / Funding",
        "Eligibility": "Programme partners and eligible insurance-sector organisations.",
        "Deadline": "Programme ongoing",
        "States/Country/Region Covered": "Argentina",
        "Funding Amount": "Not stated",
        "Website": "https://www.undp.org/stories/insuring-future",
        "Category": "Insurance",
        "Status": "Ongoing"
    },

    {
        "Funding Name": "African Development Bank – ACAPS Insurance Innovation Support",
        "Description": "Funding supporting innovation and financial inclusion in Morocco's insurance sector and strengthening insurance-sector supervision.",
        "Type": "Grant",
        "Eligibility": "ACAPS / eligible institutional implementation partners.",
        "Deadline": "Awarded April 2026",
        "States/Country/Region Covered": "Morocco",
        "Funding Amount": "US$510,000",
        "Website": "https://www.afdb.org/en/news-and-events/press-releases/morocco-african-development-bank-group-approves-510000-support-insurance-and-social-welfare-supervisory-authority-benefit-vulnerable-populations-92718",
        "Category": "Insurance",
        "Status": "Awarded"
    },

    {
        "Funding Name": "Ghana Sub-Sovereign Parametric Flood Risk Insurance",
        "Description": "Insurance opportunity involving a $10 million parametric flood-risk cover for the Greater Accra Metropolitan Area.",
        "Type": "Insurance Contract / Procurement",
        "Eligibility": "Licensed insurance companies in Ghana and eligible global insurers and reinsurers.",
        "Deadline": "EOI process – 2026",
        "States/Country/Region Covered": "Greater Accra, Ghana",
        "Funding Amount": "$10 million insurance cover",
        "Website": "https://www.undp.org/ghana/press-releases/request-expressions-interest",
        "Category": "Insurance",
        "Status": "EOI / Procurement"
    }
]

# ============================================================
# CREATE DATAFRAME
# ============================================================

insurance_df = pd.DataFrame(insurance_data)

# ============================================================
# CLEANING
# ============================================================

insurance_df = insurance_df.drop_duplicates(
    subset=["Funding Name"],
    keep="first"
).reset_index(drop=True)

# Remove accidental whitespace
for col in insurance_df.columns:
    insurance_df[col] = (
        insurance_df[col]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Make sure column order is EXACTLY what you requested
insurance_df = insurance_df[
    [
        "Funding Name",
        "Description",
        "Type",
        "Eligibility",
        "Deadline",
        "States/Country/Region Covered",
        "Funding Amount",
        "Website",
        "Category",
        "Status"
    ]
]

# ============================================================
# CHECKS
# ============================================================

print("Number of records:", len(insurance_df))
print("\nColumns:")
print(insurance_df.columns.tolist())

print("\nStatus breakdown:")
print(insurance_df["Status"].value_counts())

print("\nDuplicate names:", insurance_df["Funding Name"].duplicated().sum())

# Display
display(insurance_df)

Number of records: 14

Columns:
['Funding Name', 'Description', 'Type', 'Eligibility', 'Deadline', 'States/Country/Region Covered', 'Funding Amount', 'Website', 'Category', 'Status']

Status breakdown:
Status
Closed                       5
Closed / Implemented         2
Ongoing                      2
Open / Programme Launched    1
Closed – First Round         1
Closed / Awarded             1
Awarded                      1
EOI / Procurement            1
Name: count, dtype: int64

Duplicate names: 0


,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,UNDP Inclusive Insurance Innovation Challenge ...,Challenge supporting innovative and inclusive ...,Grant / Prize,"Nigerian innovators, startups, insurtechs, fin...",27 October 2025,Nigeria,₦30 million per winner; 3 winners,https://www.undp.org/nigeria/news/inclusive-in...,Insurance,Closed
1,UNDP Inclusive Insurance Challenge Fund – Ethi...,Challenge fund supporting proof-of-concept and...,Grant / Prize,"Licensed Ethiopian insurance entities, or othe...",Closed,Ethiopia,"$40,000 first-place prize",https://www.undp.org/ethiopia/news/call-applic...,Insurance,Closed
2,UNDP Inclusive Insurance Innovation Challenge ...,Insurance innovation challenge supporting clim...,Grant / Prize,Innovators and organisations developing practi...,Closed,Ghana,Not publicly stated,https://www.undp.org/ghana/press-releases/undp...,Insurance,Closed
3,UNDP Insurance Innovation Challenge – Bangladesh,Challenge supporting innovative insurance solu...,Grant / Challenge,Eligible organisations developing insurance so...,Not stated,Bangladesh,Not stated,https://www.undp.org/bangladesh/press-releases...,Insurance,Open / Programme Launched
4,UNDP–ICMIF Insurance Innovation Challenge,Supports mutual and cooperative insurers to sc...,Grant,Mutual and cooperative insurers in developing ...,28 April (first round),Developing economies,"Up to US$100,000 over two years",https://icmiffoundation.org/insurance-innovati...,Insurance,Closed – First Round
5,UNDP–Generali Insurance Innovation Challenge –...,Insurance Innovation Challenge focused on deve...,Grant / Challenge,Eligible insurance-sector and innovation partn...,Closed,Thailand,Not stated,https://www.undp.org/sites/g/files/zskgke326/f...,Insurance,Closed
6,UNDP Insurance Innovation Challenge – Pakistan,Insurance innovation programme supporting cont...,Prize / Challenge,Eligible organisations developing inclusive in...,Closed,Pakistan,"Up to US$40,000",https://www.undp.org/pakistan/press-releases/k...,Insurance,Closed / Awarded
7,UNDP Insurance Innovation Challenge – Tanzania,Challenge supporting innovative insurance solu...,Grant / Prize,Eligible innovators and insurance-sector organ...,Closed,Tanzania,Not stated,https://www.undp.org/tanzania,Insurance,Closed
8,UNDP Insurance Innovation Challenge – Senegal,Insurance innovation programme supporting agri...,Challenge / Funding,Eligible insurance-sector innovators and organ...,Closed,Senegal,Not stated,https://sdgfinance.undp.org/sites/default/file...,Insurance,Closed / Implemented
9,UNDP Insurance Innovation Programme – Viet Nam,Insurance innovation programme supporting digi...,Challenge / Funding,Eligible insurance-sector innovators and organ...,Closed,Viet Nam,Not stated,https://sdgfinance.undp.org/sites/default/file...,Insurance,Closed / Implemented


In [18]:
# Fix the Deadline column
# Keep actual dates where verified; use "Not stated" where the source
# does not provide an exact application deadline.

deadline_updates = {
    "UNDP Inclusive Insurance Innovation Challenge – Nigeria":
        "27 October 2025",

    "UNDP Inclusive Insurance Challenge Fund – Ethiopia":
        "Not stated",

    "UNDP Inclusive Insurance Innovation Challenge – Ghana":
        "Not stated",

    "UNDP Insurance Innovation Challenge – Bangladesh":
        "Not stated",

    "UNDP–ICMIF Insurance Innovation Challenge":
        "28 April 2025",

    "UNDP–Generali Insurance Innovation Challenge – Thailand":
        "Not stated",

    "UNDP Insurance Innovation Challenge – Pakistan":
        "Not stated",

    "UNDP Insurance Innovation Challenge – Tanzania":
        "Not stated",

    "UNDP Insurance Innovation Challenge – Senegal":
        "Not stated",

    "UNDP Insurance Innovation Programme – Viet Nam":
        "Not stated",

    "UNDP Insurance Innovation Programme – Nepal":
        "Not stated",

    "UNDP Insurance Innovation Programme – Argentina":
        "Not stated",

    "African Development Bank – ACAPS Insurance Innovation Support":
        "Not applicable – awarded April 2026",

    "Ghana Sub-Sovereign Parametric Flood Risk Insurance":
        "Not stated"
}

# Apply the corrections
for name, deadline in deadline_updates.items():
    insurance_df.loc[
        insurance_df["Funding Name"] == name,
        "Deadline"
    ] = deadline


# Make sure no row has "Closed" sitting in Deadline
insurance_df["Deadline"] = insurance_df["Deadline"].replace(
    {
        "Closed": "Not stated",
        "Programme ongoing": "Not stated",
        "Awarded April 2026": "Not applicable – awarded April 2026",
        "EOI process – 2026": "Not stated"
    }
)

# Display just the important columns to verify
display(
    insurance_df[
        ["Funding Name", "Deadline", "Status"]
    ]
)

,Funding Name,Deadline,Status
0,UNDP Inclusive Insurance Innovation Challenge ...,27 October 2025,Closed
1,UNDP Inclusive Insurance Challenge Fund – Ethi...,Not stated,Closed
2,UNDP Inclusive Insurance Innovation Challenge ...,Not stated,Closed
3,UNDP Insurance Innovation Challenge – Bangladesh,Not stated,Open / Programme Launched
4,UNDP–ICMIF Insurance Innovation Challenge,28 April 2025,Closed – First Round
5,UNDP–Generali Insurance Innovation Challenge –...,Not stated,Closed
6,UNDP Insurance Innovation Challenge – Pakistan,Not stated,Closed / Awarded
7,UNDP Insurance Innovation Challenge – Tanzania,Not stated,Closed
8,UNDP Insurance Innovation Challenge – Senegal,Not stated,Closed / Implemented
9,UNDP Insurance Innovation Programme – Viet Nam,Not stated,Closed / Implemented


In [19]:
deadline_updates = {
    "UNDP Inclusive Insurance Innovation Challenge – Nigeria":
        "27 October 2025",

    "UNDP Inclusive Insurance Challenge Fund – Ethiopia":
        "10 April 2025",

    "UNDP Inclusive Insurance Innovation Challenge – Ghana":
        "31 May 2025",

    "UNDP Insurance Innovation Challenge – Bangladesh":
        "Not publicly stated",

    "UNDP–ICMIF Insurance Innovation Challenge":
        "28 April 2025",

    "UNDP–Generali Insurance Innovation Challenge – Thailand":
        "30 September 2025",

    "UNDP Insurance Innovation Challenge – Pakistan":
        "27 December 2025",

    "UNDP Insurance Innovation Challenge – Tanzania":
        "15 April 2025",

    "UNDP Insurance Innovation Challenge – Senegal":
        "Not publicly stated",

    "UNDP Insurance Innovation Programme – Viet Nam":
        "Not publicly stated",

    "UNDP Insurance Innovation Programme – Nepal":
        "Not publicly stated",

    "UNDP Insurance Innovation Programme – Argentina":
        "Not publicly stated",

    "African Development Bank – ACAPS Insurance Innovation Support":
        "Not applicable – awarded April 2026",

    "Ghana Sub-Sovereign Parametric Flood Risk Insurance":
        "27 January 2026, 4:00 PM"
}

for name, deadline in deadline_updates.items():
    insurance_df.loc[
        insurance_df["Funding Name"] == name,
        "Deadline"
    ] = deadline

display(
    insurance_df[
        ["Funding Name", "Deadline", "Status"]
    ]
)

,Funding Name,Deadline,Status
0,UNDP Inclusive Insurance Innovation Challenge ...,27 October 2025,Closed
1,UNDP Inclusive Insurance Challenge Fund – Ethi...,10 April 2025,Closed
2,UNDP Inclusive Insurance Innovation Challenge ...,31 May 2025,Closed
3,UNDP Insurance Innovation Challenge – Bangladesh,Not publicly stated,Open / Programme Launched
4,UNDP–ICMIF Insurance Innovation Challenge,28 April 2025,Closed – First Round
5,UNDP–Generali Insurance Innovation Challenge –...,30 September 2025,Closed
6,UNDP Insurance Innovation Challenge – Pakistan,27 December 2025,Closed / Awarded
7,UNDP Insurance Innovation Challenge – Tanzania,15 April 2025,Closed
8,UNDP Insurance Innovation Challenge – Senegal,Not publicly stated,Closed / Implemented
9,UNDP Insurance Innovation Programme – Viet Nam,Not publicly stated,Closed / Implemented


In [20]:
import pandas as pd

# ============================================================
# FINAL 6 VERIFIED INSURANCE FUNDING RECORDS
# Official InsuResilience Solutions Fund sources
# ============================================================

insurance_extra = pd.DataFrame([
    {
        "Funding Name": "Flood Risk Cover for Lagos State, Nigeria",
        "Description": "InsuResilience Solutions Fund premium-financing support for parametric flood insurance protection for Lagos State.",
        "Type": "Insurance / Premium Support Grant",
        "Eligibility": "Project implemented with Lagos State and eligible insurance and risk-financing partners.",
        "Deadline": "Not an application call – project ongoing through January 2028",
        "States/Country/Region Covered": "Lagos State, Nigeria",
        "Funding Amount": "Grant-based premium financing; amount not publicly stated",
        "Website": "https://insuresilience-solutions-fund.org/our-work/premium-financing-support/",
        "Category": "Insurance",
        "Status": "Ongoing"
    },

    {
        "Funding Name": "Scaling Up Agricultural Insurance for Smallholder Farmers in Togo",
        "Description": "ISF-supported project to scale agricultural insurance protecting smallholder farmers against drought and excess rainfall.",
        "Type": "Insurance / Grant",
        "Eligibility": "Implemented through eligible insurance, agricultural and development partners.",
        "Deadline": "Not an application call – project runs May 2025 to June 2027",
        "States/Country/Region Covered": "Togo",
        "Funding Amount": "€999,816",
        "Website": "https://insuresilience-solutions-fund.org/our-work/premium-financing-support/",
        "Category": "Insurance",
        "Status": "Ongoing"
    },

    {
        "Funding Name": "Index Insurance in Uganda – Scaling Agricultural Insurance",
        "Description": "ISF-funded initiative supporting the development and expansion of index-based agricultural insurance for vulnerable smallholder farmers.",
        "Type": "Insurance / Grant",
        "Eligibility": "Eligible insurance, agricultural and implementation partners.",
        "Deadline": "Not an application call – project runs November 2024 to November 2027",
        "States/Country/Region Covered": "Uganda",
        "Funding Amount": "Grant amount not publicly stated",
        "Website": "https://insuresilience-solutions-fund.org/our-work/premium-financing-support/",
        "Category": "Insurance",
        "Status": "Ongoing"
    },

    {
        "Funding Name": "Scaling Up Embedded Area Yield Index Insurance in Ethiopia",
        "Description": "ISF-supported agricultural insurance project expanding climate-risk protection for farmers through an embedded area-yield index insurance product.",
        "Type": "Insurance / Grant",
        "Eligibility": "Eligible agricultural, insurance and implementation partners.",
        "Deadline": "Not an application call – project runs July 2025 to July 2027",
        "States/Country/Region Covered": "Ethiopia",
        "Funding Amount": "€1,308,765",
        "Website": "https://insuresilience-solutions-fund.org/wp-content/uploads/2025/06/C9.06-Ethiopia-Project-Brief.pdf",
        "Category": "Insurance",
        "Status": "Ongoing"
    },

    {
        "Funding Name": "Crop Insurance Programme for Smallholder Farmers in Kenya",
        "Description": "ISF grant-supported programme developing and scaling insurance protection for Kenyan smallholder farmers exposed to climate and drought risks.",
        "Type": "Insurance / Grant",
        "Eligibility": "Implemented through eligible insurance, agricultural and development partners.",
        "Deadline": "Not an application call – grant-funded project",
        "States/Country/Region Covered": "Kenya",
        "Funding Amount": "Grant amount not publicly stated",
        "Website": "https://insuresilience-solutions-fund.org/2021/06/28/crop-insurance-program-for-smallholder-farmers-in-kenya/",
        "Category": "Insurance",
        "Status": "Implemented / Funded"
    },

    {
        "Funding Name": "Financial Protection Against Drought and Flood in Senegal",
        "Description": "ISF-supported project developing and implementing risk-transfer insurance solutions for vulnerable populations exposed to drought and flood.",
        "Type": "Insurance / Grant",
        "Eligibility": "Eligible insurance, government and implementation partners.",
        "Deadline": "Not an application call – grant agreement project",
        "States/Country/Region Covered": "Senegal",
        "Funding Amount": "Grant amount not publicly stated",
        "Website": "https://insuresilience-solutions-fund.org/2023/12/08/scaling-up-financial-protection-against-disasters-for-vulnerable-populations-in-senegal/",
        "Category": "Insurance",
        "Status": "Implemented / Funded"
    }
])


# ============================================================
# COMBINE WITH YOUR EXISTING 14 INSURANCE RECORDS
# ============================================================

insurance_final = pd.concat(
    [insurance_df.copy(), insurance_extra],
    ignore_index=True
)


# ============================================================
# CLEAN THE DATA
# ============================================================

# Make sure the exact required columns exist
required_columns = [
    "Funding Name",
    "Description",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Website",
    "Category",
    "Status"
]

insurance_final = insurance_final[
    [col for col in required_columns if col in insurance_final.columns]
]


# Remove exact duplicate funding names
insurance_final = insurance_final.drop_duplicates(
    subset="Funding Name",
    keep="first"
).reset_index(drop=True)


# Remove accidental whitespace
for col in insurance_final.columns:
    insurance_final[col] = (
        insurance_final[col]
        .fillna("Not stated")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


# ============================================================
# FINAL CHECKS
# ============================================================

print("Number of insurance records:", len(insurance_final))
print("\nColumns:")
print(insurance_final.columns.tolist())

print("\nDuplicate names:")
print(insurance_final["Funding Name"].duplicated().sum())

print("\nStatus breakdown:")
print(insurance_final["Status"].value_counts())

print("\nFinal insurance dataframe:")
display(insurance_final)

Number of insurance records: 20

Columns:
['Funding Name', 'Description', 'Type', 'Eligibility', 'Deadline', 'States/Country/Region Covered', 'Funding Amount', 'Website', 'Category', 'Status']

Duplicate names:
0

Status breakdown:
Status
Ongoing                      6
Closed                       5
Closed / Implemented         2
Implemented / Funded         2
Open / Programme Launched    1
Closed – First Round         1
Closed / Awarded             1
Awarded                      1
EOI / Procurement            1
Name: count, dtype: int64

Final insurance dataframe:


,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,UNDP Inclusive Insurance Innovation Challenge ...,Challenge supporting innovative and inclusive ...,Grant / Prize,"Nigerian innovators, startups, insurtechs, fin...",27 October 2025,Nigeria,₦30 million per winner; 3 winners,https://www.undp.org/nigeria/news/inclusive-in...,Insurance,Closed
1,UNDP Inclusive Insurance Challenge Fund – Ethi...,Challenge fund supporting proof-of-concept and...,Grant / Prize,"Licensed Ethiopian insurance entities, or othe...",10 April 2025,Ethiopia,"$40,000 first-place prize",https://www.undp.org/ethiopia/news/call-applic...,Insurance,Closed
2,UNDP Inclusive Insurance Innovation Challenge ...,Insurance innovation challenge supporting clim...,Grant / Prize,Innovators and organisations developing practi...,31 May 2025,Ghana,Not publicly stated,https://www.undp.org/ghana/press-releases/undp...,Insurance,Closed
3,UNDP Insurance Innovation Challenge – Bangladesh,Challenge supporting innovative insurance solu...,Grant / Challenge,Eligible organisations developing insurance so...,Not publicly stated,Bangladesh,Not stated,https://www.undp.org/bangladesh/press-releases...,Insurance,Open / Programme Launched
4,UNDP–ICMIF Insurance Innovation Challenge,Supports mutual and cooperative insurers to sc...,Grant,Mutual and cooperative insurers in developing ...,28 April 2025,Developing economies,"Up to US$100,000 over two years",https://icmiffoundation.org/insurance-innovati...,Insurance,Closed – First Round
5,UNDP–Generali Insurance Innovation Challenge –...,Insurance Innovation Challenge focused on deve...,Grant / Challenge,Eligible insurance-sector and innovation partn...,30 September 2025,Thailand,Not stated,https://www.undp.org/sites/g/files/zskgke326/f...,Insurance,Closed
6,UNDP Insurance Innovation Challenge – Pakistan,Insurance innovation programme supporting cont...,Prize / Challenge,Eligible organisations developing inclusive in...,27 December 2025,Pakistan,"Up to US$40,000",https://www.undp.org/pakistan/press-releases/k...,Insurance,Closed / Awarded
7,UNDP Insurance Innovation Challenge – Tanzania,Challenge supporting innovative insurance solu...,Grant / Prize,Eligible innovators and insurance-sector organ...,15 April 2025,Tanzania,Not stated,https://www.undp.org/tanzania,Insurance,Closed
8,UNDP Insurance Innovation Challenge – Senegal,Insurance innovation programme supporting agri...,Challenge / Funding,Eligible insurance-sector innovators and organ...,Not publicly stated,Senegal,Not stated,https://sdgfinance.undp.org/sites/default/file...,Insurance,Closed / Implemented
9,UNDP Insurance Innovation Programme – Viet Nam,Insurance innovation programme supporting digi...,Challenge / Funding,Eligible insurance-sector innovators and organ...,Not publicly stated,Viet Nam,Not stated,https://sdgfinance.undp.org/sites/default/file...,Insurance,Closed / Implemented


In [21]:
import pandas as pd
import re

# ============================================================
# 1. REQUIRED COLUMNS
# ============================================================

required_columns = [
    "Funding Name",
    "Description",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Website",
    "Category",
    "Status"
]


# ============================================================
# 2. FIND THE THREE DATASETS
# ============================================================
# This handles the names we've used during the project:
# AI: ai_df / ai_final
# Health: health_df / health_final
# Insurance: insurance_df / insurance_final

def find_dataframe(possible_names):
    for name in possible_names:
        if name in globals():
            obj = globals()[name]
            if isinstance(obj, pd.DataFrame) and len(obj) > 0:
                return obj.copy(), name
    return None, None


ai_data, ai_name = find_dataframe([
    "ai_final",
    "ai_df",
    "AI_df",
    "AI_final"
])

health_data, health_name = find_dataframe([
    "health_final",
    "health_df",
    "Health_df",
    "Health_final"
])

insurance_data, insurance_name = find_dataframe([
    "insurance_final",
    "insurance_df",
    "Insurance_df",
    "Insurance_final"
])


print("DATASETS FOUND")
print("-" * 50)
print("AI:", ai_name, None if ai_data is None else ai_data.shape)
print("Health:", health_name, None if health_data is None else health_data.shape)
print("Insurance:", insurance_name, None if insurance_data is None else insurance_data.shape)


# Stop if any dataset wasn't found
if ai_data is None or health_data is None or insurance_data is None:
    raise ValueError(
        "\nOne or more datasets could not be found.\n"
        "Make sure your AI, Health and Insurance dataframes are still in memory."
    )


# ============================================================
# 3. STANDARDIZE EACH DATASET
# ============================================================

def standardize_dataframe(df, dataset_name):

    df = df.copy()

    # Strip whitespace from column names
    df.columns = df.columns.astype(str).str.strip()

    # Check required columns
    missing = [c for c in required_columns if c not in df.columns]

    if missing:
        raise ValueError(
            f"{dataset_name} is missing these columns: {missing}"
        )

    # Keep ONLY the required columns
    df = df[required_columns].copy()

    # Clean text
    for col in required_columns:
        df[col] = df[col].astype("string").str.strip()

    # Convert blank-like values to proper missing values
    df = df.replace(
        [
            "",
            " ",
            "nan",
            "NaN",
            "None",
            "none",
            "NULL",
            "null"
        ],
        pd.NA
    )

    # Add source dataset temporarily for checking
    df["_Source"] = dataset_name

    return df


ai_data = standardize_dataframe(ai_data, "AI")
health_data = standardize_dataframe(health_data, "Health")
insurance_data = standardize_dataframe(insurance_data, "Insurance")


# ============================================================
# 4. COMBINE
# ============================================================

master_df = pd.concat(
    [ai_data, health_data, insurance_data],
    ignore_index=True
)


print("\nCOMBINED DATASET")
print("-" * 50)
print("Rows:", len(master_df))
print("Columns:", len(required_columns))


# ============================================================
# 5. EXACT DUPLICATE ROWS
# ============================================================

exact_duplicates = master_df[
    master_df.duplicated(
        subset=required_columns,
        keep=False
    )
].sort_values("Funding Name")


print("\nEXACT DUPLICATE ROWS:", len(exact_duplicates))

if len(exact_duplicates) > 0:
    display(exact_duplicates)


# Remove exact duplicate rows
master_df = master_df.drop_duplicates(
    subset=required_columns,
    keep="first"
).reset_index(drop=True)


# ============================================================
# 6. DUPLICATE FUNDING NAMES
# ============================================================

duplicate_names = master_df[
    master_df["Funding Name"]
    .duplicated(keep=False)
].sort_values("Funding Name")


print("\nDUPLICATE FUNDING NAMES:", len(duplicate_names))

if len(duplicate_names) > 0:
    display(
        duplicate_names[
            [
                "Funding Name",
                "Category",
                "Description",
                "Website",
                "Status",
                "_Source"
            ]
        ]
    )


# ============================================================
# 7. NULL / BLANK CHECK
# ============================================================

print("\nNULL VALUES BY COLUMN")
print("-" * 50)

null_counts = master_df[required_columns].isna().sum()

display(
    null_counts.to_frame("Null Count")
)


# Show rows containing actual nulls
rows_with_nulls = master_df[
    master_df[required_columns].isna().any(axis=1)
]

print("\nROWS CONTAINING NULL VALUES:", len(rows_with_nulls))

if len(rows_with_nulls) > 0:
    display(rows_with_nulls)


# ============================================================
# 8. CHECK "NOT STATED" VALUES
# ============================================================

not_stated_counts = {}

for col in required_columns:
    count = master_df[col].astype("string").str.lower().isin([
        "not stated",
        "not specified",
        "not publicly stated",
        "not available",
        "n/a",
        "na",
        "not applicable",
        "varies"
    ]).sum()

    not_stated_counts[col] = count


print("\nPLACEHOLDER VALUES")
print("-" * 50)

display(
    pd.Series(not_stated_counts)
    .to_frame("Placeholder Count")
)


# ============================================================
# 9. WEBSITE VALIDATION
# ============================================================

def valid_url(url):
    if pd.isna(url):
        return False

    url = str(url).strip()

    return bool(
        re.match(
            r"^https?://[^\s]+$",
            url
        )
    )


master_df["_WebsiteValid"] = master_df["Website"].apply(valid_url)


bad_websites = master_df[
    ~master_df["_WebsiteValid"]
][
    [
        "Funding Name",
        "Website",
        "Category",
        "_Source"
    ]
]


print("\nINVALID / MISSING WEBSITES:", len(bad_websites))

if len(bad_websites) > 0:
    display(bad_websites)


# ============================================================
# 10. CHECK FOR MARKDOWN LINKS INSIDE WEBSITE COLUMN
# ============================================================

markdown_websites = master_df[
    master_df["Website"].astype("string").str.contains(
        r"\[.*\]\(https?://",
        regex=True,
        na=False
    )
]

print("\nWEBSITES STILL CONTAINING MARKDOWN:", len(markdown_websites))

if len(markdown_websites) > 0:
    display(
        markdown_websites[
            ["Funding Name", "Website", "_Source"]
        ]
    )


# ============================================================
# 11. CHECK CATEGORY DISTRIBUTION
# ============================================================

print("\nCATEGORY BREAKDOWN")
print("-" * 50)

display(
    master_df["Category"]
    .value_counts(dropna=False)
    .to_frame("Count")
)


# ============================================================
# 12. CHECK STATUS DISTRIBUTION
# ============================================================

print("\nSTATUS BREAKDOWN")
print("-" * 50)

display(
    master_df["Status"]
    .value_counts(dropna=False)
    .to_frame("Count")
)


# ============================================================
# 13. CHECK FOR SUSPICIOUS CROSS-SECTOR DUPLICATES
# ============================================================
# Same description can indicate that the same funding opportunity
# accidentally entered more than once under different names.

description_duplicates = master_df[
    master_df["Description"]
    .duplicated(keep=False)
].sort_values("Description")


print(
    "\nDUPLICATE DESCRIPTIONS:",
    len(description_duplicates)
)

if len(description_duplicates) > 0:
    display(
        description_duplicates[
            [
                "Funding Name",
                "Description",
                "Category",
                "Website",
                "_Source"
            ]
        ]
    )


# ============================================================
# 14. CHECK FOR EMPTY FUNDING NAMES
# ============================================================

empty_names = master_df[
    master_df["Funding Name"].isna()
    | (master_df["Funding Name"].str.strip() == "")
]

print("\nEMPTY FUNDING NAMES:", len(empty_names))

if len(empty_names) > 0:
    display(empty_names)


# ============================================================
# 15. REMOVE INTERNAL CHECK COLUMNS
# ============================================================

master_df = master_df.drop(
    columns=["_WebsiteValid"],
    errors="ignore"
)


# ============================================================
# 16. FINAL COLUMN ORDER
# ============================================================

master_df = master_df[
    required_columns + ["_Source"]
]


# ============================================================
# 17. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("FINAL MASTER DATASET")
print("=" * 60)

print("Total records:", len(master_df))
print("AI records:", (master_df["_Source"] == "AI").sum())
print("Health records:", (master_df["_Source"] == "Health").sum())
print("Insurance records:", (master_df["_Source"] == "Insurance").sum())

print("\nDuplicate funding names:",
      master_df["Funding Name"].duplicated().sum())

print("Exact duplicate rows:",
      master_df.duplicated(subset=required_columns).sum())

print("Rows with nulls:",
      master_df[required_columns].isna().any(axis=1).sum())

print("=" * 60)


# ============================================================
# 18. DISPLAY FINAL DATASET
# ============================================================

display(master_df)


# ============================================================
# 19. SAVE FINAL CSV
# ============================================================

final_csv = "FINAL_AI_HEALTH_INSURANCE_FUNDING.csv"

master_df.to_csv(
    final_csv,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved successfully as:")
print(final_csv)

DATASETS FOUND
--------------------------------------------------
AI: ai_final (19, 10)
Health: None None
Insurance: insurance_final (20, 10)


ValueError: 
One or more datasets could not be found.
Make sure your AI, Health and Insurance dataframes are still in memory.

In [22]:
import pandas as pd

# Find every DataFrame currently available
dfs = {
    name: obj.copy()
    for name, obj in list(globals().items())
    if isinstance(obj, pd.DataFrame)
}

print("DATAFRAMES CURRENTLY IN MEMORY:\n")

for name, df in dfs.items():
    print(f"{name}: {df.shape}")
    print("Columns:", list(df.columns))
    print("-" * 70)

DATAFRAMES CURRENTLY IN MEMORY:

ai_final: (19, 10)
Columns: ['Funding Name', 'Description', 'Type', 'Eligibility', 'Deadline', 'States/Country/Region Covered', 'Funding Amount', 'Website', 'Category', 'Status']
----------------------------------------------------------------------
insurance_raw: (82, 37)
Columns: ['CompanyID', 'CompanyRegistrationType', 'CompanyRegistrationNumber', 'CompanyRegistrationDate', 'CompanyRegistrationRenewalDate', 'CompanyName', 'CompanyAcronyms', 'CompanyCode', 'CompanyWebsite', 'CompanyAddressLine', 'CompanyAddressCityLGA', 'CompanyAddressState', 'CompanyAddressPostcode', 'CompanyPhone', 'CompanyFax', 'CompanyEmail', 'CompanyContactNameTitle', 'CompanyContactNameFirst', 'CompanyContactNameMiddle', 'CompanyContactNameLast', 'CompanyContactAddressLine', 'CompanyContactAddressCityLGA', 'CompanyContactAddressState', 'CompanyContactAddressPostcode', 'CompanyContactPhone', 'CompanyContactFax', 'CompanyContactEmail', 'CompanyCEONameTitle', 'CompanyCEONameFirst',

In [23]:
# Find the Health dataframe automatically
for name, df in {
    name: obj for name, obj in list(globals().items())
    if isinstance(obj, pd.DataFrame)
}.items():
    
    cols = set(df.columns)
    
    if {"Funding Name", "Description", "Type", "Eligibility",
        "Deadline", "States/Country/Region Covered",
        "Funding Amount", "Website", "Category", "Status"}.issubset(cols):
        
        print(name, "->", df.shape)

ai_final -> (19, 10)
insurance_df -> (14, 10)
insurance_final -> (20, 10)
insurance_funding -> (3, 10)
insurance_data -> (20, 10)
insurance_extra -> (6, 10)
ai_data -> (19, 10)
df -> (19, 10)


In [24]:
import os
import pandas as pd

print("=== FILES IN CURRENT FOLDER ===")

files = os.listdir()

for f in files:
    if f.lower().endswith((".csv", ".xlsx", ".xls", ".json", ".txt")):
        print(f)

print("\n=== DATAFRAMES ===")

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(name, obj.shape)

=== FILES IN CURRENT FOLDER ===
verified_ai_funding_final.csv

=== DATAFRAMES ===
ai_final (19, 10)
insurance_raw (82, 37)
insurance_df (14, 10)
insurance_final (20, 10)
insurance_funding (3, 10)
insurance_data (20, 10)
insurance_extra (6, 10)
ai_data (19, 10)
df (19, 10)


In [25]:
import os

print("Current folder:")
print(os.getcwd())

print("\nAll files/folders here:")
for item in os.listdir():
    print(item)

Current folder:
C:\Users\user\Documents\Python lessons\Week5

All files/folders here:
.ipynb_checkpoints
Untitled.ipynb
verified_ai_funding_final.csv


In [26]:
import pandas as pd

# ============================================================
# 1. AI DATA
# ============================================================

ai = ai_final.copy()

# ============================================================
# 2. COMBINE ALL INSURANCE VERSIONS
#    (insurance_raw is EXCLUDED — it is the 82-company NAICOM
#     dataset, not funding opportunities)
# ============================================================

insurance_sources = [
    insurance_df,
    insurance_final,
    insurance_funding,
    insurance_data,
    insurance_extra
]

insurance = pd.concat(
    insurance_sources,
    ignore_index=True
)

# Remove duplicate funding programmes
insurance["Funding Name"] = (
    insurance["Funding Name"]
    .astype(str)
    .str.strip()
)

insurance = insurance.drop_duplicates(
    subset=["Funding Name"],
    keep="first"
).reset_index(drop=True)


# ============================================================
# 3. STANDARDIZE BOTH DATASETS
# ============================================================

columns = [
    "Funding Name",
    "Description",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Website",
    "Category",
    "Status"
]

ai = ai[columns].copy()
insurance = insurance[columns].copy()


# ============================================================
# 4. COMBINE AI + INSURANCE
# ============================================================

ai_insurance = pd.concat(
    [ai, insurance],
    ignore_index=True
)

# Remove any cross-dataset duplicate funding names
ai_insurance = ai_insurance.drop_duplicates(
    subset=["Funding Name"],
    keep="first"
).reset_index(drop=True)


# ============================================================
# 5. SUMMARY
# ============================================================

print("=" * 60)
print("FINAL AI + INSURANCE DATASET")
print("=" * 60)

print("\nAI records:", len(ai))
print("Unique Insurance records:", len(insurance))
print("Combined records:", len(ai_insurance))

print("\nCategory breakdown:")
print(ai_insurance["Category"].value_counts())

print("\nStatus breakdown:")
print(ai_insurance["Status"].value_counts())

print("\nDuplicate Funding Names:",
      ai_insurance["Funding Name"].duplicated().sum())

print("\nMissing values:")
print(ai_insurance.isna().sum())

print("\nFinal shape:", ai_insurance.shape)

print("\nFirst 5 records:")
display(ai_insurance.head())

print("\nLast 5 records:")
display(ai_insurance.tail())

FINAL AI + INSURANCE DATASET

AI records: 19
Unique Insurance records: 23
Combined records: 42

Category breakdown:
Category
Insurance                  20
Artificial Intelligence    13
AI Innovation               6
Insurance Innovation        3
Name: count, dtype: int64

Status breakdown:
Status
Closed                              13
Ongoing                              6
Open                                 4
Upcoming                             4
Active                               4
Closed / Implemented                 2
Implemented / Funded                 2
Varies by opportunity                1
Open / Programme Launched            1
Closed – First Round                 1
Closed / Awarded                     1
Awarded                              1
EOI / Procurement                    1
Active / Programme launched 2026     1
Name: count, dtype: int64

Duplicate Funding Names: 0

Missing values:
Funding Name                     0
Description                      0
Type            

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,AI Fund in Collaboration with Google – NCAIR,Grant programme supporting Nigerian startups d...,Grant,Nigerian startups developing AI-based solutions,Closed (previous round),Nigeria,Up to ₦10 million per startup,https://ncair.nitda.gov.ng/aifund/,AI Innovation,Closed
1,AI Upskill Accelerator Pilot Program,U.S. Economic Development Administration fundi...,Grant Programme,Eligible U.S. entities implementing eligible i...,10 July 2026,United States,Varies by programme,https://www.eda.gov/funding/funding-opportunities,AI Innovation,Open
2,The Genesis Mission: Transforming Science and ...,U.S. Department of Energy funding supporting s...,Research Grant,"Eligible universities, research organisations,...",17 December 2026,United States,"$500,000–$16 million",https://www.grants.gov/,AI Innovation,Open
3,Innovate UK Frontier Artificial Intelligence D...,Funding supporting feasibility and discovery w...,Grant,"Eligible UK businesses, research organisations...",10 June 2026,United Kingdom,Up to £2.5 million total funding,https://www.ukri.org/opportunity/,AI Innovation,Closed
4,Future Computing Paradigms Network Plus,UK research funding supporting collaboration a...,Research Grant,Researchers based at eligible UK research orga...,13 October 2026,United Kingdom,£2.8 million total funding,https://www.ukri.org/opportunity/future-comput...,AI Innovation,Upcoming



Last 5 records:


,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
37,Crop Insurance Programme for Smallholder Farme...,ISF grant-supported programme developing and s...,Insurance / Grant,"Implemented through eligible insurance, agricu...",Not an application call – grant-funded project,Kenya,Grant amount not publicly stated,https://insuresilience-solutions-fund.org/2021...,Insurance,Implemented / Funded
38,Financial Protection Against Drought and Flood...,ISF-supported project developing and implement...,Insurance / Grant,"Eligible insurance, government and implementat...",Not an application call – grant agreement project,Senegal,Grant amount not publicly stated,https://insuresilience-solutions-fund.org/2023...,Insurance,Implemented / Funded
39,Inclusive Insurance Innovation Challenge Nigeria,Challenge supporting innovative insurance solu...,Innovation Challenge / Grant,"Innovators, startups, insurtechs, fintechs, in...",27 October 2025,Nigeria,₦30 million per winning proposal,https://www.undp.org/nigeria/news/inclusive-in...,Insurance Innovation,Closed
40,Insurance Innovation Challenge Fund,UNDP and ICMIF Foundation fund supporting inno...,Innovation Challenge Fund,Mutual and cooperative insurers and eligible o...,Previous application round closed,Developing countries,"Fund initially launched with US$600,000",https://icmiffoundation.org/insurance-innovati...,Insurance Innovation,Closed
41,Insurance Innovation Challenge – Bangladesh,UNDP and Bangladesh SME Foundation initiative ...,Innovation Challenge,Eligible innovators and organisations developi...,Not stated,Bangladesh,Not stated,https://www.undp.org/bangladesh/press-releases...,Insurance Innovation,Active / Programme launched 2026


In [27]:
ai_insurance.to_csv(
    "ai_insurance_master_42.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully!")
print("Rows:", len(ai_insurance))
print("Columns:", len(ai_insurance.columns))

Saved successfully!
Rows: 42
Columns: 10


In [28]:
import pandas as pd
import numpy as np
import re
import os

# ============================================================
# 1. FIND YOUR EXISTING 42-RECORD DATASET
# ============================================================

required_columns = [
    "Funding Name",
    "Description",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Website",
    "Category",
    "Status"
]

existing_df = None

# First look for a dataframe with exactly the required structure
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        if all(col in obj.columns for col in required_columns):
            # Prefer the largest relevant dataframe
            if existing_df is None or len(obj) > len(existing_df):
                existing_df = obj.copy()

if existing_df is None:
    raise ValueError(
        "I could not find your combined AI + Insurance dataframe. "
        "Make sure the 42-record dataframe is still in memory."
    )

print("Existing dataset found:", existing_df.shape)


# ============================================================
# 2. VERIFIED INSURANCE GRANT-FUNDED PROJECTS
# ============================================================
#
# These are NOT fabricated "open grants".
#
# They are genuine insurance-related grant-funded projects
# documented by the InsuResilience Solutions Fund (ISF).
#
# Where there was no public application deadline, we explicitly
# say that it was a grant agreement/project rather than inventing
# a deadline.
#
# Kenya and Senegal are intentionally NOT included because they
# already exist in your current dataset.
# ============================================================

insurance_verified_additions = [

    {
        "Funding Name":
            "Development of a Natural Disaster Risk Insurance Scheme for Nagaland",
        "Description":
            "ISF grant-funded project supporting development of a natural disaster risk insurance scheme for Nagaland's State Disaster Response Mitigation Fund.",
        "Type":
            "Insurance Grant",
        "Eligibility":
            "Project implemented through Nagaland State Disaster Management Authority and insurance-sector partners; not an open individual application call.",
        "Deadline":
            "Grant agreement signed 18 March 2021; not an open application deadline",
        "States/Country/Region Covered":
            "Nagaland, India",
        "Funding Amount":
            "Grant amount not publicly stated",
        "Website":
            "https://insuresilience-solutions-fund.org/2021/03/18/development-of-a-natural-disaster-risk-insurance-scheme-for-nagaland/",
        "Category":
            "Insurance",
        "Status":
            "Implemented / Funded"
    },

    {
        "Funding Name":
            "Flood Risk Cover for Lagos State in Nigeria",
        "Description":
            "ISF grant-funded project supporting development and implementation of a sub-sovereign index-based flood insurance solution for vulnerable populations in Lagos State.",
        "Type":
            "Insurance Grant",
        "Eligibility":
            "Implemented through Lagos State Government and insurance and risk-finance partners; not an open application call.",
        "Deadline":
            "Grant agreement signed 10 November 2022; not an open application deadline",
        "States/Country/Region Covered":
            "Lagos State, Nigeria",
        "Funding Amount":
            "EUR 782,895 grant",
        "Website":
            "https://insuresilience-solutions-fund.org/2022/11/10/flood-risk-cover-for-lagos-state-in-nigeria/",
        "Category":
            "Insurance",
        "Status":
            "Implemented / Funded"
    },

    {
        "Funding Name":
            "Integrated Financial and Technical Services for Agriculture in Burkina Faso",
        "Description":
            "ISF grant-funded project supporting parametric drought and excess-rainfall insurance for smallholder farmers in Burkina Faso.",
        "Type":
            "Insurance Grant",
        "Eligibility":
            "Implemented through Yelen Assurance, RCPB and Cordaid; not an open individual application call.",
        "Deadline":
            "Grant agreement signed 23 May 2024; not an open application deadline",
        "States/Country/Region Covered":
            "Burkina Faso",
        "Funding Amount":
            "Grant amount not publicly stated",
        "Website":
            "https://insuresilience-solutions-fund.org/2024/05/23/integrated-financial-and-technical-services-for-agriculture-in-burkina-faso/",
        "Category":
            "Insurance",
        "Status":
            "Implemented / Funded"
    },

    {
        "Funding Name":
            "Enhancing Resilience of Smallholder Farmers in Southern Malawi through a Risk Layering Approach",
        "Description":
            "ISF grant-funded project supporting development of climate-risk insurance solutions for smallholder farmers exposed to droughts, floods and cyclones.",
        "Type":
            "Insurance Grant",
        "Eligibility":
            "Implemented through Opportunity International Malawi and insurance-sector partners; not an open application call.",
        "Deadline":
            "Grant agreement signed 10 October 2024; not an open application deadline",
        "States/Country/Region Covered":
            "Southern Malawi",
        "Funding Amount":
            "Grant amount not publicly stated",
        "Website":
            "https://insuresilience-solutions-fund.org/2024/10/10/enhancing-resilience-of-smallholder-farmers-in-southern-malawi-through-a-risk-layering-approach-for-managing-climate-related-risks/",
        "Category":
            "Insurance",
        "Status":
            "Implemented / Funded"
    },

    {
        "Funding Name":
            "Securing Livestock Livelihoods in Kyrgyzstan through Forecast Index Insurance",
        "Description":
            "ISF grant-funded project supporting development and implementation of a forecast-based livestock insurance solution for vulnerable rural households.",
        "Type":
            "Insurance Grant",
        "Eligibility":
            "Implemented through Blue Marble Microinsurance, local insurers and agricultural partners; not an open application call.",
        "Deadline":
            "Grant agreement announced 17 December 2024; not an open application deadline",
        "States/Country/Region Covered":
            "Kyrgyzstan",
        "Funding Amount":
            "Grant amount not publicly stated",
        "Website":
            "https://insuresilience-solutions-fund.org/2024/12/17/securing-livestock-livelihoods-in-kyrgyzstan-leveraging-meso-level-forecast-index-insurance-solutions/",
        "Category":
            "Insurance",
        "Status":
            "Implemented / Funded"
    },

    {
        "Funding Name":
            "Rice Value Chain Climate Cover in Côte d'Ivoire",
        "Description":
            "ISF grant-funded project supporting drought and excess-rainfall insurance for vulnerable rice producers through a hybrid weather and area-yield insurance solution.",
        "Type":
            "Insurance Grant",
        "Eligibility":
            "Implemented through ARC Ltd, FUSCOP and technical partners; not an open application call.",
        "Deadline":
            "Grant agreement signed 11 June 2024; not an open application deadline",
        "States/Country/Region Covered":
            "Côte d'Ivoire",
        "Funding Amount":
            "Grant amount not publicly stated",
        "Website":
            "https://insuresilience-solutions-fund.org/2024/06/11/rice-value-chain-climate-cover-in-cote-divoire/",
        "Category":
            "Insurance",
        "Status":
            "Implemented / Funded"
    },

    {
        "Funding Name":
            "Flood Protection for Vulnerable Areas in Togo",
        "Description":
            "ISF grant-funded project supporting an index-based flood insurance solution for vulnerable populations in Lomé and Kpalimé.",
        "Type":
            "Insurance Grant",
        "Eligibility":
            "Implemented through AXA Climate, PADIE and reinsurance partners; not an open application call.",
        "Deadline":
            "Grant agreement signed 4 March 2024; not an open application deadline",
        "States/Country/Region Covered":
            "Togo",
        "Funding Amount":
            "Grant amount not publicly stated",
        "Website":
            "https://insuresilience-solutions-fund.org/2024/03/04/flood-protection-for-vulnerable-areas-in-togo/",
        "Category":
            "Insurance",
        "Status":
            "Implemented / Funded"
    },

    {
        "Funding Name":
            "Building Climate-Resilient Agro-Ecosystems of Smallholder Farming",
        "Description":
            "ISF grant-funded project supporting the expansion and improvement of crop and livestock insurance solutions for vulnerable smallholder farmers.",
        "Type":
            "Insurance Grant",
        "Eligibility":
            "Implemented through DHAN Foundation, People Mutuals, IBISA and project partners; not an open application call.",
        "Deadline":
            "Grant agreement announced 15 June 2022; not an open application deadline",
        "States/Country/Region Covered":
            "India",
        "Funding Amount":
            "Grant amount not publicly stated",
        "Website":
            "https://insuresilience-solutions-fund.org/2022/06/15/building-climate-resilient-agro-ecosystems-of-smallholder-farming/",
        "Category":
            "Insurance",
        "Status":
            "Implemented / Funded"
    },

    {
        "Funding Name":
            "Increased Resilience Against Drought and Extreme Rainfall for Smallholder Farmers in Uganda",
        "Description":
            "ISF grant-funded project supporting improvement and development of agricultural insurance products for Ugandan smallholder farmers.",
        "Type":
            "Insurance Grant",
        "Eligibility":
            "Implemented through Sanlam, AgroConsortium, eLEAF and other insurance-sector partners; not an open application call.",
        "Deadline":
            "Grant agreement announced 20 September 2021; not an open application deadline",
        "States/Country/Region Covered":
            "Uganda",
        "Funding Amount":
            "Grant amount not publicly stated",
        "Website":
            "https://insuresilience-solutions-fund.org/2021/09/20/increased-resilience-against-drought-and-extreme-rainfall-for-smallholder-farmers-in-uganda/",
        "Category":
            "Insurance",
        "Status":
            "Implemented / Funded"
    },

    {
        "Funding Name":
            "Scaling Up and Improving National Agriculture Insurance Scheme in Rwanda",
        "Description":
            "ISF grant-funded project supporting improvement and expansion of livestock and crop insurance products for Rwanda's smallholder farmers.",
        "Type":
            "Insurance Grant",
        "Eligibility":
            "Implemented through Rwanda's agriculture insurance ecosystem and project partners; not an open application call.",
        "Deadline":
            "Grant agreement announced 1 March 2022; not an open application deadline",
        "States/Country/Region Covered":
            "Rwanda",
        "Funding Amount":
            "Grant amount not publicly stated",
        "Website":
            "https://insuresilience-solutions-fund.org/2022/03/01/scaling-up-and-improving-national-agriculture-insurance-scheme-in-rwanda/",
        "Category":
            "Insurance",
        "Status":
            "Implemented / Funded"
    },

    {
        "Funding Name":
            "MAR Insurance Programme Premium Financing Endowment",
        "Description":
            "ISF grant-based premium financing support for climate-risk insurance designed to improve affordability and resilience in the Mesoamerican Reef region.",
        "Type":
            "Insurance Grant",
        "Eligibility":
            "ISF-supported project partners implementing climate-risk insurance solutions; not an open individual application call.",
        "Deadline":
            "Project brief published 3 December 2024; not an open application deadline",
        "States/Country/Region Covered":
            "Mesoamerican Reef Region",
        "Funding Amount":
            "Grant amount not publicly stated",
        "Website":
            "https://insuresilience-solutions-fund.org/2024/12/03/project-brief-p1-08-c3-25-mar-fund/",
        "Category":
            "Insurance",
        "Status":
            "Implemented / Funded"
    },

    {
        "Funding Name":
            "Caribbean Regional Reef Insurance Programme",
        "Description":
            "ISF grant-funded project supporting development and implementation of a regional reef insurance programme to finance post-hurricane response and strengthen climate resilience.",
        "Type":
            "Insurance Grant",
        "Eligibility":
            "Implemented through Caribbean environmental funds and insurance/reinsurance partners; not an open application call.",
        "Deadline":
            "Grant project period September 2025–October 2027; not an open application deadline",
        "States/Country/Region Covered":
            "Caribbean",
        "Funding Amount":
            "EUR 720,395 grant",
        "Website":
            "https://insuresilience-solutions-fund.org/wp-content/uploads/2025/11/C10.16-Caribbean-Project-Brief.pdf",
        "Category":
            "Insurance",
        "Status":
            "Ongoing / Funded"
    }
]


# ============================================================
# 3. CREATE ADDITIONS DATAFRAME
# ============================================================

insurance_additions_df = pd.DataFrame(insurance_verified_additions)

# Force exact column order
insurance_additions_df = insurance_additions_df[required_columns]


# ============================================================
# 4. STANDARDIZE EXISTING DATA
# ============================================================

existing_df = existing_df[required_columns].copy()

for col in required_columns:
    existing_df[col] = existing_df[col].astype(str).str.strip()

    # Turn common fake-null strings into actual blanks
    existing_df[col] = existing_df[col].replace(
        ["nan", "None", "none", "NaN", ""],
        np.nan
    )

# Clean funding names for duplicate detection
existing_df["_name_clean"] = (
    existing_df["Funding Name"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z0-9]+", " ", regex=True)
    .str.strip()
)


insurance_additions_df["_name_clean"] = (
    insurance_additions_df["Funding Name"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z0-9]+", " ", regex=True)
    .str.strip()
)


# ============================================================
# 5. REMOVE ANY ADDITION THAT ALREADY EXISTS
# ============================================================

existing_names = set(existing_df["_name_clean"])

before_additions = len(insurance_additions_df)

insurance_additions_df = insurance_additions_df[
    ~insurance_additions_df["_name_clean"].isin(existing_names)
].copy()

removed_existing = before_additions - len(insurance_additions_df)

print("New verified insurance records:", len(insurance_additions_df))
print("Possible duplicates removed:", removed_existing)


# ============================================================
# 6. COMBINE
# ============================================================

final_df = pd.concat(
    [
        existing_df.drop(columns=["_name_clean"]),
        insurance_additions_df.drop(columns=["_name_clean"])
    ],
    ignore_index=True
)


# ============================================================
# 7. FINAL CLEANING
# ============================================================

# Strip whitespace
for col in required_columns:
    final_df[col] = final_df[col].astype(str).str.strip()

# Convert fake nulls to actual NaN
final_df = final_df.replace(
    ["nan", "None", "none", "NaN", ""],
    np.nan
)

# Remove exact duplicate rows
final_df = final_df.drop_duplicates().reset_index(drop=True)


# ============================================================
# 8. DUPLICATE FUNDING NAME CHECK
# ============================================================

name_check = (
    final_df["Funding Name"]
    .fillna("")
    .str.lower()
    .str.replace(r"[^a-z0-9]+", " ", regex=True)
    .str.strip()
)

duplicate_names = final_df[name_check.duplicated(keep=False)]

print("\n================ FINAL CHECK ================")

print("Final records:", len(final_df))
print("Final columns:", len(final_df.columns))

print("\nDuplicate Funding Names:", len(duplicate_names))

if len(duplicate_names) > 0:
    print("\nDUPLICATE NAMES:")
    print(
        final_df.loc[
            name_check.duplicated(keep=False),
            ["Funding Name", "Website"]
        ].to_string(index=False)
    )


# ============================================================
# 9. NULL / BLANK CHECK
# ============================================================

missing_values = final_df.isna().sum()

print("\nMissing values:")
print(missing_values)

total_missing = missing_values.sum()

print("\nTotal missing cells:", total_missing)


# ============================================================
# 10. DUPLICATE WEBSITE CHECK
# ============================================================

website_series = (
    final_df["Website"]
    .fillna("")
    .str.lower()
    .str.strip()
)

duplicate_websites = website_series[
    (website_series != "") &
    website_series.duplicated(keep=False)
]

print("\nDuplicate website URLs:", len(duplicate_websites))

if len(duplicate_websites) > 0:
    print("\nRepeated URLs:")
    print(duplicate_websites.to_string())


# ============================================================
# 11. URL FORMAT CHECK
# ============================================================

bad_urls = final_df[
    ~final_df["Website"].astype(str).str.match(
        r"^https?://",
        na=False
    )
]

print("\nRecords with suspicious website format:", len(bad_urls))

if len(bad_urls) > 0:
    print(
        bad_urls[
            ["Funding Name", "Website"]
        ].to_string(index=False)
    )


# ============================================================
# 12. CATEGORY BREAKDOWN
# ============================================================

print("\nCategory breakdown:")
print(final_df["Category"].value_counts(dropna=False))


# ============================================================
# 13. STATUS BREAKDOWN
# ============================================================

print("\nStatus breakdown:")
print(final_df["Status"].value_counts(dropna=False))


# ============================================================
# 14. TYPE BREAKDOWN
# ============================================================

print("\nType breakdown:")
print(final_df["Type"].value_counts(dropna=False))


# ============================================================
# 15. FINAL QUALITY TEST
# ============================================================

checks = {
    "Correct number of columns":
        list(final_df.columns) == required_columns,

    "No duplicate funding names":
        len(duplicate_names) == 0,

    "No missing values":
        total_missing == 0,

    "No duplicate complete rows":
        not final_df.duplicated().any(),

    "All websites have valid URL format":
        len(bad_urls) == 0
}

print("\n================ QUALITY TEST ================")

for check, result in checks.items():
    print(f"{'PASS' if result else 'FAIL'} - {check}")


# ============================================================
# 16. SAVE FINAL DATASET
# ============================================================

output_file = "FINAL_AI_INSURANCE_FUNDING.csv"

final_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n==============================================")
print("FINAL FILE SAVED:")
print(os.path.abspath(output_file))
print("==============================================")


# ============================================================
# 17. DISPLAY FINAL DATASET
# ============================================================

print("\nFIRST 10 RECORDS:")
display(final_df.head(10))

print("\nLAST 10 RECORDS:")
display(final_df.tail(10))

Existing dataset found: (42, 10)
New verified insurance records: 12
Possible duplicates removed: 0

================ FINAL CHECK ================
Final records: 54
Final columns: 10

Duplicate Funding Names: 0

Missing values:
Funding Name                     0
Description                      0
Type                             0
Eligibility                      0
Deadline                         0
States/Country/Region Covered    0
Funding Amount                   0
Website                          0
Category                         0
Status                           0
dtype: int64

Total missing cells: 0

Duplicate website URLs: 24

Repeated URLs:
2                               https://www.grants.gov/
5                               https://www.grants.gov/
6     https://ec.europa.eu/info/funding-tenders/oppo...
7     https://ec.europa.eu/info/funding-tenders/oppo...
8     https://ec.europa.eu/info/funding-tenders/oppo...
10                          https://www.nsf.gov/funding
13    

,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
0,AI Fund in Collaboration with Google – NCAIR,Grant programme supporting Nigerian startups d...,Grant,Nigerian startups developing AI-based solutions,Closed (previous round),Nigeria,Up to ₦10 million per startup,https://ncair.nitda.gov.ng/aifund/,AI Innovation,Closed
1,AI Upskill Accelerator Pilot Program,U.S. Economic Development Administration fundi...,Grant Programme,Eligible U.S. entities implementing eligible i...,10 July 2026,United States,Varies by programme,https://www.eda.gov/funding/funding-opportunities,AI Innovation,Open
2,The Genesis Mission: Transforming Science and ...,U.S. Department of Energy funding supporting s...,Research Grant,"Eligible universities, research organisations,...",17 December 2026,United States,"$500,000–$16 million",https://www.grants.gov/,AI Innovation,Open
3,Innovate UK Frontier Artificial Intelligence D...,Funding supporting feasibility and discovery w...,Grant,"Eligible UK businesses, research organisations...",10 June 2026,United Kingdom,Up to £2.5 million total funding,https://www.ukri.org/opportunity/,AI Innovation,Closed
4,Future Computing Paradigms Network Plus,UK research funding supporting collaboration a...,Research Grant,Researchers based at eligible UK research orga...,13 October 2026,United Kingdom,£2.8 million total funding,https://www.ukri.org/opportunity/future-comput...,AI Innovation,Upcoming
5,AI Pathways To The Future,Grant programme supporting AI-related educatio...,Grant,Eligible organisations applying under the rele...,9 August 2026,Indonesia,Not stated,https://www.grants.gov/,AI Innovation,Open
6,Horizon Europe Digital Calls – Trustworthy Art...,Horizon Europe funding supporting trustworthy ...,Funding Opportunity,"Eligible research organisations, universities,...",15 April 2026,European Union and Horizon Europe Associated C...,€221.8 million across related Digital call topics,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
7,Horizon Europe Next-Generation AI Agents for R...,Research and innovation funding supporting nex...,Funding Opportunity,"Eligible research organisations, universities,...",15 April 2026,Europe,€38 million,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
8,Horizon Europe Efficient and Compliant Access ...,Innovation funding supporting efficient and co...,Funding Opportunity,"Eligible research institutions, companies and ...",15 April 2026,Europe,€46.5 million,https://ec.europa.eu/info/funding-tenders/oppo...,Artificial Intelligence,Closed
9,European Innovation Council (EIC) Advanced Inn...,Horizon Europe challenge funding supporting br...,Grant,"Startups, SMEs and eligible research teams",Varies by challenge,European Union and Horizon Europe Associated C...,Varies by challenge,https://eic.ec.europa.eu/eic-funding-opportuni...,Artificial Intelligence,Upcoming



LAST 10 RECORDS:


,Funding Name,Description,Type,Eligibility,Deadline,States/Country/Region Covered,Funding Amount,Website,Category,Status
44,Integrated Financial and Technical Services fo...,ISF grant-funded project supporting parametric...,Insurance Grant,"Implemented through Yelen Assurance, RCPB and ...",Grant agreement signed 23 May 2024; not an ope...,Burkina Faso,Grant amount not publicly stated,https://insuresilience-solutions-fund.org/2024...,Insurance,Implemented / Funded
45,Enhancing Resilience of Smallholder Farmers in...,ISF grant-funded project supporting developmen...,Insurance Grant,Implemented through Opportunity International ...,Grant agreement signed 10 October 2024; not an...,Southern Malawi,Grant amount not publicly stated,https://insuresilience-solutions-fund.org/2024...,Insurance,Implemented / Funded
46,Securing Livestock Livelihoods in Kyrgyzstan t...,ISF grant-funded project supporting developmen...,Insurance Grant,Implemented through Blue Marble Microinsurance...,Grant agreement announced 17 December 2024; no...,Kyrgyzstan,Grant amount not publicly stated,https://insuresilience-solutions-fund.org/2024...,Insurance,Implemented / Funded
47,Rice Value Chain Climate Cover in Côte d'Ivoire,ISF grant-funded project supporting drought an...,Insurance Grant,"Implemented through ARC Ltd, FUSCOP and techni...",Grant agreement signed 11 June 2024; not an op...,Côte d'Ivoire,Grant amount not publicly stated,https://insuresilience-solutions-fund.org/2024...,Insurance,Implemented / Funded
48,Flood Protection for Vulnerable Areas in Togo,ISF grant-funded project supporting an index-b...,Insurance Grant,"Implemented through AXA Climate, PADIE and rei...",Grant agreement signed 4 March 2024; not an op...,Togo,Grant amount not publicly stated,https://insuresilience-solutions-fund.org/2024...,Insurance,Implemented / Funded
49,Building Climate-Resilient Agro-Ecosystems of ...,ISF grant-funded project supporting the expans...,Insurance Grant,"Implemented through DHAN Foundation, People Mu...",Grant agreement announced 15 June 2022; not an...,India,Grant amount not publicly stated,https://insuresilience-solutions-fund.org/2022...,Insurance,Implemented / Funded
50,Increased Resilience Against Drought and Extre...,ISF grant-funded project supporting improvemen...,Insurance Grant,"Implemented through Sanlam, AgroConsortium, eL...",Grant agreement announced 20 September 2021; n...,Uganda,Grant amount not publicly stated,https://insuresilience-solutions-fund.org/2021...,Insurance,Implemented / Funded
51,Scaling Up and Improving National Agriculture ...,ISF grant-funded project supporting improvemen...,Insurance Grant,Implemented through Rwanda's agriculture insur...,Grant agreement announced 1 March 2022; not an...,Rwanda,Grant amount not publicly stated,https://insuresilience-solutions-fund.org/2022...,Insurance,Implemented / Funded
52,MAR Insurance Programme Premium Financing Endo...,ISF grant-based premium financing support for ...,Insurance Grant,ISF-supported project partners implementing cl...,Project brief published 3 December 2024; not a...,Mesoamerican Reef Region,Grant amount not publicly stated,https://insuresilience-solutions-fund.org/2024...,Insurance,Implemented / Funded
53,Caribbean Regional Reef Insurance Programme,ISF grant-funded project supporting developmen...,Insurance Grant,Implemented through Caribbean environmental fu...,Grant project period September 2025–October 20...,Caribbean,"EUR 720,395 grant",https://insuresilience-solutions-fund.org/wp-c...,Insurance,Ongoing / Funded


In [29]:
import pandas as pd
import re

# ============================================================
# FINAL NON-DESTRUCTIVE DATA QUALITY AUDIT
# ============================================================

# Find the 54-record dataframe automatically
candidate_dfs = []

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        if obj.shape[1] == 10 and "Funding Name" in obj.columns:
            candidate_dfs.append((name, obj))

print("DATAFRAMES WITH FUNDING SCHEMA:")
for name, obj in candidate_dfs:
    print(f"  {name}: {obj.shape}")

# Prefer the dataframe with the largest number of records
if not candidate_dfs:
    raise ValueError("No dataframe with the required 10-column funding structure was found.")

df_check = max(candidate_dfs, key=lambda x: len(x[1]))[1].copy()

print("\nUsing dataframe:", [x[0] for x in candidate_dfs if x[1] is df_check][0])
print("Shape:", df_check.shape)


# ============================================================
# 1. REQUIRED COLUMNS
# ============================================================

required_columns = [
    "Funding Name",
    "Description",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Website",
    "Category",
    "Status"
]

print("\n" + "="*60)
print("1. COLUMN CHECK")
print("="*60)

missing_columns = [
    col for col in required_columns
    if col not in df_check.columns
]

if missing_columns:
    print("FAIL - Missing columns:", missing_columns)
else:
    print("PASS - All required columns are present")


# ============================================================
# 2. EXACT DUPLICATES
# ============================================================

print("\n" + "="*60)
print("2. DUPLICATE CHECK")
print("="*60)

duplicate_names = df_check[
    df_check["Funding Name"].duplicated(keep=False)
]

if len(duplicate_names) == 0:
    print("PASS - No duplicate Funding Names")
else:
    print("WARNING - Duplicate Funding Names found:")
    print(duplicate_names[["Funding Name", "Website"]].to_string(index=False))


duplicate_rows = df_check[df_check.duplicated(keep=False)]

if len(duplicate_rows) == 0:
    print("PASS - No duplicate complete rows")
else:
    print("WARNING - Duplicate complete rows:", len(duplicate_rows))


# ============================================================
# 3. NEAR-DUPLICATE FUNDING NAMES
# ============================================================

print("\n" + "="*60)
print("3. POSSIBLE NEAR-DUPLICATES")
print("="*60)

names = (
    df_check["Funding Name"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z0-9 ]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

normalized_duplicates = names[names.duplicated(keep=False)]

if len(normalized_duplicates) == 0:
    print("PASS - No obvious normalized-name duplicates")
else:
    print("WARNING - Possible near-duplicates:")
    print(df_check.loc[
        normalized_duplicates.index,
        ["Funding Name", "Category", "Website"]
    ].to_string(index=False))


# ============================================================
# 4. MISSING / PLACEHOLDER VALUES
# ============================================================

print("\n" + "="*60)
print("4. PLACEHOLDER VALUE CHECK")
print("="*60)

placeholders = [
    "not stated",
    "not specified",
    "not available",
    "n/a",
    "na",
    "none",
    "unknown",
    "not applicable",
    "varies by programme",
    "varies by opportunity",
    "previous application round closed",
    "not publicly stated"
]

for col in required_columns:
    values = df_check[col].astype(str).str.strip().str.lower()

    found = df_check[
        values.isin(placeholders)
    ]

    if len(found) > 0:
        print(f"\n{col}: {len(found)} placeholder values")
        print(found[["Funding Name", col]].to_string(index=False))

print("\nNOTE: Placeholder values are NOT automatically errors.")
print("They simply need human review.")


# ============================================================
# 5. WEBSITE FORMAT CHECK
# ============================================================

print("\n" + "="*60)
print("5. WEBSITE CHECK")
print("="*60)

url_pattern = re.compile(
    r"^https?://[^\s]+$",
    re.IGNORECASE
)

bad_urls = []

for i, url in df_check["Website"].items():
    url_string = str(url).strip()

    if not url_pattern.match(url_string):
        bad_urls.append((i, url_string))

if len(bad_urls) == 0:
    print("PASS - All website fields have URL format")
else:
    print("WARNING - Possible bad URLs:")
    for item in bad_urls:
        print(item)


# ============================================================
# 6. DUPLICATE WEBSITES
# ============================================================

print("\n" + "="*60)
print("6. REPEATED WEBSITE CHECK")
print("="*60)

website_counts = df_check["Website"].value_counts()

repeated_websites = website_counts[
    website_counts > 1
]

if len(repeated_websites) == 0:
    print("PASS - No repeated websites")
else:
    print(
        f"INFO - {len(repeated_websites)} website URLs are shared "
        f"by multiple records."
    )

    for website, count in repeated_websites.items():
        print(f"\n{count} records -> {website}")

        print(
            df_check.loc[
                df_check["Website"] == website,
                "Funding Name"
            ].tolist()
        )

print("\nIMPORTANT:")
print("Repeated websites are NOT automatically duplicates.")
print("Different opportunities can legitimately use the same official funding portal.")


# ============================================================
# 7. STATUS / DEADLINE REVIEW
# ============================================================

print("\n" + "="*60)
print("7. STATUS / DEADLINE REVIEW")
print("="*60)

statuses = df_check["Status"].astype(str).str.lower()
deadlines = df_check["Deadline"].astype(str).str.lower()

# Look for records claiming to be open/active/upcoming
# but having wording suggesting closure.
open_mask = statuses.str.contains(
    r"open|active|upcoming|ongoing",
    regex=True,
    na=False
)

closed_words = deadlines.str.contains(
    r"closed|awarded|implemented|previous round",
    regex=True,
    na=False
)

possible_status_conflicts = df_check[
    open_mask & closed_words
]

if len(possible_status_conflicts) == 0:
    print("PASS - No obvious status/deadline conflicts")
else:
    print("WARNING - Review these:")
    print(
        possible_status_conflicts[
            ["Funding Name", "Deadline", "Status"]
        ].to_string(index=False)
    )


# ============================================================
# 8. CATEGORY CHECK
# ============================================================

print("\n" + "="*60)
print("8. CATEGORY BREAKDOWN")
print("="*60)

print(
    df_check["Category"]
    .value_counts()
    .to_string()
)


# ============================================================
# 9. STATUS BREAKDOWN
# ============================================================

print("\n" + "="*60)
print("9. STATUS BREAKDOWN")
print("="*60)

print(
    df_check["Status"]
    .value_counts()
    .to_string()
)


# ============================================================
# 10. RECORD LENGTH / EMPTY TEXT CHECK
# ============================================================

print("\n" + "="*60)
print("10. EMPTY / VERY SHORT TEXT CHECK")
print("="*60)

text_columns = [
    "Funding Name",
    "Description",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Website",
    "Category",
    "Status"
]

for col in text_columns:
    short = df_check[
        df_check[col].astype(str).str.strip().str.len() < 3
    ]

    if len(short) > 0:
        print(f"\nWARNING - {col}: {len(short)} very short values")
        print(short[["Funding Name", col]].to_string(index=False))


# ============================================================
# 11. INSURANCE PROJECT REVIEW
# ============================================================

print("\n" + "="*60)
print("11. INSURANCE PROJECT / OPPORTUNITY REVIEW")
print("="*60)

insurance_mask = df_check["Category"].astype(str).str.contains(
    "insurance",
    case=False,
    na=False
)

insurance_check = df_check[insurance_mask].copy()

project_words = (
    r"implemented|grant agreement|project period|project brief|"
    r"programme ongoing|awarded|funded"
)

project_mask = (
    insurance_check["Status"]
    .astype(str)
    .str.contains(project_words, case=False, regex=True, na=False)
    |
    insurance_check["Deadline"]
    .astype(str)
    .str.contains(project_words, case=False, regex=True, na=False)
)

project_records = insurance_check[project_mask]

print("Insurance records:", len(insurance_check))
print("Potentially project/implemented records:", len(project_records))

if len(project_records) > 0:
    print("\nReview these records:")
    print(
        project_records[
            ["Funding Name", "Deadline", "Funding Amount", "Status"]
        ].to_string(index=False)
    )


# ============================================================
# 12. FINAL SUMMARY
# ============================================================

print("\n" + "="*60)
print("FINAL AUDIT SUMMARY")
print("="*60)

print("Total records:", len(df_check))
print("Total columns:", len(df_check.columns))
print("Exact duplicate names:", df_check["Funding Name"].duplicated().sum())
print("Exact duplicate rows:", df_check.duplicated().sum())
print("Total missing cells:", int(df_check.isna().sum().sum()))
print("Repeated website URLs:", len(repeated_websites))
print("Potential near-duplicates:", len(normalized_duplicates))
print("Potential insurance project records:", len(project_records))

print("\nAUDIT COMPLETE.")
print("No records were deleted or modified.")

DATAFRAMES WITH FUNDING SCHEMA:
  ai_final: (19, 10)
  insurance_df: (14, 10)
  insurance_final: (20, 10)
  insurance_funding: (3, 10)
  insurance_data: (20, 10)
  insurance_extra: (6, 10)
  ai_data: (19, 10)
  df: (19, 10)
  ai: (19, 10)
  insurance: (23, 10)
  ai_insurance: (42, 10)
  final_df: (54, 10)
  duplicate_names: (0, 10)
  bad_urls: (0, 10)


IndexError: list index out of range

In [31]:
# ============================================================
# SAVE FINAL FUNDING DATASET TO EXCEL
# ============================================================

import pandas as pd
import os

# Use the confirmed final dataframe
final_df = final_df.copy()

# Required column order
columns = [
    "Funding Name",
    "Description",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Website",
    "Category",
    "Status"
]

# Make sure columns are in the correct order
final_df = final_df[columns]

# Save location
output_file = os.path.join(
    os.getcwd(),
    "verified_funding_database_final.xlsx"
)

# Save to Excel
final_df.to_excel(
    output_file,
    index=False,
    engine="openpyxl"
)

print("=" * 60)
print("FINAL EXCEL FILE CREATED")
print("=" * 60)

print("File:", output_file)
print("Records:", len(final_df))
print("Columns:", len(final_df.columns))

print("\nColumn order:")
print(final_df.columns.tolist())

print("\nFinal dataset shape:", final_df.shape)

print("\nCategory breakdown:")
print(final_df["Category"].value_counts())

print("\nMissing cells:")
print(final_df.isna().sum())

print("\nDuplicate Funding Names:",
      final_df["Funding Name"].duplicated().sum())

print("\nSUCCESS - Your final Excel file is ready.")

FINAL EXCEL FILE CREATED
File: C:\Users\user\Documents\Python lessons\Week5\verified_funding_database_final.xlsx
Records: 54
Columns: 10

Column order:
['Funding Name', 'Description', 'Type', 'Eligibility', 'Deadline', 'States/Country/Region Covered', 'Funding Amount', 'Website', 'Category', 'Status']

Final dataset shape: (54, 10)

Category breakdown:
Category
Insurance                  32
Artificial Intelligence    13
AI Innovation               6
Insurance Innovation        3
Name: count, dtype: int64

Missing cells:
Funding Name                     0
Description                      0
Type                             0
Eligibility                      0
Deadline                         0
States/Country/Region Covered    0
Funding Amount                   0
Website                          0
Category                         0
Status                           0
dtype: int64

Duplicate Funding Names: 0

SUCCESS - Your final Excel file is ready.


In [32]:
import pandas as pd
import re
from pathlib import Path

# ============================================================
# 1. VERIFY EXISTING DATASET
# ============================================================

if "final_df" not in globals():
    raise ValueError(
        "final_df was not found. Make sure your current 54-record "
        "AI + Insurance dataset is loaded."
    )

base_df = final_df.copy()

print("Existing dataset:", base_df.shape)


# ============================================================
# 2. VERIFIED HEALTH FUNDING RECORDS
# ============================================================

health_records = [

    {
        "Funding Name": "Great Health for America",
        "Description": (
            "CDC cooperative agreement supporting national organizations "
            "working to improve chronic disease prevention and public health "
            "outcomes in the United States."
        ),
        "Type": "Cooperative Agreement",
        "Eligibility": (
            "Eligible U.S. organizations meeting the requirements in the "
            "CDC funding opportunity announcement."
        ),
        "Deadline": "3 August 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "$32 million total; 4 expected awards",
        "Website": "https://www.grants.gov/search-results-detail/363051",
        "Category": "Health",
        "Status": "Closed"
    },

    {
        "Funding Name": "Center for Indigenous Innovation and Health",
        "Description": (
            "Funding supporting research, education, service, partnerships "
            "and technical assistance addressing chronic disease and health "
            "care access gaps affecting Indigenous populations."
        ),
        "Type": "Cooperative Agreement",
        "Eligibility": (
            "Eligible nonprofit private institutions of higher education "
            "meeting the program requirements."
        ),
        "Deadline": "15 July 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "$2 million total; $500,000–$1 million per award",
        "Website": "https://www.grants.gov/search-results-detail/362800",
        "Category": "Health",
        "Status": "Closed"
    },

    {
        "Funding Name": "Fiscal Year 2027 Expanding Nutrition Services",
        "Description": (
            "HRSA funding designed to increase access to nutrition services "
            "at HRSA-funded health centers and increase the number of patients "
            "receiving nutrition services."
        ),
        "Type": "Grant",
        "Eligibility": (
            "Health Center Program award recipients with an active H80 award. "
            "Individuals are not eligible."
        ),
        "Deadline": "9 September 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": (
            "$125 million total; $350,000 award amount; "
            "357 expected awards"
        ),
        "Website": "https://www.grants.gov/search-results-detail/362823",
        "Category": "Health",
        "Status": "Open"
    },

    {
        "Funding Name": "Global Infectious Disease Research Training Program",
        "Description": (
            "NIH Fogarty international research-training funding supporting "
            "collaborative programs that strengthen infectious disease "
            "research capacity in low- and middle-income countries."
        ),
        "Type": "Research Training Grant",
        "Eligibility": (
            "Eligible U.S. institutions collaborating with researchers or "
            "institutions in eligible low- and middle-income countries; "
            "eligible foreign institutions may also apply under the program "
            "requirements."
        ),
        "Deadline": "6 August 2026",
        "States/Country/Region Covered": (
            "Low- and middle-income countries / International"
        ),
        "Funding Amount": "Not publicly stated",
        "Website": "https://www.fic.nih.gov/Programs/Pages/infectious-disease.aspx",
        "Category": "Health",
        "Status": "Closed"
    },

    {
        "Funding Name": (
            "NIH Director's Pioneer Award "
            "(DP1 Clinical Trial Optional)"
        ),
        "Description": (
            "NIH Common Fund award supporting exceptionally creative "
            "investigators pursuing highly innovative research with potential "
            "for major impact across biomedical and behavioral science."
        ),
        "Type": "Research Grant",
        "Eligibility": (
            "Eligible investigators and institutions meeting the NIH "
            "Director's Pioneer Award requirements."
        ),
        "Deadline": "10 September 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by award",
        "Website": "https://www.grants.nih.gov/funding/explore-nih-opportunities",
        "Category": "Health",
        "Status": "Upcoming"
    },

    {
        "Funding Name": (
            "NIH Director's Transformative Research Award "
            "(R01 Clinical Trial Optional)"
        ),
        "Description": (
            "NIH Common Fund funding supporting transformative research "
            "projects that have the potential to create or overturn major "
            "paradigms in biomedical or behavioral research."
        ),
        "Type": "Research Grant",
        "Eligibility": (
            "Eligible investigators and institutions meeting NIH "
            "Director's Transformative Research Award requirements."
        ),
        "Deadline": "4 September 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by award",
        "Website": "https://www.grants.nih.gov/funding/explore-nih-opportunities",
        "Category": "Health",
        "Status": "Upcoming"
    },

    {
        "Funding Name": (
            "NIH Director's Early Independence Award "
            "(DP5 Clinical Trial Optional)"
        ),
        "Description": (
            "NIH funding supporting recent doctoral degree recipients who "
            "seek to transition directly to independent research without "
            "a traditional postdoctoral training period."
        ),
        "Type": "Research Grant",
        "Eligibility": (
            "Eligible early-career researchers and U.S. domestic institutions "
            "meeting NIH Early Independence Award requirements."
        ),
        "Deadline": "11 September 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by award",
        "Website": "https://grants.nih.gov/funding/activity-codes/DP5",
        "Category": "Health",
        "Status": "Upcoming"
    },

    {
        "Funding Name": (
            "Atopic Dermatitis Research Network "
            "(ADRN U19 Clinical Trial Optional)"
        ),
        "Description": (
            "NIH research funding supporting collaborative research into "
            "atopic dermatitis and related scientific and clinical questions."
        ),
        "Type": "Research Grant",
        "Eligibility": (
            "Eligible research institutions and investigators meeting "
            "the NIH funding opportunity requirements."
        ),
        "Deadline": "25 September 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by award",
        "Website": "https://www.grants.nih.gov/funding/explore-nih-opportunities",
        "Category": "Health",
        "Status": "Upcoming"
    },

    {
        "Funding Name": (
            "Nutrition Obesity Research Centers "
            "(NORCs P30 Clinical Trial Optional)"
        ),
        "Description": (
            "NIH funding supporting research centers focused on nutrition, "
            "obesity and related health research."
        ),
        "Type": "Research Center Grant",
        "Eligibility": (
            "Eligible U.S. research institutions and investigators meeting "
            "the applicable NIH funding opportunity requirements."
        ),
        "Deadline": "21 October 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by award",
        "Website": "https://www.grants.nih.gov/funding/explore-nih-opportunities",
        "Category": "Health",
        "Status": "Upcoming"
    },

    {
        "Funding Name": (
            "Emerging Global Leader Award "
            "(K43 Independent Clinical Trial Required)"
        ),
        "Description": (
            "Fogarty International Center career-development funding "
            "supporting emerging global health research leaders in "
            "low- and middle-income countries."
        ),
        "Type": "Career Development Grant",
        "Eligibility": (
            "Eligible early-career investigators and institutions meeting "
            "Fogarty and NIH requirements for the K43 program."
        ),
        "Deadline": "3 December 2026",
        "States/Country/Region Covered": (
            "Low- and middle-income countries / International"
        ),
        "Funding Amount": "Varies by award",
        "Website": "https://www.fic.nih.gov/Funding/Pages/Fogarty-Funding-Opps.aspx",
        "Category": "Health",
        "Status": "Upcoming"
    }

]


# ============================================================
# 3. CREATE HEALTH DATAFRAME
# ============================================================

health_df = pd.DataFrame(health_records)

print("\nHealth records added:", len(health_df))


# ============================================================
# 4. STANDARDIZE COLUMN ORDER
# ============================================================

required_columns = [
    "Funding Name",
    "Description",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Website",
    "Category",
    "Status"
]

base_df = base_df[required_columns]
health_df = health_df[required_columns]


# ============================================================
# 5. COMBINE AI + INSURANCE + HEALTH
# ============================================================

combined_df = pd.concat(
    [base_df, health_df],
    ignore_index=True
)

print("\nBefore duplicate removal:", combined_df.shape)


# ============================================================
# 6. CLEAN TEXT
# ============================================================

for col in required_columns:
    combined_df[col] = (
        combined_df[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


# ============================================================
# 7. REMOVE DUPLICATE FUNDING NAMES
# ============================================================

before_duplicates = len(combined_df)

combined_df = combined_df.drop_duplicates(
    subset=["Funding Name"],
    keep="first"
).reset_index(drop=True)

duplicates_removed = before_duplicates - len(combined_df)

print("Duplicate funding names removed:", duplicates_removed)


# ============================================================
# 8. REMOVE COMPLETELY DUPLICATE ROWS
# ============================================================

before_rows = len(combined_df)

combined_df = combined_df.drop_duplicates(
    keep="first"
).reset_index(drop=True)

complete_duplicates_removed = before_rows - len(combined_df)

print("Duplicate complete rows removed:", complete_duplicates_removed)


# ============================================================
# 9. NULL / MISSING VALUE CHECK
# ============================================================

missing_values = combined_df[required_columns].isna().sum()

blank_values = (
    combined_df[required_columns]
    .astype(str)
    .apply(lambda col: col.str.strip().isin(["", "nan", "None"]).sum())
)

print("\n================ MISSING VALUE CHECK ================")
print(missing_values)

print("\nBlank-string check:")
print(blank_values)

total_missing = missing_values.sum() + blank_values.sum()

print("\nTotal missing/blank cells:", total_missing)


# ============================================================
# 10. URL VALIDATION
# ============================================================

url_pattern = re.compile(
    r"^https?://[^\s]+$",
    re.IGNORECASE
)

bad_url_rows = combined_df[
    ~combined_df["Website"].apply(
        lambda x: bool(url_pattern.match(str(x)))
    )
]

print("\n================ URL CHECK ================")
print("Invalid URLs:", len(bad_url_rows))

if len(bad_url_rows) > 0:
    print(bad_url_rows[["Funding Name", "Website"]])


# ============================================================
# 11. DUPLICATE WEBSITE CHECK
# ============================================================

duplicate_websites = (
    combined_df[
        combined_df["Website"].duplicated(keep=False)
    ]
    .sort_values("Website")
)

print("\n================ WEBSITE CHECK ================")
print(
    "Records sharing website URLs:",
    len(duplicate_websites)
)

if len(duplicate_websites) > 0:
    print(
        duplicate_websites[
            ["Funding Name", "Website"]
        ].to_string(index=False)
    )


# ============================================================
# 12. CATEGORY CHECK
# ============================================================

print("\n================ CATEGORY BREAKDOWN ================")
print(combined_df["Category"].value_counts())


# ============================================================
# 13. STATUS CHECK
# ============================================================

print("\n================ STATUS BREAKDOWN ================")
print(combined_df["Status"].value_counts())


# ============================================================
# 14. TYPE CHECK
# ============================================================

print("\n================ TYPE BREAKDOWN ================")
print(combined_df["Type"].value_counts())


# ============================================================
# 15. FINAL QUALITY TESTS
# ============================================================

print("\n================ FINAL QUALITY TEST ================")

# Column test
if list(combined_df.columns) == required_columns:
    print("PASS - Correct 10 columns")
else:
    print("FAIL - Column structure problem")

# Duplicate names
if combined_df["Funding Name"].duplicated().sum() == 0:
    print("PASS - No duplicate funding names")
else:
    print("FAIL - Duplicate funding names remain")

# Missing values
if total_missing == 0:
    print("PASS - No missing/blank values")
else:
    print("FAIL - Missing/blank values exist")

# Complete duplicates
if combined_df.duplicated().sum() == 0:
    print("PASS - No duplicate complete rows")
else:
    print("FAIL - Duplicate complete rows remain")

# URL format
if len(bad_url_rows) == 0:
    print("PASS - All websites have valid URL format")
else:
    print("FAIL - Invalid website URLs exist")


# ============================================================
# 16. HEALTH-SPECIFIC CHECK
# ============================================================

health_final = combined_df[
    combined_df["Category"].eq("Health")
].copy()

print("\n================ HEALTH CHECK ================")
print("Health records:", len(health_final))

print("\nHealth funding names:")
print(
    health_final["Funding Name"]
    .to_string(index=False)
)


# ============================================================
# 17. FINAL SUMMARY
# ============================================================

print("\n================ FINAL DATASET ================")

print("Total records:", len(combined_df))
print("Total columns:", len(combined_df.columns))

print("\nCategory totals:")
print(combined_df["Category"].value_counts())

print("\nDuplicate funding names:",
      combined_df["Funding Name"].duplicated().sum())

print("\nTotal missing/blank cells:",
      total_missing)


# ============================================================
# 18. SAVE FINAL DATASET
# ============================================================

output_folder = Path.cwd()

excel_path = output_folder / "verified_AI_Health_Insurance_Funding_FINAL.xlsx"
csv_path = output_folder / "verified_AI_Health_Insurance_Funding_FINAL.csv"

combined_df.to_excel(
    excel_path,
    index=False,
    sheet_name="Funding Database"
)

combined_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 19. SAVE SEPARATE HEALTH SHEET
# ============================================================

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    combined_df.to_excel(
        writer,
        index=False,
        sheet_name="All Funding"
    )

    health_final.to_excel(
        writer,
        index=False,
        sheet_name="Health Funding"
    )


print("\n================ FILES SAVED ================")
print("Excel:", excel_path)
print("CSV:", csv_path)

print("\nDONE.")

Existing dataset: (54, 10)

Health records added: 10

Before duplicate removal: (64, 10)
Duplicate funding names removed: 0
Duplicate complete rows removed: 0

================ MISSING VALUE CHECK ================
Funding Name                     0
Description                      0
Type                             0
Eligibility                      0
Deadline                         0
States/Country/Region Covered    0
Funding Amount                   0
Website                          0
Category                         0
Status                           0
dtype: int64

Blank-string check:
Funding Name                     0
Description                      0
Type                             0
Eligibility                      0
Deadline                         0
States/Country/Region Covered    0
Funding Amount                   0
Website                          0
Category                         0
Status                           0
dtype: int64

Total missing/blank cells: 0

========

In [32]:
import pandas as pd
import re
from pathlib import Path

# ============================================================
# 1. VERIFY EXISTING DATASET
# ============================================================

if "final_df" not in globals():
    raise ValueError(
        "final_df was not found. Make sure your current 54-record "
        "AI + Insurance dataset is loaded."
    )

base_df = final_df.copy()

print("Existing dataset:", base_df.shape)


# ============================================================
# 2. VERIFIED HEALTH FUNDING RECORDS
# ============================================================

health_records = [

    {
        "Funding Name": "Great Health for America",
        "Description": (
            "CDC cooperative agreement supporting national organizations "
            "working to improve chronic disease prevention and public health "
            "outcomes in the United States."
        ),
        "Type": "Cooperative Agreement",
        "Eligibility": (
            "Eligible U.S. organizations meeting the requirements in the "
            "CDC funding opportunity announcement."
        ),
        "Deadline": "3 August 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "$32 million total; 4 expected awards",
        "Website": "https://www.grants.gov/search-results-detail/363051",
        "Category": "Health",
        "Status": "Closed"
    },

    {
        "Funding Name": "Center for Indigenous Innovation and Health",
        "Description": (
            "Funding supporting research, education, service, partnerships "
            "and technical assistance addressing chronic disease and health "
            "care access gaps affecting Indigenous populations."
        ),
        "Type": "Cooperative Agreement",
        "Eligibility": (
            "Eligible nonprofit private institutions of higher education "
            "meeting the program requirements."
        ),
        "Deadline": "15 July 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "$2 million total; $500,000–$1 million per award",
        "Website": "https://www.grants.gov/search-results-detail/362800",
        "Category": "Health",
        "Status": "Closed"
    },

    {
        "Funding Name": "Fiscal Year 2027 Expanding Nutrition Services",
        "Description": (
            "HRSA funding designed to increase access to nutrition services "
            "at HRSA-funded health centers and increase the number of patients "
            "receiving nutrition services."
        ),
        "Type": "Grant",
        "Eligibility": (
            "Health Center Program award recipients with an active H80 award. "
            "Individuals are not eligible."
        ),
        "Deadline": "9 September 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": (
            "$125 million total; $350,000 award amount; "
            "357 expected awards"
        ),
        "Website": "https://www.grants.gov/search-results-detail/362823",
        "Category": "Health",
        "Status": "Open"
    },

    {
        "Funding Name": "Global Infectious Disease Research Training Program",
        "Description": (
            "NIH Fogarty international research-training funding supporting "
            "collaborative programs that strengthen infectious disease "
            "research capacity in low- and middle-income countries."
        ),
        "Type": "Research Training Grant",
        "Eligibility": (
            "Eligible U.S. institutions collaborating with researchers or "
            "institutions in eligible low- and middle-income countries; "
            "eligible foreign institutions may also apply under the program "
            "requirements."
        ),
        "Deadline": "6 August 2026",
        "States/Country/Region Covered": (
            "Low- and middle-income countries / International"
        ),
        "Funding Amount": "Not publicly stated",
        "Website": "https://www.fic.nih.gov/Programs/Pages/infectious-disease.aspx",
        "Category": "Health",
        "Status": "Closed"
    },

    {
        "Funding Name": (
            "NIH Director's Pioneer Award "
            "(DP1 Clinical Trial Optional)"
        ),
        "Description": (
            "NIH Common Fund award supporting exceptionally creative "
            "investigators pursuing highly innovative research with potential "
            "for major impact across biomedical and behavioral science."
        ),
        "Type": "Research Grant",
        "Eligibility": (
            "Eligible investigators and institutions meeting the NIH "
            "Director's Pioneer Award requirements."
        ),
        "Deadline": "10 September 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by award",
        "Website": "https://www.grants.nih.gov/funding/explore-nih-opportunities",
        "Category": "Health",
        "Status": "Upcoming"
    },

    {
        "Funding Name": (
            "NIH Director's Transformative Research Award "
            "(R01 Clinical Trial Optional)"
        ),
        "Description": (
            "NIH Common Fund funding supporting transformative research "
            "projects that have the potential to create or overturn major "
            "paradigms in biomedical or behavioral research."
        ),
        "Type": "Research Grant",
        "Eligibility": (
            "Eligible investigators and institutions meeting NIH "
            "Director's Transformative Research Award requirements."
        ),
        "Deadline": "4 September 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by award",
        "Website": "https://www.grants.nih.gov/funding/explore-nih-opportunities",
        "Category": "Health",
        "Status": "Upcoming"
    },

    {
        "Funding Name": (
            "NIH Director's Early Independence Award "
            "(DP5 Clinical Trial Optional)"
        ),
        "Description": (
            "NIH funding supporting recent doctoral degree recipients who "
            "seek to transition directly to independent research without "
            "a traditional postdoctoral training period."
        ),
        "Type": "Research Grant",
        "Eligibility": (
            "Eligible early-career researchers and U.S. domestic institutions "
            "meeting NIH Early Independence Award requirements."
        ),
        "Deadline": "11 September 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by award",
        "Website": "https://grants.nih.gov/funding/activity-codes/DP5",
        "Category": "Health",
        "Status": "Upcoming"
    },

    {
        "Funding Name": (
            "Atopic Dermatitis Research Network "
            "(ADRN U19 Clinical Trial Optional)"
        ),
        "Description": (
            "NIH research funding supporting collaborative research into "
            "atopic dermatitis and related scientific and clinical questions."
        ),
        "Type": "Research Grant",
        "Eligibility": (
            "Eligible research institutions and investigators meeting "
            "the NIH funding opportunity requirements."
        ),
        "Deadline": "25 September 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by award",
        "Website": "https://www.grants.nih.gov/funding/explore-nih-opportunities",
        "Category": "Health",
        "Status": "Upcoming"
    },

    {
        "Funding Name": (
            "Nutrition Obesity Research Centers "
            "(NORCs P30 Clinical Trial Optional)"
        ),
        "Description": (
            "NIH funding supporting research centers focused on nutrition, "
            "obesity and related health research."
        ),
        "Type": "Research Center Grant",
        "Eligibility": (
            "Eligible U.S. research institutions and investigators meeting "
            "the applicable NIH funding opportunity requirements."
        ),
        "Deadline": "21 October 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by award",
        "Website": "https://www.grants.nih.gov/funding/explore-nih-opportunities",
        "Category": "Health",
        "Status": "Upcoming"
    },

    {
        "Funding Name": (
            "Emerging Global Leader Award "
            "(K43 Independent Clinical Trial Required)"
        ),
        "Description": (
            "Fogarty International Center career-development funding "
            "supporting emerging global health research leaders in "
            "low- and middle-income countries."
        ),
        "Type": "Career Development Grant",
        "Eligibility": (
            "Eligible early-career investigators and institutions meeting "
            "Fogarty and NIH requirements for the K43 program."
        ),
        "Deadline": "3 December 2026",
        "States/Country/Region Covered": (
            "Low- and middle-income countries / International"
        ),
        "Funding Amount": "Varies by award",
        "Website": "https://www.fic.nih.gov/Funding/Pages/Fogarty-Funding-Opps.aspx",
        "Category": "Health",
        "Status": "Upcoming"
    }

]


# ============================================================
# 3. CREATE HEALTH DATAFRAME
# ============================================================

health_df = pd.DataFrame(health_records)

print("\nHealth records added:", len(health_df))


# ============================================================
# 4. STANDARDIZE COLUMN ORDER
# ============================================================

required_columns = [
    "Funding Name",
    "Description",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Website",
    "Category",
    "Status"
]

base_df = base_df[required_columns]
health_df = health_df[required_columns]


# ============================================================
# 5. COMBINE AI + INSURANCE + HEALTH
# ============================================================

combined_df = pd.concat(
    [base_df, health_df],
    ignore_index=True
)

print("\nBefore duplicate removal:", combined_df.shape)


# ============================================================
# 6. CLEAN TEXT
# ============================================================

for col in required_columns:
    combined_df[col] = (
        combined_df[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


# ============================================================
# 7. REMOVE DUPLICATE FUNDING NAMES
# ============================================================

before_duplicates = len(combined_df)

combined_df = combined_df.drop_duplicates(
    subset=["Funding Name"],
    keep="first"
).reset_index(drop=True)

duplicates_removed = before_duplicates - len(combined_df)

print("Duplicate funding names removed:", duplicates_removed)


# ============================================================
# 8. REMOVE COMPLETELY DUPLICATE ROWS
# ============================================================

before_rows = len(combined_df)

combined_df = combined_df.drop_duplicates(
    keep="first"
).reset_index(drop=True)

complete_duplicates_removed = before_rows - len(combined_df)

print("Duplicate complete rows removed:", complete_duplicates_removed)


# ============================================================
# 9. NULL / MISSING VALUE CHECK
# ============================================================

missing_values = combined_df[required_columns].isna().sum()

blank_values = (
    combined_df[required_columns]
    .astype(str)
    .apply(lambda col: col.str.strip().isin(["", "nan", "None"]).sum())
)

print("\n================ MISSING VALUE CHECK ================")
print(missing_values)

print("\nBlank-string check:")
print(blank_values)

total_missing = missing_values.sum() + blank_values.sum()

print("\nTotal missing/blank cells:", total_missing)


# ============================================================
# 10. URL VALIDATION
# ============================================================

url_pattern = re.compile(
    r"^https?://[^\s]+$",
    re.IGNORECASE
)

bad_url_rows = combined_df[
    ~combined_df["Website"].apply(
        lambda x: bool(url_pattern.match(str(x)))
    )
]

print("\n================ URL CHECK ================")
print("Invalid URLs:", len(bad_url_rows))

if len(bad_url_rows) > 0:
    print(bad_url_rows[["Funding Name", "Website"]])


# ============================================================
# 11. DUPLICATE WEBSITE CHECK
# ============================================================

duplicate_websites = (
    combined_df[
        combined_df["Website"].duplicated(keep=False)
    ]
    .sort_values("Website")
)

print("\n================ WEBSITE CHECK ================")
print(
    "Records sharing website URLs:",
    len(duplicate_websites)
)

if len(duplicate_websites) > 0:
    print(
        duplicate_websites[
            ["Funding Name", "Website"]
        ].to_string(index=False)
    )


# ============================================================
# 12. CATEGORY CHECK
# ============================================================

print("\n================ CATEGORY BREAKDOWN ================")
print(combined_df["Category"].value_counts())


# ============================================================
# 13. STATUS CHECK
# ============================================================

print("\n================ STATUS BREAKDOWN ================")
print(combined_df["Status"].value_counts())


# ============================================================
# 14. TYPE CHECK
# ============================================================

print("\n================ TYPE BREAKDOWN ================")
print(combined_df["Type"].value_counts())


# ============================================================
# 15. FINAL QUALITY TESTS
# ============================================================

print("\n================ FINAL QUALITY TEST ================")

# Column test
if list(combined_df.columns) == required_columns:
    print("PASS - Correct 10 columns")
else:
    print("FAIL - Column structure problem")

# Duplicate names
if combined_df["Funding Name"].duplicated().sum() == 0:
    print("PASS - No duplicate funding names")
else:
    print("FAIL - Duplicate funding names remain")

# Missing values
if total_missing == 0:
    print("PASS - No missing/blank values")
else:
    print("FAIL - Missing/blank values exist")

# Complete duplicates
if combined_df.duplicated().sum() == 0:
    print("PASS - No duplicate complete rows")
else:
    print("FAIL - Duplicate complete rows remain")

# URL format
if len(bad_url_rows) == 0:
    print("PASS - All websites have valid URL format")
else:
    print("FAIL - Invalid website URLs exist")


# ============================================================
# 16. HEALTH-SPECIFIC CHECK
# ============================================================

health_final = combined_df[
    combined_df["Category"].eq("Health")
].copy()

print("\n================ HEALTH CHECK ================")
print("Health records:", len(health_final))

print("\nHealth funding names:")
print(
    health_final["Funding Name"]
    .to_string(index=False)
)


# ============================================================
# 17. FINAL SUMMARY
# ============================================================

print("\n================ FINAL DATASET ================")

print("Total records:", len(combined_df))
print("Total columns:", len(combined_df.columns))

print("\nCategory totals:")
print(combined_df["Category"].value_counts())

print("\nDuplicate funding names:",
      combined_df["Funding Name"].duplicated().sum())

print("\nTotal missing/blank cells:",
      total_missing)


# ============================================================
# 18. SAVE FINAL DATASET
# ============================================================

output_folder = Path.cwd()

excel_path = output_folder / "verified_AI_Health_Insurance_Funding_FINAL.xlsx"
csv_path = output_folder / "verified_AI_Health_Insurance_Funding_FINAL.csv"

combined_df.to_excel(
    excel_path,
    index=False,
    sheet_name="Funding Database"
)

combined_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 19. SAVE SEPARATE HEALTH SHEET
# ============================================================

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    combined_df.to_excel(
        writer,
        index=False,
        sheet_name="All Funding"
    )

    health_final.to_excel(
        writer,
        index=False,
        sheet_name="Health Funding"
    )


print("\n================ FILES SAVED ================")
print("Excel:", excel_path)
print("CSV:", csv_path)

print("\nDONE.")

Existing dataset: (54, 10)

Health records added: 10

Before duplicate removal: (64, 10)
Duplicate funding names removed: 0
Duplicate complete rows removed: 0

================ MISSING VALUE CHECK ================
Funding Name                     0
Description                      0
Type                             0
Eligibility                      0
Deadline                         0
States/Country/Region Covered    0
Funding Amount                   0
Website                          0
Category                         0
Status                           0
dtype: int64

Blank-string check:
Funding Name                     0
Description                      0
Type                             0
Eligibility                      0
Deadline                         0
States/Country/Region Covered    0
Funding Amount                   0
Website                          0
Category                         0
Status                           0
dtype: int64

Total missing/blank cells: 0

========

In [33]:
import pandas as pd
import re
from pathlib import Path

# ============================================================
# 1. VERIFY EXISTING DATASET
# ============================================================

if "final_df" not in globals():
    raise ValueError(
        "final_df was not found. Make sure your current "
        "AI + Insurance dataset is loaded."
    )

base_df = final_df.copy()

print("Existing final_df:", base_df.shape)


# ============================================================
# 2. REQUIRED COLUMN STRUCTURE
# ============================================================

required_columns = [
    "Funding Name",
    "Description",
    "Type",
    "Eligibility",
    "Deadline",
    "States/Country/Region Covered",
    "Funding Amount",
    "Website",
    "Category",
    "Status"
]

# Check columns before doing anything
missing_columns = [
    col for col in required_columns
    if col not in base_df.columns
]

if missing_columns:
    raise ValueError(
        f"final_df is missing these required columns: {missing_columns}"
    )

base_df = base_df[required_columns].copy()


# ============================================================
# 3. VERIFIED HEALTH FUNDING RECORDS
# ============================================================

health_records = [

    {
        "Funding Name": "Great Health for America",
        "Description": (
            "CDC cooperative agreement supporting national organizations "
            "working to improve chronic disease prevention and public health "
            "outcomes in the United States."
        ),
        "Type": "Cooperative Agreement",
        "Eligibility": (
            "Eligible U.S. organizations meeting the requirements in the "
            "CDC funding opportunity announcement."
        ),
        "Deadline": "3 August 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "$32 million total; 4 expected awards",
        "Website": "https://www.grants.gov/search-results-detail/363051",
        "Category": "Health",
        "Status": "Closed"
    },

    {
        "Funding Name": "Center for Indigenous Innovation and Health",
        "Description": (
            "Funding supporting research, education, service, partnerships "
            "and technical assistance addressing chronic disease and health "
            "care access gaps affecting Indigenous populations."
        ),
        "Type": "Cooperative Agreement",
        "Eligibility": (
            "Eligible nonprofit private institutions of higher education "
            "meeting the program requirements."
        ),
        "Deadline": "15 July 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "$2 million total; $500,000–$1 million per award",
        "Website": "https://www.grants.gov/search-results-detail/362800",
        "Category": "Health",
        "Status": "Closed"
    },

    {
        "Funding Name": "Fiscal Year 2027 Expanding Nutrition Services",
        "Description": (
            "HRSA funding designed to increase access to nutrition services "
            "at HRSA-funded health centers and increase the number of patients "
            "receiving nutrition services."
        ),
        "Type": "Grant",
        "Eligibility": (
            "Health Center Program award recipients with an active H80 award. "
            "Individuals are not eligible."
        ),
        "Deadline": "9 September 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": (
            "$125 million total; $350,000 award amount; "
            "357 expected awards"
        ),
        "Website": "https://www.grants.gov/search-results-detail/362823",
        "Category": "Health",
        "Status": "Open"
    },

    {
        "Funding Name": "Global Infectious Disease Research Training Program",
        "Description": (
            "NIH Fogarty international research-training funding supporting "
            "collaborative programs that strengthen infectious disease "
            "research capacity in low- and middle-income countries."
        ),
        "Type": "Research Training Grant",
        "Eligibility": (
            "Eligible U.S. institutions collaborating with researchers or "
            "institutions in eligible low- and middle-income countries; "
            "eligible foreign institutions may also apply under program "
            "requirements."
        ),
        "Deadline": "6 August 2026",
        "States/Country/Region Covered": (
            "Low- and middle-income countries / International"
        ),
        "Funding Amount": "Not publicly stated",
        "Website": "https://www.fic.nih.gov/Programs/Pages/infectious-disease.aspx",
        "Category": "Health",
        "Status": "Closed"
    },

    {
        "Funding Name": (
            "NIH Director's Pioneer Award (DP1 Clinical Trial Optional)"
        ),
        "Description": (
            "NIH Common Fund award supporting exceptionally creative "
            "investigators pursuing highly innovative research with potential "
            "for major impact across biomedical and behavioral science."
        ),
        "Type": "Research Grant",
        "Eligibility": (
            "Eligible investigators and institutions meeting the NIH "
            "Director's Pioneer Award requirements."
        ),
        "Deadline": "10 September 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by award",
        "Website": (
            "https://www.grants.nih.gov/funding/explore-nih-opportunities"
        ),
        "Category": "Health",
        "Status": "Upcoming"
    },

    {
        "Funding Name": (
            "NIH Director's Transformative Research Award "
            "(R01 Clinical Trial Optional)"
        ),
        "Description": (
            "NIH Common Fund funding supporting transformative research "
            "projects that have the potential to create or overturn major "
            "paradigms in biomedical or behavioral research."
        ),
        "Type": "Research Grant",
        "Eligibility": (
            "Eligible investigators and institutions meeting NIH "
            "Director's Transformative Research Award requirements."
        ),
        "Deadline": "4 September 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by award",
        "Website": (
            "https://www.grants.nih.gov/funding/explore-nih-opportunities"
        ),
        "Category": "Health",
        "Status": "Upcoming"
    },

    {
        "Funding Name": (
            "NIH Director's Early Independence Award "
            "(DP5 Clinical Trial Optional)"
        ),
        "Description": (
            "NIH funding supporting recent doctoral degree recipients who "
            "seek to transition directly to independent research without "
            "a traditional postdoctoral training period."
        ),
        "Type": "Research Grant",
        "Eligibility": (
            "Eligible early-career researchers and U.S. domestic institutions "
            "meeting NIH Early Independence Award requirements."
        ),
        "Deadline": "11 September 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by award",
        "Website": (
            "https://grants.nih.gov/funding/activity-codes/DP5"
        ),
        "Category": "Health",
        "Status": "Upcoming"
    },

    {
        "Funding Name": (
            "Atopic Dermatitis Research Network "
            "(ADRN U19 Clinical Trial Optional)"
        ),
        "Description": (
            "NIH research funding supporting collaborative research into "
            "atopic dermatitis and related scientific and clinical questions."
        ),
        "Type": "Research Grant",
        "Eligibility": (
            "Eligible research institutions and investigators meeting "
            "the NIH funding opportunity requirements."
        ),
        "Deadline": "25 September 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by award",
        "Website": (
            "https://www.grants.nih.gov/funding/explore-nih-opportunities"
        ),
        "Category": "Health",
        "Status": "Upcoming"
    },

    {
        "Funding Name": (
            "Nutrition Obesity Research Centers "
            "(NORCs P30 Clinical Trial Optional)"
        ),
        "Description": (
            "NIH funding supporting research centers focused on nutrition, "
            "obesity and related health research."
        ),
        "Type": "Research Center Grant",
        "Eligibility": (
            "Eligible U.S. research institutions and investigators meeting "
            "the applicable NIH funding opportunity requirements."
        ),
        "Deadline": "21 October 2026",
        "States/Country/Region Covered": "United States",
        "Funding Amount": "Varies by award",
        "Website": (
            "https://www.grants.nih.gov/funding/explore-nih-opportunities"
        ),
        "Category": "Health",
        "Status": "Upcoming"
    },

    {
        "Funding Name": (
            "Emerging Global Leader Award "
            "(K43 Independent Clinical Trial Required)"
        ),
        "Description": (
            "Fogarty International Center career-development funding "
            "supporting emerging global health research leaders in "
            "low- and middle-income countries."
        ),
        "Type": "Career Development Grant",
        "Eligibility": (
            "Eligible early-career investigators and institutions meeting "
            "Fogarty and NIH requirements for the K43 program."
        ),
        "Deadline": "3 December 2026",
        "States/Country/Region Covered": (
            "Low- and middle-income countries / International"
        ),
        "Funding Amount": "Varies by award",
        "Website": (
            "https://www.fic.nih.gov/Funding/Pages/Fogarty-Funding-Opps.aspx"
        ),
        "Category": "Health",
        "Status": "Upcoming"
    }
]


# ============================================================
# 4. CREATE HEALTH DATAFRAME
# ============================================================

health_df = pd.DataFrame(health_records)

health_df = health_df[required_columns].copy()

print("\nHealth records prepared:", len(health_df))


# ============================================================
# 5. CLEAN TEXT IN BOTH DATASETS
# ============================================================

for col in required_columns:

    base_df[col] = (
        base_df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    health_df[col] = (
        health_df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


# ============================================================
# 6. COMBINE AI + INSURANCE + HEALTH
# ============================================================

combined_df = pd.concat(
    [base_df, health_df],
    ignore_index=True
)

print("\nBefore duplicate removal:", combined_df.shape)


# ============================================================
# 7. REMOVE DUPLICATE FUNDING NAMES
# ============================================================

before_duplicates = len(combined_df)

combined_df = (
    combined_df
    .drop_duplicates(
        subset=["Funding Name"],
        keep="first"
    )
    .reset_index(drop=True)
)

duplicates_removed = before_duplicates - len(combined_df)

print(
    "Duplicate funding names removed:",
    duplicates_removed
)


# ============================================================
# 8. REMOVE COMPLETE DUPLICATE ROWS
# ============================================================

before_complete_duplicates = len(combined_df)

combined_df = (
    combined_df
    .drop_duplicates(keep="first")
    .reset_index(drop=True)
)

complete_duplicates_removed = (
    before_complete_duplicates - len(combined_df)
)

print(
    "Complete duplicate rows removed:",
    complete_duplicates_removed
)


# ============================================================
# 9. MISSING / BLANK VALUE CHECK
# ============================================================

missing_values = combined_df[required_columns].isna().sum()

blank_values = (
    combined_df[required_columns]
    .astype(str)
    .apply(
        lambda col: col.str.strip()
        .isin(["", "nan", "None"])
        .sum()
    )
)

total_missing = (
    missing_values.sum() +
    blank_values.sum()
)

print("\n================ MISSING VALUE CHECK ================")
print(missing_values)

print("\nBlank-string check:")
print(blank_values)

print("\nTotal missing/blank cells:", total_missing)


# ============================================================
# 10. URL FORMAT CHECK
# ============================================================

url_pattern = re.compile(
    r"^https?://[^\s]+$",
    re.IGNORECASE
)

bad_url_rows = combined_df[
    ~combined_df["Website"].apply(
        lambda x: bool(url_pattern.match(str(x)))
    )
]

print("\n================ URL CHECK ================")
print("Invalid URL format:", len(bad_url_rows))

if len(bad_url_rows) > 0:
    print(
        bad_url_rows[
            ["Funding Name", "Website"]
        ].to_string(index=False)
    )


# ============================================================
# 11. DUPLICATE WEBSITE CHECK
# ============================================================

duplicate_websites = (
    combined_df[
        combined_df["Website"].duplicated(keep=False)
    ]
    .sort_values("Website")
)

print("\n================ WEBSITE CHECK ================")

print(
    "Records sharing website URLs:",
    len(duplicate_websites)
)

if len(duplicate_websites) > 0:
    print(
        duplicate_websites[
            ["Funding Name", "Website"]
        ].to_string(index=False)
    )


# ============================================================
# 12. CATEGORY CHECK
# ============================================================

print("\n================ CATEGORY BREAKDOWN ================")

category_counts = combined_df["Category"].value_counts()

print(category_counts)


# ============================================================
# 13. STATUS CHECK
# ============================================================

print("\n================ STATUS BREAKDOWN ================")

print(
    combined_df["Status"].value_counts()
)


# ============================================================
# 14. TYPE CHECK
# ============================================================

print("\n================ TYPE BREAKDOWN ================")

print(
    combined_df["Type"].value_counts()
)


# ============================================================
# 15. FINAL QUALITY TESTS
# ============================================================

print("\n================ FINAL QUALITY TEST ================")

# Column structure
if list(combined_df.columns) == required_columns:
    print("PASS - Correct 10 columns")
else:
    print("FAIL - Column structure problem")


# Duplicate names
duplicate_name_count = (
    combined_df["Funding Name"].duplicated().sum()
)

if duplicate_name_count == 0:
    print("PASS - No duplicate funding names")
else:
    print(
        f"FAIL - {duplicate_name_count} duplicate funding names remain"
    )


# Missing values
if total_missing == 0:
    print("PASS - No missing/blank values")
else:
    print(
        f"FAIL - {total_missing} missing/blank cells exist"
    )


# Complete duplicates
complete_duplicate_count = (
    combined_df.duplicated().sum()
)

if complete_duplicate_count == 0:
    print("PASS - No duplicate complete rows")
else:
    print(
        f"FAIL - {complete_duplicate_count} complete duplicates remain"
    )


# URL format
if len(bad_url_rows) == 0:
    print("PASS - All websites have valid URL format")
else:
    print(
        f"FAIL - {len(bad_url_rows)} invalid website URLs exist"
    )


# ============================================================
# 16. HEALTH-SPECIFIC CHECK
# ============================================================

health_final = combined_df[
    combined_df["Category"].eq("Health")
].copy()

print("\n================ HEALTH CHECK ================")

print(
    "Health records:",
    len(health_final)
)

print("\nHealth funding names:")

print(
    health_final["Funding Name"]
    .to_string(index=False)
)


# ============================================================
# 17. FINAL DATASET SUMMARY
# ============================================================

print("\n================ FINAL DATASET ================")

print(
    "Total records:",
    len(combined_df)
)

print(
    "Total columns:",
    len(combined_df.columns)
)

print("\nCategory totals:")

print(
    combined_df["Category"].value_counts()
)

print(
    "\nDuplicate funding names:",
    combined_df["Funding Name"].duplicated().sum()
)

print(
    "Total missing/blank cells:",
    total_missing
)


# ============================================================
# 18. SAVE FINAL EXCEL + CSV
# ============================================================

output_folder = Path.cwd()

excel_path = (
    output_folder /
    "verified_AI_Health_Insurance_Funding_FINAL.xlsx"
)

csv_path = (
    output_folder /
    "verified_AI_Health_Insurance_Funding_FINAL.csv"
)


# ============================================================
# 19. SAVE MULTIPLE SHEETS TO EXCEL
# ============================================================

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    # Complete database
    combined_df.to_excel(
        writer,
        index=False,
        sheet_name="Funding Database"
    )

    # AI records
    ai_final = combined_df[
        combined_df["Category"].isin(
            ["AI Innovation", "Artificial Intelligence"]
        )
    ].copy()

    ai_final.to_excel(
        writer,
        index=False,
        sheet_name="AI Funding"
    )

    # Insurance records
    insurance_final = combined_df[
        combined_df["Category"].isin(
            ["Insurance", "Insurance Innovation"]
        )
    ].copy()

    insurance_final.to_excel(
        writer,
        index=False,
        sheet_name="Insurance Funding"
    )

    # Health records
    health_final.to_excel(
        writer,
        index=False,
        sheet_name="Health Funding"
    )


# ============================================================
# 20. SAVE CSV
# ============================================================

combined_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 21. FINAL OUTPUT
# ============================================================

print("\n================ FILES SAVED ================")

print("Excel file:")
print(excel_path)

print("\nCSV file:")
print(csv_path)

print("\n================ DONE ================")

print(
    f"Final dataset contains {len(combined_df)} records "
    f"and {len(combined_df.columns)} columns."
)

Existing final_df: (54, 10)

Health records prepared: 10

Before duplicate removal: (64, 10)
Duplicate funding names removed: 0
Complete duplicate rows removed: 0

================ MISSING VALUE CHECK ================
Funding Name                     0
Description                      0
Type                             0
Eligibility                      0
Deadline                         0
States/Country/Region Covered    0
Funding Amount                   0
Website                          0
Category                         0
Status                           0
dtype: int64

Blank-string check:
Funding Name                     0
Description                      0
Type                             0
Eligibility                      0
Deadline                         0
States/Country/Region Covered    0
Funding Amount                   0
Website                          0
Category                         0
Status                           0
dtype: int64

Total missing/blank cells: 0

====

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\user\\Documents\\Python lessons\\Week5\\verified_AI_Health_Insurance_Funding_FINAL.xlsx'

In [34]:
excel_path = Path.cwd() / "FINAL_Funding_Database_64_Records.xlsx"
csv_path = Path.cwd() / "FINAL_Funding_Database_64_Records.csv"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    combined_df.to_excel(
        writer,
        index=False,
        sheet_name="Funding Database"
    )

    combined_df[combined_df["Category"].isin(
        ["Artificial Intelligence", "AI Innovation"]
    )].to_excel(
        writer,
        index=False,
        sheet_name="AI"
    )

    combined_df[combined_df["Category"].isin(
        ["Insurance", "Insurance Innovation"]
    )].to_excel(
        writer,
        index=False,
        sheet_name="Insurance"
    )

    combined_df[combined_df["Category"] == "Health"].to_excel(
        writer,
        index=False,
        sheet_name="Health"
    )

combined_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("Excel saved:", excel_path)
print("CSV saved:", csv_path)

Excel saved: C:\Users\user\Documents\Python lessons\Week5\FINAL_Funding_Database_64_Records.xlsx
CSV saved: C:\Users\user\Documents\Python lessons\Week5\FINAL_Funding_Database_64_Records.csv
